# Consolidated Publication Figures -- Dogwhistle Benchmarking Audit

This notebook consolidates every figure-generation code path found in this
repository (full Step-1 inventory + Step-2 provenance table + Step-3
cross-stage results are in the final summary cell) into a single, runnable,
provenance-annotated notebook. It was built by copying the exact plotting
logic out of the source files listed below -- including a known
sort-direction bug, originally left in place and flagged inline (since
fixed -- see "Known bugs fixed" in the Step 4 summary below) -- and
adding a cross-stage numerical
consistency check section (Step 3) that did not exist before.

**Source files consolidated here:**

1. `audit_pipeline/stage5_figures.py` -- the pipeline's own Stage 5, driven
   by `PipelineVariant` (Stage 1-4 outputs in, Stage 5 PNGs out).
2. `generate_figures_final.py` (repo root) -- a second, independent
   "camera-ready" figure script, **not** variant-aware, writing PDFs to
   `outputs/figures_final/`.
3. `generate_fig2_annotation_di_pairwise.py` (repo root) -- a one-off figure
   script that explicitly imports its styling helpers from
   `generate_figures_final.py` and merges two different Stage 4 pairwise
   files itself.
4. `auditing/03_audit_visualizations.ipynb` -- figures from a **separate,
   deprecated, exploratory 00-03 notebook pipeline** (`auditing/`) whose
   input directories (`outputs/coverage_audits/`, `outputs/annotation_audits/`,
   `outputs/disparity_audits/`) have since been relocated under
   `outputs/deprecated/`. Its metric definitions (Case A/B/C, DI ratio,
   presence rate) look similar to `audit_pipeline`'s but are computed by
   entirely separate code with its own Stage 00-03 numbering (**not** the
   same numbering as `audit_pipeline`'s Stage 1-5), and -- discovered while
   building this notebook -- it never stratifies by coding level (L1-L4) at
   all, unlike `audit_pipeline`. Included here per the guardrail that
   conflicting/duplicate figure logic must be surfaced, not silently
   dropped.

**Output paths are preserved exactly as in the source files** so nothing
downstream (the paper's LaTeX includes) breaks. The one disclosed exception
is the `auditing/` section's *input* directories, updated to
`outputs/deprecated/...` because the directories the original notebook
pointed at no longer exist at that location -- without this the notebook
cannot run top-to-bottom at all. This is a path fix, not a logic fix; no
plotting/filtering/threshold code was changed.


## Setup -- imports, repository paths, and the output sandbox

`REPO_ROOT` is computed explicitly (rather than relying on notebook cwd)
because `generate_figures_final.py` and `generate_fig2_annotation_di_pairwise.py`
take a *relative* `data_dir="outputs/"` default in their original form --
this notebook always passes an absolute path built from `REPO_ROOT` instead,
so figure generation works regardless of where Jupyter's working directory
happens to be. `audit_pipeline.stage5_figures` does not have this problem:
its paths come from `audit_pipeline.config`, already anchored to an absolute
`WORKDIR`.

**Sandboxing (2026-07-03 style pass).** A prior validation pass leaked 31
files to real `outputs/` paths because an output-redirect monkeypatch was
installed *after* some helper functions were already defined and callable --
a few figures got written for real before the redirect took effect. This
notebook fixes that class of mistake structurally rather than carefully: it
does not monkeypatch `_save`/`_save_s5` at all. Instead, `OUTPUTS_DIR` itself
-- the one root every figure-writing path in this notebook derives from,
whether directly, via `GFF_DATA_DIR`, or via `PipelineVariant.out_s5` --
is pointed at a fresh sandbox directory (`outputs/_style_pass_sandbox/`) in
*this same, first executable cell*, before a single figure function is even
defined, let alone called. Read-only subdirectories (`stage1`-`stage4`,
`unioned_data`, `deprecated`) are symlinked into the sandbox so figure code
still sees real pipeline data; output subdirectories (`figures_final`,
`audit_visualizations`, `stage5`) are created fresh and empty. There is no
window in which a helper function exists but still points at a real
production path, because nothing downstream can observe `OUTPUTS_DIR` before
this cell finishes running.

Set `SANDBOX_MODE = False` only as the one-line change described in the
final "Step 7 -- promote to production" cell, once the style pass has been
reviewed and approved.


In [ ]:
from __future__ import annotations

import os
import shutil
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "audit_pipeline").is_dir():
    if REPO_ROOT == REPO_ROOT.parent:
        raise RuntimeError("Could not locate repo root (no audit_pipeline/ found above cwd).")
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

_REAL_OUTPUTS_DIR = REPO_ROOT / "outputs"

# --- Output sandbox: the ONLY place this notebook decides where figures go ---
# SANDBOX_MODE = True: every figure-writing path resolves under
# outputs/_style_pass_sandbox/ (git-ignored). SANDBOX_MODE = False: every
# figure-writing path resolves under the real outputs/ tree the paper's
# LaTeX includes read from -- see "Step 7 -- promote to production" at the
# end of this notebook before ever flipping this.
SANDBOX_MODE = True
SANDBOX_DIR = _REAL_OUTPUTS_DIR / "_style_pass_sandbox"

if SANDBOX_MODE:
    if SANDBOX_DIR.exists():
        shutil.rmtree(SANDBOX_DIR)
    SANDBOX_DIR.mkdir(parents=True)
    for _name in ("stage1", "stage2", "stage3", "stage4", "unioned_data", "deprecated"):
        _src = _REAL_OUTPUTS_DIR / _name
        if _src.exists():
            os.symlink(_src, SANDBOX_DIR / _name)
    for _name in ("figures_final", "audit_visualizations", "stage5"):
        (SANDBOX_DIR / _name).mkdir(parents=True, exist_ok=True)
    OUTPUTS_DIR = SANDBOX_DIR
else:
    OUTPUTS_DIR = _REAL_OUTPUTS_DIR

GFF_DATA_DIR = str(OUTPUTS_DIR) + "/"        # generate_figures_final.py's data_dir param

import matplotlib as mpl
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, Normalize
from pandas.errors import EmptyDataError

from audit_pipeline.config import (
    DI_THRESHOLD,
    N_MIN,
    VARIANT_FULL,
    VARIANT_TIER12,
    PipelineVariant,
)
from audit_pipeline.helpers import map_reporting_group, norm_target

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"DI_THRESHOLD (audit_pipeline.config) = {DI_THRESHOLD}")
print(f"SANDBOX_MODE = {SANDBOX_MODE}  ->  OUTPUTS_DIR = {OUTPUTS_DIR}")

# This notebook regenerates the PRIMARY ("full", all glossary tiers) analysis
# figures by default, matching how stage5_figures.py / generate_figures_final.py
# / generate_fig2_annotation_di_pairwise.py are actually invoked for the paper.
# The tier1+2 robustness-check variant (VARIANT_TIER12) uses the identical
# code paths below with variant=VARIANT_TIER12 substituted; it is not run a
# second time in this notebook to keep runtime bounded, but every
# PipelineVariant-driven figure function below accepts a `variant` argument
# for that purpose.
#
# variant.out_s1 .. out_s4 (reads) always resolve under the real outputs/
# tree via audit_pipeline.config.WORKDIR, regardless of SANDBOX_MODE -- that
# is correct and desired, since those stages' outputs are read-only inputs
# here. Only out_s5 (the only stage5_figures.py *writes* to) is overridden
# to land under the sandbox when SANDBOX_MODE is on.
variant = VARIANT_FULL
if SANDBOX_MODE:
    variant = variant._replace(name="sandbox", out_s5=OUTPUTS_DIR / "stage5")


### Canary check -- prove the sandbox redirect is live before it matters

Writes a tiny throwaway figure through `variant.out_s5` -- the exact
path expression the real Stage 5 figure code below will use -- and
asserts it lands under the sandbox and that the corresponding real
production path (`outputs/stage5/level_stratified/_canary.png`) was
never touched. Also checks `GFF_DATA_DIR` resolves under the sandbox.
This runs before any real plotting code, so a failure here stops the
notebook before anything of consequence happens.


In [ ]:
_canary_real_path = _REAL_OUTPUTS_DIR / "stage5" / "level_stratified" / "_canary.png"
_canary_sandbox_path = variant.out_s5 / "level_stratified" / "_canary.png"
_canary_sandbox_path.parent.mkdir(parents=True, exist_ok=True)

assert not _canary_real_path.exists(), (
    f"CANARY SETUP INVALID: {_canary_real_path} already exists from a prior run -- "
    f"clean it up by hand before trusting this check."
)

_canary_fig = plt.figure()
_canary_fig.savefig(_canary_sandbox_path)
plt.close(_canary_fig)

assert _canary_sandbox_path.exists(), "CANARY FAILED: sandbox write did not land where expected."
assert not _canary_real_path.exists(), (
    f"CANARY FAILED: {_canary_real_path} exists -- the sandbox redirect is NOT "
    f"intercepting writes to the real production path. Stopping before any real "
    f"plotting code runs."
)
_canary_sandbox_path.unlink()

assert SANDBOX_MODE and str(SANDBOX_DIR) in GFF_DATA_DIR, (
    "CANARY FAILED: GFF_DATA_DIR does not resolve under the sandbox."
)

print(f"CANARY PASSED: writes land under {SANDBOX_DIR}, not under {_REAL_OUTPUTS_DIR}")
print(f"  variant.out_s5    = {variant.out_s5}")
print(f"  GFF_DATA_DIR      = {GFF_DATA_DIR}")


## `audit_pipeline/stage5_figures.py` -- shared utilities

Copied verbatim from `stage5_figures.py` (module-level style block, palette
constants, and helper functions), with a `_s5` suffix added to every helper
name (`_read` -> `_read_s5`, `_save` -> `_save_s5`, etc.) purely to avoid
colliding with `generate_figures_final.py`'s own same-named helpers, which
are semantically different (different DPI, different file format, different
`tight_layout` handling) and are kept separately below rather than merged --
merging them would be a behavior change, not a consolidation.

One line was added to `_save_s5` (not present in the original
`stage5_figures.py`): a `plt.show()` call right before `plt.close(fig)`, so
each figure renders inline in this notebook's cell output as it's produced.
Marked inline with an `# added:` comment. Nothing about the figure itself
(data, layout, threshold, colors) is touched.

**2026-07-03 style pass:** the six palette constants (`_BLUE`, `_GREEN`,
`_ORANGE`, `_RED`, `_PURPLE`, `_TEAL`) and the L2-L4 entries of `_CL_COLOR`
were changed to the exact hex values `generate_figures_final.py`'s
`STYLE["colors"]` already uses, so "correct"/"case A" is the same green,
"failure"/"case B"/"fail" is the same vermillion, and "presence"/"pass" is
the same blue in every figure in this notebook, not just within one figure
family. Only the literal hex values changed at their single definition
point -- every figure function below still references these same constant
names, so no data, threshold, sort, or filter logic moved. `L1` keeps its
own accent color since `generate_figures_final.py` figures never facet on
L1. See the constants cell below for the full old-vs-new mapping.


In [ ]:
# ── Global style ──────────────────────────────────────────────────────────────
mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans"],
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.labelsize": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.linewidth": 0.8,
        "axes.grid": True,
        "axes.grid.axis": "x",
        "grid.color": "#e0e0e0",
        "grid.linewidth": 0.6,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "legend.framealpha": 0.7,
        "legend.edgecolor": "#cccccc",
        "figure.facecolor": "white",
        "axes.facecolor": "#fafafa",
        "savefig.facecolor": "white",
    }
)

# Shared palettes
# 2026-07-03 style pass: aligned to generate_figures_final.py's STYLE["colors"]
# (Okabe-Ito colorblind-safe palette) so the same semantic category -- pass/
# presence, correct/case A, fail/failure/case B/negative-delta -- renders as
# the same hex wherever it appears across BOTH figure families, not just
# within one. Only the literal hex values changed; every call site below
# keeps referencing these same constant names, so no plotting logic moved.
_BLUE = "#0072B2"      # was #4C78A8 -- now matches STYLE["colors"]["presence"/"pass"]
_GREEN = "#009E73"     # was #54A24B -- now matches STYLE["colors"]["correct"/"case_a"]
_ORANGE = "#D55E00"    # was #F28E2B -- now matches STYLE["colors"]["failure"/"case_b"/"fail"]
_RED = "#D55E00"       # was #E15759 -- now matches STYLE["colors"]["fail"/"delta_neg"] (never
                       # co-occurs with _ORANGE in the same figure, so sharing this hex is safe)
_PURPLE = "#CC79A7"    # was #B279A2 -- now matches STYLE["colors"]["L4"]
_TEAL = "#56B4E9"      # was #72B7B2 -- now matches STYLE["colors"]["L3"/"type_cov"]

# Coding-level accent colors used for facet titles / highlights.
# L2-L4 aligned to generate_figures_final.py's STYLE["colors"]; L1 has no gff
# counterpart (gff figures never facet on L1) so it keeps its own accent.
_CL_COLOR: dict[str, str] = {
    "L1": "#E15759",  # s5-only level; no canonical gff color to align to
    "L2": "#E69F00",  # was #F28E2B -- now matches STYLE["colors"]["L2"]
    "L3": "#56B4E9",  # was #4C78A8 -- now matches STYLE["colors"]["L3"]
    "L4": "#CC79A7",  # was #B279A2 -- now matches STYLE["colors"]["L4"]
}


def _cl_color_s5(cl: str) -> str:
    """Return the accent color assigned to a coding level label."""
    return _CL_COLOR.get(str(cl), "#777777")

def _read_s5(path):
    """Read a TSV artifact, returning an empty frame when absent or empty."""
    try:
        return pd.read_csv(path, sep="\t", low_memory=False)
    except (FileNotFoundError, EmptyDataError):
        return pd.DataFrame()

def _save_s5(fig, path):
    """Save a figure with shared layout and export settings."""
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()  # added: display the figure inline in the notebook output before closing it
    plt.close(fig)

def _filter_self_ref_s5(df: pd.DataFrame) -> pd.DataFrame:
    """Drop self-referential dogwhistle rows when the flag is available."""
    if "is_self_referential" not in df.columns:
        return df.copy()
    return df[~df["is_self_referential"].fillna(False)].copy()

def _top_n_s5(df: pd.DataFrame, by: str, n: int, ascending: bool = False) -> pd.DataFrame:
    """Return the top ``n`` rows by a metric column if that column exists.

    ``ascending=False`` (default) selects the *highest* n values -- correct
    for metrics where higher = more extreme (token frequency, label gaps,
    etc.). Pass ``ascending=True`` for metrics where *lower* = worse (e.g.
    DI ratios), so "top n" means "n most disparate", not "n with the highest
    raw value".
    """
    if df.empty or by not in df.columns:
        return df
    return df.sort_values(by, ascending=ascending).head(n).copy()

def _style_ax_s5(ax, grid_axis: str = "x") -> None:
    """Remove top/right spines, set tick params."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(length=3, width=0.7)
    ax.grid(True, axis=grid_axis, color="#e0e0e0", linewidth=0.6, zorder=0)

def _facet_title_s5(ax, label: str) -> None:
    """Colour-coded panel title for coding-level facets."""
    color = _cl_color_s5(label)
    ax.set_title(label, color=color, fontweight="bold", fontsize=11, pad=6, loc="left")


### Level-stratified figures (`stage5_figures.level_stratified_figures`)

**Function:** `level_stratified_figures(level_dir, s1_dir, s2_dir, s3_dir)`

**Source TSVs read** (all via `variant.out_sN`, i.e. Stage 1-3 of
`audit_pipeline`, primary/"full" variant by default):
- `stage1/s1_coverage_by_level_target.tsv` -- **Stage 1**
- `stage2/s2_annotation_by_level_target.tsv` -- **Stage 2**
- `stage3/s3_coverage_disparity.tsv`, `stage3/s3_annotation_disparity.tsv`,
  `stage3/s3_cross_level_consistency.tsv` -- **Stage 3**

**Output PNGs** (under `variant.out_s5/level_stratified/`):
`s5_lev_coverage_presence_vs_type.png`, `s5_lev_coverage_token_frequency.png`,
`s5_lev_annotation_rates.png`, `s5_lev_annotation_case_ab.png`,
`s5_lev_disparity_di_histograms.png`, `s5_lev_disparity_worst_di.png`,
`s5_lev_annotation_label_gap.png`, `s5_lev_cross_level_deltas.png`.

Note the worst-DI figure here (`s5_lev_disparity_worst_di.png`) reads its
pairwise disparity numbers from **Stage 3** (`target_a`/`target_b` are raw,
un-collapsed taxonomy targets) -- contrast with the group-collapsed section
below, whose worst-DI figure reads **Stage 4a** (`by_group`) instead, and
with `generate_figures_final.py`'s equivalent figures (`fig4`/`fig5`), which
read **Stage 4b** (`by_level_group`). All three exist in this repo and none
of them agree on which source file is "the" pairwise DI source -- this is
exactly the kind of divergence Step 3 below checks numerically.

Previously contained the `_top_n_s5` sort-direction bug at the
`_top_n_s5(sub, "worst_di_ratio", 8)` call -- fixed: this call now
passes `ascending=True` so the n MOST disparate pairs are selected.


In [ ]:
def level_stratified_figures(
    level_dir: Path,
    s1_dir: Path,
    s2_dir: Path,
    s3_dir: Path,
) -> None:
    """Render the coding-level faceted figure set from Stages 1-3 outputs.

    Parameters
    ----------
    level_dir : Path
        Output directory for level-stratified figures.
    s1_dir, s2_dir, s3_dir : Path
        Input directories for the corresponding stage artifacts.
    """
    s1 = _filter_self_ref_s5(_read_s5(s1_dir / "s1_coverage_by_level_target.tsv"))
    s2 = _filter_self_ref_s5(_read_s5(s2_dir / "s2_annotation_by_level_target.tsv"))
    s3c = _read_s5(s3_dir / "s3_coverage_disparity.tsv")
    s3a = _read_s5(s3_dir / "s3_annotation_disparity.tsv")
    s3x = _read_s5(s3_dir / "s3_cross_level_consistency.tsv")

    def _coding_levels(df):
        """Return coding levels in paper order, dropping absent facets."""
        return [
            c
            for c in ["L1", "L2", "L3", "L4"]
            if c in set(df["coding_level"].dropna().astype(str))
        ]

    if not s1.empty:
        cls = _coding_levels(s1)

        # -- Coverage: presence rate + type coverage, faceted by coding level --
        fig, axes = plt.subplots(2, 2, figsize=(6.97, 7.2), sharey=False)  # was (14, 10); matches gff textwidth_in
        fig.suptitle(
            "Coverage — Presence Rate and Type Coverage by Target Group",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        for i, cl in enumerate(cls[:4]):
            d = _top_n_s5(
                s1[s1["coding_level"].astype(str) == cl], "token_frequency", 10
            ).copy()
            d["label"] = d["target"].astype(str)
            ax = axes.flatten()[i]
            y = range(len(d))
            ax.barh(
                y,
                d["presence_rate"],
                color=_cl_color_s5(cl),
                alpha=0.85,
                label="Presence rate",
                edgecolor="white",
                linewidth=0.5,
            )
            ax.barh(
                y,
                d["type_coverage"],
                color=_GREEN,
                alpha=0.55,
                label="Type coverage",
                edgecolor="white",
                linewidth=0.5,
            )
            ax.set_yticks(list(y))
            ax.set_yticklabels(d["label"], fontsize=8)
            ax.invert_yaxis()
            ax.set_xlim(0, 1.05)
            ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
            _facet_title_s5(ax, cl)
            _style_ax_s5(ax, grid_axis="x")
            ax.legend(loc="lower right", fontsize=7)
        for j in range(len(cls), 4):
            axes.flatten()[j].axis("off")
        _save_s5(fig, level_dir / "s5_lev_coverage_presence_vs_type.png")

        # -- Coverage: token frequency top-10, faceted by coding level --
        fig, axes = plt.subplots(2, 2, figsize=(6.97, 7.2), sharey=False)  # was (14, 10); matches gff textwidth_in
        fig.suptitle(
            "Coverage — Top-10 Token Frequency by Coding Level",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        for i, cl in enumerate(cls[:4]):
            d = _top_n_s5(s1[s1["coding_level"].astype(str) == cl], "token_frequency", 10)
            ax = axes.flatten()[i]
            ax.barh(
                d["target"].astype(str),
                d["token_frequency"],
                color=_cl_color_s5(cl),
                edgecolor="white",
                linewidth=0.5,
            )
            ax.invert_yaxis()
            ax.set_xlabel("Token frequency", fontsize=8)
            _facet_title_s5(ax, cl)
            _style_ax_s5(ax, grid_axis="x")
        for j in range(len(cls), 4):
            axes.flatten()[j].axis("off")
        _save_s5(fig, level_dir / "s5_lev_coverage_token_frequency.png")

    if not s2.empty:
        cls = _coding_levels(s2)

        # -- Annotation rates, faceted by coding level --
        fig, axes = plt.subplots(2, 2, figsize=(6.97, 7.5), sharey=False)  # was (15, 10); matches gff textwidth_in
        fig.suptitle(
            "Annotation Quality — Correct vs. Failure Rate by Coding Level",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        for i, cl in enumerate(cls[:4]):
            d = _top_n_s5(
                s2[s2["coding_level"].astype(str) == cl], "total_matches", 8
            ).copy()
            d["label"] = (
                d["target"].astype(str) + " (" + d["coding_level"].astype(str) + ")"
            )
            ax = axes.flatten()[i]
            x = range(len(d))
            bar_w = 0.38
            ax.bar(
                x,
                d["correct_labeling_rate"],
                width=bar_w,
                label="Correct",
                color=_GREEN,
                edgecolor="white",
                linewidth=0.5,
            )
            ax.bar(
                [t + bar_w for t in x],
                d["failure_rate"],
                width=bar_w,
                label="Failure",
                color=_ORANGE,
                edgecolor="white",
                linewidth=0.5,
            )
            ax.set_xticks([t + bar_w / 2 for t in x])
            ax.set_xticklabels(d["label"], rotation=40, ha="right", fontsize=8)
            ax.set_ylim(0, 1.05)
            ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
            ax.set_ylabel("Rate", fontsize=8)
            _facet_title_s5(ax, cl)
            _style_ax_s5(ax, grid_axis="y")
            ax.legend(loc="upper right")
        for j in range(len(cls), 4):
            axes.flatten()[j].axis("off")
        _save_s5(fig, level_dir / "s5_lev_annotation_rates.png")

        # -- Case A/B stacked, faceted by coding level --
        fig, axes = plt.subplots(2, 2, figsize=(6.97, 7.5), sharey=False)  # was (15, 10); matches gff textwidth_in
        fig.suptitle(
            "Annotation Quality — Case A vs. B Match Counts by Coding Level",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        for i, cl in enumerate(cls[:4]):
            d = _top_n_s5(
                s2[s2["coding_level"].astype(str) == cl], "total_matches", 8
            ).copy()
            d["label"] = (
                d["target"].astype(str) + " (" + d["coding_level"].astype(str) + ")"
            )
            ax = axes.flatten()[i]
            ax.barh(
                d["label"],
                d["case_a_present_hateful"],
                color=_GREEN,
                label="Case A (hateful)",
                edgecolor="white",
                linewidth=0.5,
            )
            ax.barh(
                d["label"],
                d["case_b_present_nonhateful"],
                left=d["case_a_present_hateful"],
                color=_ORANGE,
                label="Case B (non-hateful)",
                edgecolor="white",
                linewidth=0.5,
            )
            ax.invert_yaxis()
            ax.set_xlabel("Match count", fontsize=8)
            _facet_title_s5(ax, cl)
            _style_ax_s5(ax, grid_axis="x")
            ax.legend(loc="lower right")
        for j in range(len(cls), 4):
            axes.flatten()[j].axis("off")
        _save_s5(fig, level_dir / "s5_lev_annotation_case_ab.png")

    if not s3c.empty:
        cls = _coding_levels(s3c)

        # -- DI ratio histograms, faceted by coding level --
        fig, axes = plt.subplots(2, 2, figsize=(6.97, 7.2), sharey=False)  # was (14, 10); matches gff textwidth_in
        fig.suptitle(
            "Disparity — Presence & Type Coverage DI Ratio Distributions by Coding Level",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        for i, cl in enumerate(cls[:4]):
            d = s3c[s3c["coding_level"].astype(str) == cl]
            ax = axes.flatten()[i]
            ax.hist(
                d["presence_rate_di_ratio"].dropna(),
                bins=15,
                alpha=0.75,
                color=_BLUE,
                edgecolor="white",
                linewidth=0.5,
                label="Presence DI",
            )
            ax.hist(
                d["type_coverage_di_ratio"].dropna(),
                bins=15,
                alpha=0.55,
                color=_TEAL,
                edgecolor="white",
                linewidth=0.5,
                label="Type DI",
            )
            ax.axvline(
                DI_THRESHOLD,
                color=_RED,
                linestyle="--",
                linewidth=1.2,
                label=f"4/5 threshold ({DI_THRESHOLD})",
            )
            ax.set_xlabel("DI ratio", fontsize=8)
            ax.set_ylabel("Count", fontsize=8)
            _facet_title_s5(ax, cl)
            _style_ax_s5(ax, grid_axis="y")
            ax.legend()
        for j in range(len(cls), 4):
            axes.flatten()[j].axis("off")
        _save_s5(fig, level_dir / "s5_lev_disparity_di_histograms.png")

        # -- Worst DI by pair, faceted by coding level --
        fig, axes = plt.subplots(2, 2, figsize=(6.97, 7.2), sharey=False)  # was (14, 10); matches gff textwidth_in
        fig.suptitle(
            "Disparity — Worst Coverage DI Ratio by Target Pair and Coding Level",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        for i, cl in enumerate(cls[:4]):
            sub = s3c[s3c["coding_level"].astype(str) == cl].copy()
            sub["pair"] = (
                sub["target_a"].astype(str) + " vs " + sub["target_b"].astype(str)
            )
            # FIXED (see stage5_figures.py c92ea92-series fix): _top_n_s5() now takes
            # an explicit ascending= arg; "worst_di_ratio" (lower = more disparate =
            # worse) passes ascending=True so the n MOST disparate pairs are selected.
            d = _top_n_s5(sub, "worst_di_ratio", 8, ascending=True).sort_values(
                "worst_di_ratio", ascending=True
            )
            colors = [_RED if v < DI_THRESHOLD else _BLUE for v in d["worst_di_ratio"]]
            ax = axes.flatten()[i]
            ax.barh(
                d["pair"],
                d["worst_di_ratio"],
                color=colors,
                edgecolor="white",
                linewidth=0.5,
            )
            ax.axvline(
                DI_THRESHOLD,
                color=_RED,
                linestyle="--",
                linewidth=1.2,
                label=f"4/5 rule ({DI_THRESHOLD})",
            )
            ax.set_xlim(0, 1.05)
            ax.set_xlabel("Worst DI ratio", fontsize=8)
            _facet_title_s5(ax, cl)
            _style_ax_s5(ax, grid_axis="x")
            ax.legend(fontsize=7)
        for j in range(len(cls), 4):
            axes.flatten()[j].axis("off")
        _save_s5(fig, level_dir / "s5_lev_disparity_worst_di.png")

    if not s3a.empty:
        cls = _coding_levels(s3a)

        # -- Annotation label gap, faceted by coding level --
        fig, axes = plt.subplots(2, 2, figsize=(6.97, 7.2), sharey=False)  # was (14, 10); matches gff textwidth_in
        fig.suptitle(
            "Disparity — Annotation Labeling Rate Gap by Target Pair and Coding Level",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        for i, cl in enumerate(cls[:4]):
            sub = s3a[s3a["coding_level"].astype(str) == cl].copy()
            sub["pair"] = (
                sub["target_a"].astype(str) + " vs " + sub["target_b"].astype(str)
            )
            d = _top_n_s5(sub, "labeling_rate_gap_abs", 8).sort_values(
                "labeling_rate_gap_abs", ascending=True
            )
            ax = axes.flatten()[i]
            ax.barh(
                d["pair"],
                d["labeling_rate_gap_abs"],
                color=_RED,
                edgecolor="white",
                linewidth=0.5,
            )
            ax.set_xlabel("|Correct rate gap|", fontsize=8)
            ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
            _facet_title_s5(ax, cl)
            _style_ax_s5(ax, grid_axis="x")
        for j in range(len(cls), 4):
            axes.flatten()[j].axis("off")
        _save_s5(fig, level_dir / "s5_lev_annotation_label_gap.png")

    if not s3x.empty:
        d = s3x.copy()
        d["transition"] = (
            d["target"].astype(str)
            + " ("
            + d["taxonomy_level"].astype(str)
            + "): "
            + d["coding_level_from"].astype(str)
            + " \u2192 "
            + d["coding_level_to"].astype(str)
        )
        d = _top_n_s5(
            d.assign(abs_presence=d["presence_rate_delta"].abs()), "abs_presence", 20
        )
        fig, ax = plt.subplots(figsize=(6.97, 6.5))  # was (13, 8); matches gff textwidth_in
        fig.suptitle(
            "Cross-Level Consistency — Presence and Labeling Rate \u0394 by Coding Level",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        y = range(len(d))
        ax.barh(
            y,
            d["presence_rate_delta"],
            color=_BLUE,
            alpha=0.85,
            label="Presence \u0394",
            edgecolor="white",
            linewidth=0.5,
        )
        ax.barh(
            y,
            d["correct_labeling_rate_delta"],
            color=_GREEN,
            alpha=0.65,
            label="Labeling \u0394",
            edgecolor="white",
            linewidth=0.5,
        )
        ax.set_yticks(list(y))
        ax.set_yticklabels(d["transition"], fontsize=8)
        ax.axvline(0, color="black", linewidth=1)
        ax.set_xlabel("Delta", fontsize=9)
        _style_ax_s5(ax, grid_axis="x")
        ax.legend()
        _save_s5(fig, level_dir / "s5_lev_cross_level_deltas.png")


In [ ]:
level_stratified_figures(
    variant.out_s5 / "level_stratified",
    variant.out_s1,
    variant.out_s2,
    variant.out_s3,
)


### Group-collapsed figures (`stage5_figures.group_collapsed_figures`)

**Function:** `group_collapsed_figures(group_dir, s4_dir)`

**Source TSVs read** (mixed within a single function -- see note below):
- `stage4/by_level_group/s4b_coverage_by_level_group.tsv` -- **Stage 4b**
  (used for the coverage heatmap only)
- `stage4/by_group/s4a_annotation_by_group.tsv` -- **Stage 4a**
  (annotation rates + Case A/B counts)
- `stage4/by_group/s4a_pairwise_disparity_by_group.tsv` -- **Stage 4a**
  (DI histograms + worst-DI/label-gap panel)

**Output PNGs** (under `variant.out_s5/group_collapsed/`):
`s5_grp_coverage_scatter.png`, `s5_grp_annotation_rates.png`,
`s5_grp_case_ab_counts.png`, `s5_grp_di_histograms.png`,
`s5_grp_worst_di_and_label_gap.png`.

**This is the exact ambiguity called out in the task brief**: the coverage
heatmap in this same function reads Stage 4**b** (`by_level_group` -- the
file a reviewer checking taxonomy-preserving numbers would likely reach for
first), while the pairwise-disparity figures three lines later, in the same
function, read Stage 4**a** (`by_group`) instead -- and, per the section
above, the level-stratified pairwise figure reads **Stage 3**. Three
different pairwise-DI sources across two sections of this one file.

Previously contained the `_top_n_s5` sort-direction bug at
`_top_n_s5(pair_plot, "worst_di_ratio", 20)` -- fixed: this call now
passes `ascending=True` so the n MOST disparate pairs are selected.


In [ ]:
def group_collapsed_figures(group_dir: Path, s4_dir: Path) -> None:
    """Render reporting-group figures from Stage 4 collapsed outputs.

    Parameters
    ----------
    group_dir : Path
        Output directory for group-collapsed figures.
    s4_dir : Path
        Input directory containing Stage 4 artifacts.
    """
    # Use by-level-group coverage so duplicated taxonomy mappings can be deduplicated
    # explicitly by report group + coding level for a stable heatmap matrix.
    cov = _read_s5(s4_dir / "by_level_group/s4b_coverage_by_level_group.tsv")
    ann = _filter_self_ref_s5(_read_s5(s4_dir / "by_group/s4a_annotation_by_group.tsv"))
    pair = _read_s5(s4_dir / "by_group/s4a_pairwise_disparity_by_group.tsv")

    if not cov.empty:
        d = cov.copy()
        d["group_label"] = (
            d["report_level"].astype(str).str.strip()
            + ": "
            + d["report_target"].astype(str).str.strip()
        )
        d["coding_level"] = d["coding_level"].astype(str).str.strip()

        # Keep only primary coding levels used in paper-facing visuals.
        d = d[d["coding_level"].isin(["L2", "L3", "L4"])].copy()
        # Some groups may appear in multiple taxonomy rows; keep first group-level row.
        d = d.drop_duplicates(subset=["group_label", "coding_level"], keep="first")

        group_order = [
            "disability: unspecific",
            "gender: men",
            "gender: women",
            "lgbtq: LGB",
            "lgbtq: Trans/NB",
            "origin: specific country",
            "origin: undocumented",
            "origin: immigrant",
            "origin: migrant worker",
            "politics: communist",
            "politics: democrat",
            "politics: libertarian",
            "politics: leftist",
            "politics: liberal",
            "politics: conservative",
            "politics: republican",
            "race: asian",
            "race: black",
            "race: latinx",
            "race: middle eastern",
            "race: white",
            "religion: jewish",
            "religion: muslim",
        ]
        existing_groups = set(d["group_label"].astype(str))
        group_labels = [g for g in group_order if g in existing_groups]
        remaining = sorted(existing_groups - set(group_labels))
        group_labels.extend(remaining)
        coding_levels = ["L2", "L3", "L4"]

        presence_mat = d.pivot(
            index="group_label", columns="coding_level", values="presence_rate"
        ).reindex(index=group_labels, columns=coding_levels)
        type_mat = d.pivot(
            index="group_label", columns="coding_level", values="type_coverage"
        ).reindex(index=group_labels, columns=coding_levels)

        # Distinguish structural missing (NaN) from true 0% presence cells.
        cmap = LinearSegmentedColormap.from_list(
            "presence_rate",
            [
                (0.0, "#CBD4D0"),
                (0.35, "#8FC6A2"),
                (0.7, "#3B956F"),
                (1.0, "#14523A"),
            ],
        )
        cmap.set_bad(color="#E8E8E8")

        fig_h = max(7, len(group_labels) * 0.42 + 2.8)
        fig, ax = plt.subplots(figsize=(6.97, fig_h))  # was 8.4; matches gff textwidth_in
        fig.suptitle(
            "Coverage Heatmap — Presence Rate by Reporting Group × Coding Level",
            fontsize=13,
            fontweight="bold",
            y=0.99,
        )

        data = presence_mat.values.astype(float)
        masked = np.ma.array(data, mask=np.isnan(data))
        im = ax.imshow(
            masked,
            aspect="auto",
            cmap=cmap,
            vmin=0.0,
            vmax=1.0,
            interpolation="none",
        )

        flagged = []
        for r, grp in enumerate(group_labels):
            for c, cl in enumerate(coding_levels):
                pr = presence_mat.loc[grp, cl]
                tc = type_mat.loc[grp, cl]

                if pd.isna(pr):
                    ax.text(
                        c,
                        r,
                        "-",
                        ha="center",
                        va="center",
                        fontsize=8,
                        color="#8a8a8a",
                    )
                    continue

                has_type_gap = pd.notna(tc) and 0.0 < tc < 0.999
                txt_color = "white" if pr > 0.5 else "#1a1a1a"
                label = f"{pr:.0%}"
                if has_type_gap:
                    label = label + "$^{*}$"
                ax.text(
                    c,
                    r,
                    label,
                    ha="center",
                    va="center",
                    fontsize=8.5,
                    fontweight="bold",
                    color=txt_color,
                )

                if has_type_gap:
                    flagged.append(
                        f"{grp} / {cl}: type_coverage={tc:.3f}, presence_rate={pr:.3f}"
                    )
                    ax.add_patch(
                        mpatches.FancyBboxPatch(
                            (c - 0.47, r - 0.47),
                            0.94,
                            0.94,
                            boxstyle="square,pad=0",
                            linewidth=1.9,
                            edgecolor="#C0392B",
                            facecolor="none",
                            zorder=3,
                        )
                    )

        ax.set_xticks(range(len(coding_levels)))
        ax.set_xticklabels(
            [
                "L2\n(Stereotype-based)",
                "L3\n(Concept / policy)",
                "L4\n(Persona signals)",
            ],
            fontsize=8.5,
        )
        ax.xaxis.set_ticks_position("top")
        ax.xaxis.set_label_position("top")
        ax.set_yticks(range(len(group_labels)))
        ax.set_yticklabels(group_labels, fontsize=8.2)
        ax.tick_params(axis="both", which="both", length=0)
        ax.grid(False)

        cbar = fig.colorbar(im, ax=ax, fraction=0.032, pad=0.01, aspect=30)
        cbar.set_label("Presence rate", fontsize=8.5, labelpad=6)
        cbar.set_ticks([0.0, 0.25, 0.5, 0.75, 1.0])
        cbar.ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        cbar.ax.tick_params(labelsize=8)

        # Taxonomy section lines and right-side labels.
        section_order = [
            ("disability", "Disability"),
            ("gender", "Gender / LGBTQ"),
            ("lgbtq", "Gender / LGBTQ"),
            ("origin", "Origin"),
            ("politics", "Politics"),
            ("race", "Race"),
            ("religion", "Religion"),
        ]
        section_key_rank = {k: i for i, (k, _) in enumerate(section_order)}

        def _section_key(label: str) -> str:
            return str(label).split(":", 1)[0].strip()

        section_rows = {}
        for i, label in enumerate(group_labels):
            k = _section_key(label)
            section_rows.setdefault(k, []).append(i)

        ordered_sections = sorted(
            [(k, rows) for k, rows in section_rows.items()],
            key=lambda x: section_key_rank.get(x[0], 999),
        )

        # Draw lines between contiguous section blocks.
        for idx_sec in range(len(ordered_sections) - 1):
            last_row = max(ordered_sections[idx_sec][1])
            ax.axhline(last_row + 0.5, color="white", linewidth=1.5, zorder=4)

        ax2 = ax.twinx()
        ax2.set_ylim(ax.get_ylim())
        ax2.set_yticks([])
        section_display = {k: disp for k, disp in section_order}
        for k, rows in ordered_sections:
            mid = (min(rows) + max(rows)) / 2.0
            y = 1 - (mid / max(1, len(group_labels) - 1))
            ax2.text(
                1.02,
                y,
                section_display.get(k, k.title()),
                transform=ax2.transAxes,
                va="center",
                ha="left",
                fontsize=7.5,
                color="#555555",
            )

        legend_handles = [
            mpatches.Patch(
                facecolor="#E8E8E8",
                edgecolor="#cfcfcf",
                label="No glossary entries at this level",
            ),
            mpatches.Patch(
                facecolor="#CBD4D0",
                edgecolor="none",
                label="0% presence (entries exist, none found)",
            ),
            mpatches.FancyBboxPatch(
                (0, 0),
                1,
                1,
                boxstyle="square,pad=0",
                linewidth=1.5,
                edgecolor="#C0392B",
                facecolor="#d8d8d8",
                label="Type coverage < 100% (border + *)",
            ),
        ]
        ax.legend(
            handles=legend_handles,
            loc="lower left",
            bbox_to_anchor=(0, -0.13),
            fontsize=7.5,
            framealpha=0.9,
            ncol=1,
            handlelength=1.3,
            borderpad=0.6,
        )

        note = (
            "Cells marked * have partial type coverage (0 < type_coverage < 1.0). "
            "Gray cells are no-glossary combinations."
        )
        fig.text(
            0.012, 0.01, note, ha="left", va="bottom", fontsize=7.8, color="#555555"
        )

        _save_s5(fig, group_dir / "s5_grp_coverage_scatter.png")

    if not ann.empty:
        ann_plot = ann[
            (ann["report_level"] != "disability")
            | (ann["report_target"] == "unspecific")
        ].copy()
        ann_plot["label"] = (
            ann_plot["report_level"].astype(str)
            + ": "
            + ann_plot["report_target"].astype(str)
            + " ("
            + ann_plot["coding_level"].astype(str)
            + ")"
        )
        ann_plot = ann_plot.sort_values("label")

        # -- Annotation rates bar chart --
        fig, ax = plt.subplots(figsize=(6.97, 6))  # was (15, 6); matches gff textwidth_in
        fig.suptitle(
            "Annotation Quality — Correct vs. Failure Rate by Reporting Group",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        x = range(len(ann_plot))
        bar_w = 0.38
        ax.bar(
            x,
            ann_plot["correct_labeling_rate"],
            width=bar_w,
            color=_GREEN,
            label="Correct",
            edgecolor="white",
            linewidth=0.5,
        )
        ax.bar(
            [v + bar_w for v in x],
            ann_plot["failure_rate"],
            width=bar_w,
            color=_ORANGE,
            label="Failure",
            edgecolor="white",
            linewidth=0.5,
        )
        ax.set_xticks([v + bar_w / 2 for v in x])
        ax.set_xticklabels(ann_plot["label"], rotation=45, ha="right", fontsize=7.5)
        ax.set_ylim(0, 1.05)
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        ax.set_ylabel("Rate", fontsize=9)
        _style_ax_s5(ax, grid_axis="y")
        ax.legend()
        _save_s5(fig, group_dir / "s5_grp_annotation_rates.png")

        # -- Case A/B stacked horizontal bar --
        d = _top_n_s5(ann_plot, "case_b_present_nonhateful", 20).sort_values(
            "case_b_present_nonhateful", ascending=True
        )
        fig, ax = plt.subplots(figsize=(6.97, max(6, 1 + 0.35 * len(d))))  # was width 13; matches gff textwidth_in
        fig.suptitle(
            "Annotation Quality — Case A vs. B Match Counts by Reporting Group",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        ax.barh(
            d["label"],
            d["case_a_present_hateful"],
            color=_GREEN,
            label="Case A (hateful)",
            edgecolor="white",
            linewidth=0.5,
        )
        ax.barh(
            d["label"],
            d["case_b_present_nonhateful"],
            left=d["case_a_present_hateful"],
            color=_ORANGE,
            label="Case B (non-hateful)",
            edgecolor="white",
            linewidth=0.5,
        )
        ax.set_xlabel("Match count", fontsize=9)
        _style_ax_s5(ax, grid_axis="x")
        ax.legend()
        _save_s5(fig, group_dir / "s5_grp_case_ab_counts.png")

    if not pair.empty:
        # -- DI ratio histograms --
        fig, axes = plt.subplots(1, 2, figsize=(6.97, 4.5))  # was (13, 5); matches gff textwidth_in
        fig.suptitle(
            "Disparity — Pairwise Coverage DI Ratio Distributions by Reporting Group",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        for ax, col, label, color in [
            (axes[0], "presence_rate_di_ratio", "Presence DI", _BLUE),
            (axes[1], "type_coverage_di_ratio", "Type-Coverage DI", _TEAL),
        ]:
            ax.hist(
                pair[col].dropna(),
                bins=20,
                alpha=0.85,
                color=color,
                edgecolor="white",
                linewidth=0.5,
            )
            ax.axvline(
                DI_THRESHOLD,
                color=_RED,
                linestyle="--",
                linewidth=1.2,
                label="4/5 threshold",
            )
            ax.set_xlabel("DI ratio", fontsize=8)
            ax.set_ylabel("Count", fontsize=8)
            ax.set_title(label, fontweight="bold")
            _style_ax_s5(ax, grid_axis="y")
            ax.legend(fontsize=7)
        _save_s5(fig, group_dir / "s5_grp_di_histograms.png")

        # -- Worst DI + labeling gap panel --
        pair_plot = pair.copy()
        pair_plot["pair"] = (
            pair_plot["target_a"].astype(str)
            + " vs "
            + pair_plot["target_b"].astype(str)
        )
        # FIXED (see stage5_figures.py c92ea92-series fix): _top_n_s5() now takes
        # an explicit ascending= arg; "worst_di_ratio" (lower = more disparate =
        # worse) passes ascending=True so the n MOST disparate pairs are selected.
        left = _top_n_s5(pair_plot, "worst_di_ratio", 20, ascending=True).sort_values(
            "worst_di_ratio", ascending=True
        )
        right = _top_n_s5(pair_plot, "labeling_rate_gap_abs", 20).sort_values(
            "labeling_rate_gap_abs", ascending=True
        )

        fig, axes = plt.subplots(1, 2, figsize=(6.97, 7.5))  # was (17, 10); matches gff textwidth_in
        fig.suptitle(
            "Disparity — Worst Pairwise DI and Annotation Gap by Reporting Group",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        left_colors = [
            _RED if v < DI_THRESHOLD else _BLUE for v in left["worst_di_ratio"]
        ]
        axes[0].barh(
            left["pair"],
            left["worst_di_ratio"],
            color=left_colors,
            edgecolor="white",
            linewidth=0.5,
        )
        axes[0].axvline(
            DI_THRESHOLD,
            color=_RED,
            linestyle="--",
            linewidth=1.2,
            label=f"4/5 rule ({DI_THRESHOLD})",
        )
        axes[0].set_xlabel("Worst DI ratio", fontsize=8)
        axes[0].set_title("Worst DI pairs", fontweight="bold")
        _style_ax_s5(axes[0], grid_axis="x")
        axes[0].legend(fontsize=7)

        axes[1].barh(
            right["pair"],
            right["labeling_rate_gap_abs"],
            color=_RED,
            edgecolor="white",
            linewidth=0.5,
        )
        axes[1].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        axes[1].set_xlabel("|Correct rate gap|", fontsize=8)
        axes[1].set_title("Largest labeling gaps", fontweight="bold")
        _style_ax_s5(axes[1], grid_axis="x")
        _save_s5(fig, group_dir / "s5_grp_worst_di_and_label_gap.png")


In [ ]:
group_collapsed_figures(variant.out_s5 / "group_collapsed", variant.out_s4)


### ElSherief delta figures (`stage5_figures.elsherief_figures`)

**Function:** `elsherief_figures(els_dir, s4_dir)`

**Source TSVs read** (all **Stage 4c**, the ElSherief-subset delta rollup):
- `stage4/elsherief/s4c_coverage_delta_union_vs_elsherief.tsv`
- `stage4/elsherief/s4c_annotation_delta_union_vs_elsherief.tsv`
- `stage4/elsherief/s4c_pairwise_delta_union_vs_elsherief.tsv`

**Output PNGs** (under `variant.out_s5/elsherief/`):
`s5_els_coverage_delta.png`, `s5_els_annotation_delta.png`,
`s5_els_pairwise_delta.png`.


In [ ]:
def elsherief_figures(els_dir: Path, s4_dir: Path) -> None:
    """Render delta figures comparing the union benchmark to ElSherief.

    Parameters
    ----------
    els_dir : Path
        Output directory for ElSherief delta figures.
    s4_dir : Path
        Input directory containing Stage 4 ElSherief artifacts.
    """
    cov = _read_s5(s4_dir / "elsherief/s4c_coverage_delta_union_vs_elsherief.tsv")
    ann = _read_s5(s4_dir / "elsherief/s4c_annotation_delta_union_vs_elsherief.tsv")
    pair = _read_s5(s4_dir / "elsherief/s4c_pairwise_delta_union_vs_elsherief.tsv")

    def _diverging_bar(ax, labels, values, xlabel):
        """Draw a horizontal bar chart where sign is encoded by color."""
        colors = [_GREEN if v >= 0 else _RED for v in values.fillna(0)]
        ax.barh(labels, values, color=colors, edgecolor="white", linewidth=0.5)
        ax.axvline(0, color="black", linewidth=1)
        ax.set_xlabel(xlabel, fontsize=9)
        _style_ax_s5(ax, grid_axis="x")

    if not cov.empty:
        d = cov.copy()
        d["label"] = (
            d["report_level"].astype(str) + ": " + d["report_target"].astype(str)
        )
        d = d.sort_values("presence_rate_delta_union_minus_elsherief", ascending=True)
        fig, ax = plt.subplots(figsize=(6.97, max(5, 1 + 0.3 * len(d))))
        fig.suptitle(
            "ElSherief Subset — Presence Rate \u0394 (Union \u2212 ElSherief)",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        _diverging_bar(
            ax,
            d["label"],
            d["presence_rate_delta_union_minus_elsherief"],
            "Presence rate \u0394",
        )
        _save_s5(fig, els_dir / "s5_els_coverage_delta.png")

    if not ann.empty:
        d = ann.copy()
        d["label"] = (
            d["report_level"].astype(str) + ": " + d["report_target"].astype(str)
        )
        d = d.sort_values(
            "correct_labeling_rate_delta_union_minus_elsherief", ascending=True
        )
        fig, ax = plt.subplots(figsize=(6.97, max(5, 1 + 0.3 * len(d))))
        fig.suptitle(
            "ElSherief Subset — Labeling Rate \u0394 (Union \u2212 ElSherief)",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        _diverging_bar(
            ax,
            d["label"],
            d["correct_labeling_rate_delta_union_minus_elsherief"],
            "Correct labeling rate \u0394",
        )
        _save_s5(fig, els_dir / "s5_els_annotation_delta.png")

    if not pair.empty:
        dx = pair["worst_di_ratio_delta_union_minus_elsherief"]
        dy = pair["label_gap_abs_delta_union_minus_elsherief"]
        fig, ax = plt.subplots(figsize=(5.0, 5.0))  # was (9, 7); scaled toward columnwidth-ish scatter
        fig.suptitle(
            "ElSherief Subset — Pairwise DI \u0394 vs. Label Gap \u0394",
            fontsize=13,
            fontweight="bold",
            y=0.98,
        )
        ax.scatter(
            dx, dy, alpha=0.75, s=50, c=_BLUE, edgecolors="#2c5f8a", linewidths=0.5
        )
        ax.axvline(0, color="#888888", linewidth=0.9, linestyle="--")
        ax.axhline(0, color="#888888", linewidth=0.9, linestyle="--")
        ax.set_xlabel("Worst DI \u0394 (union \u2212 ElSherief)", fontsize=9)
        ax.set_ylabel("|Label gap| \u0394 (union \u2212 ElSherief)", fontsize=9)
        _style_ax_s5(ax, grid_axis="both")
        _save_s5(fig, els_dir / "s5_els_pairwise_delta.png")


In [ ]:
elsherief_figures(variant.out_s5 / "elsherief", variant.out_s4)


## `generate_figures_final.py` -- shared utilities

Copied verbatim: the `STYLE` dict, `apply_style()`, `_read()`, `_need()`,
`_save()`, `_panel_label()`, and `_declutter_labels()` (used only by
`appF_cross_level_deltas`). These are the ACL/EMNLP camera-ready styling
helpers -- distinct from, and **not** merged with, the `_s5`-suffixed
helpers above (different DPI: 300 vs 220; different format: PDF vs PNG;
different `tight_layout`/`constrained_layout` handling). Every figure
function below calls these under their original (unsuffixed) names, exactly
as in the source file.

As with `_save_s5` above, one line was added to `_save` that is not present
in the original `generate_figures_final.py`: a `plt.show()` call right
before `plt.close(fig)`, so each figure renders inline in this notebook's
cell output. Marked with an `# added:` comment; no figure content changed.


In [ ]:
STYLE = {
    "title": None,
    "axis_label_fontsize": 10,
    "tick_label_fontsize": 8,
    "legend_fontsize": 8,
    "annotation_fontsize": 8,
    "panel_label_fontsize": 10,
    "colors": {
        "L2": "#E69F00",
        "L3": "#56B4E9",
        "L4": "#CC79A7",
        "correct": "#009E73",
        "failure": "#D55E00",
        "case_a": "#009E73",
        "case_b": "#D55E00",
        "presence": "#0072B2",
        "type_cov": "#56B4E9",
        "pass": "#0072B2",
        "fail": "#D55E00",
        "delta_pos": "#009E73",
        "delta_neg": "#D55E00",
        "neutral": "#999999",
    },
    "dpi": 300,
    "format": "pdf",
    "bbox_inches": "tight",
    "pad_inches": 0.05,
    "columnwidth_in": 3.35,
    "textwidth_in": 6.97,
}

def apply_style():
    """Apply ACL/EMNLP rcParams at the start of every figure function."""
    mpl.rcParams.update(
        {
            "font.size": STYLE["tick_label_fontsize"],
            "axes.labelsize": STYLE["axis_label_fontsize"],
            "xtick.labelsize": STYLE["tick_label_fontsize"],
            "ytick.labelsize": STYLE["tick_label_fontsize"],
            "legend.fontsize": STYLE["legend_fontsize"],
            "axes.spines.top": False,
            "axes.spines.right": False,
            "figure.dpi": STYLE["dpi"],
            # 2026-07-03: explicitly reset -- stage5_figures.py's utilities cell
            # (run earlier in this notebook, same kernel) sets these globally via
            # its own mpl.rcParams.update() and apply_style() never reset them
            # back, so every gff figure was silently inheriting stage5's grid
            # lines + off-white background. Visible as three vertical white grid
            # lines through fig1_coverage_heatmap's three columns.
            "axes.grid": False,
            "axes.facecolor": "white",
        }
    )

def _read(path: str | Path) -> pd.DataFrame:
    """Read a TSV file; return empty DataFrame with a warning if missing."""
    p = Path(path)
    if not p.exists():
        print(f"WARNING: Missing file {p}")
        return pd.DataFrame()
    return pd.read_csv(p, sep="\t", low_memory=False)

def _need(paths: list[str | Path], fig_name: str) -> bool:
    """Return True iff every path exists; print warnings for any missing."""
    missing = [p for p in paths if not Path(p).exists()]
    for m in missing:
        print(f"WARNING: Skipping {fig_name}: missing {m}")
    return len(missing) == 0

def _save(fig: plt.Figure, path: str | Path) -> None:
    """Save figure as PDF with ACL standard settings, then close.

    Calls tight_layout only for figures that do NOT use constrained_layout
    (constrained_layout handles its own spacing; calling tight_layout on top
    of it overrides it and breaks legend placement).
    """
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    if not fig.get_constrained_layout():
        # fig1 uses manual add_axes; tight_layout skips incompatible axes silently
        try:
            fig.tight_layout(pad=0.4)
        except Exception:
            pass
    fig.savefig(str(p), format="pdf", bbox_inches="tight", pad_inches=0.08)
    plt.show()  # added: display the figure inline in the notebook output before closing it
    plt.close(fig)
    print(f"Generated: {p}")

def _panel_label(ax: plt.Axes, label: str, color: str) -> None:
    """Place a level label (L2 / L3 / L4) just above the upper-left corner."""
    ax.text(
        0.03,
        1.01,
        label,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=STYLE["panel_label_fontsize"],
        fontweight="bold",
        color=color,
        clip_on=False,
    )

def _declutter_labels(
    ax: plt.Axes,
    fig: plt.Figure,
    end_points: list[tuple[str, float, float, str]],
    fontsize: float,
    x_pad_px: float = 6.0,
) -> None:
    """Place a direct end-of-line label per group, greedily decluttered in
    pixel space so labels closer than ~one line-height collapse into a
    vertically stacked, non-overlapping column.

    end_points: list of (label_text, x_final_data, y_final_data, color).
    Decluttering must happen in *display* pixels, not data coordinates,
    because the data only spans [0, 1] while actual visual separation is
    governed by rendered font size at the figure's draw DPI.
    """
    if not end_points:
        return

    fig.canvas.draw()
    dpi = fig.dpi
    # One line-height at the render DPI, plus a little leading, is the
    # minimum gap that avoids glyphs from adjacent labels touching.
    min_gap_px = fontsize * (dpi / 72.0) * 1.7  # was 1.25 -- more breathing room (2026-07-03 style pass, appF label crowding)
    edge_margin_px = min_gap_px * 0.4

    bbox = ax.get_window_extent()
    top_bound = bbox.y1 - edge_margin_px
    bottom_bound = bbox.y0 + edge_margin_px

    items = []
    for label_text, x_final, y_final, color in end_points:
        px, py = ax.transData.transform((x_final, y_final))
        items.append([label_text, x_final, y_final, color, px, py])

    # Top to bottom on screen = descending pixel y.
    items.sort(key=lambda it: -it[5])

    # Pass 1 (top-down): push each label below the one above it, clamped to
    # the axes' top edge.
    resolved_py = []
    prev_py = None
    for it in items:
        py = min(it[5], top_bound)
        cur = py if prev_py is None else min(py, prev_py - min_gap_px)
        resolved_py.append(cur)
        prev_py = cur

    # Pass 2 (bottom-up): if the stack ran past the bottom edge, pull labels
    # back up so the whole column fits within the axes.
    if resolved_py[-1] < bottom_bound:
        resolved_py[-1] = bottom_bound
        for i in range(len(resolved_py) - 2, -1, -1):
            resolved_py[i] = max(resolved_py[i], resolved_py[i + 1] + min_gap_px)

    inv = ax.transData.inverted()
    for it, py_resolved in zip(items, resolved_py):
        label_text, x_final, y_final, color, px, _py = it
        x_label, y_label = inv.transform((px + x_pad_px, py_resolved))
        ax.annotate(
            label_text,
            xy=(x_final, y_final),
            xycoords="data",
            xytext=(x_label, y_label),
            textcoords="data",
            ha="left",
            va="center",
            fontsize=fontsize,
            color=color,
            clip_on=False,
            zorder=4,
        )


## "Camera-ready" figures (`generate_figures_final.py`)

13 standalone figure functions (`fig1`-`fig7`, `appA`-`appG`), each already
documenting its own source file(s) in its docstring -- reproduced below
verbatim. This script is **not** `PipelineVariant`-aware: every function
takes a plain `data_dir` string and always reads/writes under it, so it can
only ever target the primary ("full") analysis outputs, never the tier1+2
robustness check. That is a genuine capability gap versus `stage5_figures.py`,
noted here rather than silently patched.


### `fig1_coverage_heatmap`

**Output file:** `outputs/figures_final/fig1_coverage_heatmap.pdf`

**Source TSV(s):** `outputs/stage1/s1_coverage_by_level_target.tsv`, `outputs/unioned_data/06_glossary_label_reference.tsv`

**Pipeline stage:** Stage 1 (`stage1/s1_coverage_by_level_target.tsv`) + upstream preprocessing output (`unioned_data/06_glossary_label_reference.tsv`, produced before Stage 1 by the `data_preprocessing/` notebooks)

**Docstring (verbatim from source):**

```
Heatmap of dogwhistle presence_rate for each (reporting group × coding level).

Source files:
- stage1/s1_coverage_by_level_target.tsv  : presence_rate, type_coverage per cell
- unioned_data/06_glossary_label_reference.tsv : confirms entry existence per cell

Visual encoding:
- Sequential green (light→dark) = presence_rate 0→1
- Gray cell = no glossary entries at that coding level for that group
- Red border + asterisk in cell text = type_coverage < 1.0
- Thin black separator lines between reporting dimension groups
- Right-side italic dimension label per group
```


In [ ]:
def fig1_coverage_heatmap(data_dir: str = "outputs/") -> None:
    """
    Heatmap of dogwhistle presence_rate for each (reporting group × coding level).

    Source files:
    - stage1/s1_coverage_by_level_target.tsv  : presence_rate, type_coverage per cell
    - unioned_data/06_glossary_label_reference.tsv : confirms entry existence per cell

    Visual encoding:
    - Sequential green (light→dark) = presence_rate 0→1
    - Gray cell = no glossary entries at that coding level for that group
    - Red border + asterisk in cell text = type_coverage < 1.0
    - Thin black separator lines between reporting dimension groups
    - Right-side italic dimension label per group
    """
    apply_style()
    base = Path(data_dir)
    s1_p = base / "stage1/s1_coverage_by_level_target.tsv"
    gl_p = base / "unioned_data/06_glossary_label_reference.tsv"
    if not _need([s1_p, gl_p], "fig1"):
        return

    s1 = _read(s1_p)
    if s1.empty:
        return

    # Map taxonomy_level → display dimension; filter to the 7 reported dimensions
    DIM_MAP = {
        "race": "race",
        "religion": "religion",
        "sexuality": "lgbtq",
        "gender": "gender",
        "origin": "origin",
        "politics": "politics",
        "disability": "disability",
    }
    DIM_ORDER = [
        "race",
        "religion",
        "lgbtq",
        "gender",
        "origin",
        "politics",
        "disability",
    ]
    LEVELS = ["L2", "L3", "L4"]
    COL_LABELS = ["L2", "L3", "L4"]

    s1 = s1[s1["taxonomy_level"].isin(DIM_MAP)].copy()
    # Explicit target exclusions: self-referential and pan-group categories excluded
    # from pipeline reporting (report_include = False).  The taxonomy_level filter
    # handles most cases, but defence-in-depth avoids silent leakage if taxonomy
    # assignment changes in a future pipeline run.
    _EXCLUDE = {"referential_white_supremacist", "minority"}
    s1 = s1[~s1["target"].isin(_EXCLUDE)].copy()
    s1["dimension"] = s1["taxonomy_level"].map(DIM_MAP)

    # Build ordered row list: DIM_ORDER, then alphabetical within each dimension
    rows: list[tuple[str, str]] = []  # (target, dimension)
    dim_ranges: dict[str, tuple[int, int]] = {}
    for dim in DIM_ORDER:
        targets = sorted(s1[s1["dimension"] == dim]["target"].unique())
        start = len(rows)
        rows.extend((t, dim) for t in targets)
        dim_ranges[dim] = (start, len(rows))

    n_rows, n_cols = len(rows), len(LEVELS)

    # Build grids (indexed [row, col])
    pr_grid = np.full((n_rows, n_cols), np.nan)
    tc_grid = np.full((n_rows, n_cols), np.nan)
    has_entry = np.zeros((n_rows, n_cols), dtype=bool)

    for i, (target, _) in enumerate(rows):
        for j, level in enumerate(LEVELS):
            mask = (s1["target"] == target) & (s1["coding_level"] == level)
            sub = s1[mask]
            if not sub.empty:
                has_entry[i, j] = True
                pr_grid[i, j] = float(sub["presence_rate"].iloc[0])
                tc_grid[i, j] = float(sub["type_coverage"].iloc[0])

    # Custom green colormap: clipped so 0% maps to light (not white) green
    base_cmap = mpl.colormaps.get_cmap("Greens")
    greens = LinearSegmentedColormap.from_list(
        "greens_clipped",
        [
            base_cmap(0.15),
            base_cmap(0.38),
            base_cmap(0.62),
            base_cmap(0.85),
            base_cmap(1.0),
        ],
    )
    greens.set_bad(color="#cccccc")  # Gray for missing cells

    pr_masked = np.ma.masked_where(~has_entry, pr_grid)

    # Figure layout: main heatmap + colorbar below
    cell_h = 0.36  # inches per row
    cbar_h = 0.55  # inches for colorbar + legend area
    main_h = n_rows * cell_h
    fig_h = main_h + cbar_h + 0.5
    fig_w = fig_h * 0.85  # was STYLE["textwidth_in"] * 0.67 (fixed, independent of content height, giving an extreme ~0.35:1 final aspect); tied to fig_h for a 2:3 (width:height) FINAL aspect per author request. Empirically calibrated: bbox_inches="tight" trims unused right-side margin at this nominal figsize, so a plain fig_h*2/3 nominal width under-shoots the target 2:3 in the saved PDF -- 0.85 hits ~2:3 after that trim. .tex embed width (0.67\textwidth) is unchanged, so the wider source makes the final printed height shrink to a reasonable single-page size instead of the previous ~13in.

    fig = plt.figure(figsize=(fig_w, fig_h))

    # Reserve proportional space: [left, bottom, width, height] in figure fraction
    left_frac = 0.22
    right_pad = 0.18  # space for dimension labels
    main_width = 1.0 - left_frac - right_pad
    main_bottom = cbar_h / fig_h + 0.04
    main_height = main_h / fig_h

    ax = fig.add_axes([left_frac, main_bottom, main_width, main_height])
    ax.set_title("")

    # Draw heatmap
    im = ax.imshow(
        pr_masked,
        cmap=greens,
        vmin=0,
        vmax=1,
        aspect="auto",
        extent=[-0.5, n_cols - 0.5, n_rows - 0.5, -0.5],  # x: cols, y: rows (0 at top)
    )

    # Cell annotations and red borders
    for i in range(n_rows):
        for j in range(n_cols):
            if not has_entry[i, j]:
                continue
            pr = pr_grid[i, j]
            tc = tc_grid[i, j]
            label = f"{int(round(pr * 100))}%{'*' if tc < 1.0 else ''}"
            text_color = "white" if pr > 0.55 else "black"
            ax.text(
                j,
                i,
                label,
                ha="center",
                va="center",
                fontsize=STYLE["annotation_fontsize"],
                color=text_color,
            )
            if tc < 1.0:
                rect = mpatches.Rectangle(
                    (j - 0.5, i - 0.5),
                    1.0,
                    1.0,
                    linewidth=2,
                    edgecolor="red",
                    facecolor="none",
                    clip_on=False,
                )
                ax.add_patch(rect)

    # Dimension separator lines
    for dim in DIM_ORDER:
        _, end_idx = dim_ranges[dim]
        if end_idx < n_rows:
            ax.axhline(end_idx - 0.5, color="black", linewidth=0.9, clip_on=False)

    # Right-side dimension labels (in axes transData, extended x range)
    for dim in DIM_ORDER:
        s_idx, e_idx = dim_ranges[dim]
        mid_y = (s_idx + e_idx - 1) / 2
        ax.text(
            n_cols - 0.5 + 0.25,
            mid_y,
            dim,
            ha="left",
            va="center",
            fontsize=STYLE["tick_label_fontsize"],
            fontstyle="italic",
            transform=ax.transData,
            clip_on=False,
        )

    # Column headers (on top)
    ax.set_xticks(range(n_cols))
    ax.set_xticklabels(COL_LABELS, fontsize=STYLE["tick_label_fontsize"])
    ax.xaxis.set_ticks_position("top")
    ax.xaxis.set_label_position("top")

    # Y-axis: target names
    ax.set_yticks(range(n_rows))
    ax.set_yticklabels([t for t, _ in rows], fontsize=STYLE["tick_label_fontsize"])

    ax.set_xlim(-0.5, n_cols - 0.5 + 1.5)
    ax.set_ylim(n_rows - 0.5, -0.5)
    ax.tick_params(left=False, top=False, bottom=False)
    for spine in ax.spines.values():
        spine.set_visible(False)

    # Layout constants for the bottom strip (in inches, then converted to fractions)
    # gap_in must accommodate colorbar tick labels + "Presence rate" label beneath the bar
    legend_h_in = 0.22  # single-row legend height
    legend_bot_in = 0.04  # clearance below legend to figure bottom
    gap_in = 0.7  # was 0.38 -- more clearance: colorbar tick labels collided with "Presence rate" caption after the 0.67x width resize
    cbar_h_in = 0.20  # colorbar bar height
    gap2_in = 0.12  # gap between colorbar top and main heatmap bottom
    bottom_strip = legend_bot_in + legend_h_in + gap_in + cbar_h_in + gap2_in

    legend_bot = legend_bot_in / fig_h
    cbar_bottom = (legend_bot_in + legend_h_in + gap_in) / fig_h
    cbar_ht = cbar_h_in / fig_h

    # Recalculate main heatmap bottom to clear the bottom strip
    main_bottom = bottom_strip / fig_h
    ax.set_position([left_frac, main_bottom, main_width, main_height])

    # Horizontal colorbar
    cbar_left = left_frac
    cbar_width = main_width * 0.78
    cax = fig.add_axes([cbar_left, cbar_bottom, cbar_width, cbar_ht])
    cb = mpl.colorbar.ColorbarBase(
        cax, cmap=greens, norm=Normalize(vmin=0, vmax=1), orientation="horizontal"
    )
    cb.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
    cb.set_ticklabels(
        ["0%", "25%", "50%", "75%", "100%"], fontsize=STYLE["tick_label_fontsize"]
    )
    cb.set_label("Presence rate", fontsize=STYLE["axis_label_fontsize"])

    # Legend in a single horizontal row below the colorbar (no overlap)
    legend_handles = [
        mpatches.Patch(
            facecolor="#cccccc",
            edgecolor="gray",
            linewidth=0.5,
            label="No glossary entries at this level",
        ),
        mpatches.Patch(
            facecolor=greens(0.0),
            edgecolor="gray",
            linewidth=0.5,
            label="0% presence (entries exist, none found)",
        ),
        mpatches.Patch(
            facecolor="white",
            edgecolor="red",
            linewidth=2,
            label="Type coverage < 100% (* in label)",
        ),
    ]
    # Anchor to the left figure edge so the legend sits cleanly below the colorbar
    fig.legend(
        handles=legend_handles,
        loc="lower left",
        bbox_to_anchor=(0.01, legend_bot),
        ncol=1,
        fontsize=7,
        frameon=False,
        handlelength=1.2,
        handletextpad=0.4,
    )

    _save(fig, base / "figures_final/fig1_coverage_heatmap.pdf")


In [ ]:
fig1_coverage_heatmap(data_dir=GFF_DATA_DIR)


### `annotation_rates_by_level`

**Renamed** from `fig2_annotation_rates_by_level` during the 2026-07-03 style
pass. The `fig2_` prefix collided with `generate_fig2_annotation_di_pairwise.py`'s
`fig2_annotation_di_by_pair_level.pdf` -- the paper's *actual* Figure 2. Confirmed
against `acl_latex (5).tex` (the current draft, the only one of 14 candidate
`.tex` exports found that references `fig2_annotation_di_by_pair_level.pdf` at
all) that this figure is the **first** `\begin{figure}` environment in the main
body (`\label{fig:annotation_rates}`), i.e. **Figure 1** -- not "Figure 8" as
originally believed when this rename was scoped. Given the paper's figure order
has already shifted at least twice during this project, the output filename was
made **position-independent** (no number at all) per author direction, so this
class of staleness cannot recur.

**Output file:** `outputs/figures_final/annotation_rates_by_level.pdf`
(was `fig2_annotation_rates_by_level.pdf`)

**Source TSV(s):** `outputs/stage2/s2_annotation_by_level_target.tsv`

**Pipeline stage:** Stage 2 (`stage2/s2_annotation_by_level_target.tsv`)

**`.tex` embed width:** `\columnwidth` (single-column, ACL two-column layout)
-- relevant for the Step 4 sizing pass below, since the function currently
sizes itself at `STYLE["textwidth_in"]` (full double-column width).

**Docstring (verbatim from source):**

```
Three-panel horizontal bar chart: correct vs. failure rates per group × level.

Source files:
- stage2/s2_annotation_by_level_target.tsv

Visual encoding:
- Teal bars = correct_labeling_rate (Case A / (A+B))
- Vermillion bars = failure_rate (Case B / (A+B))
- Only groups with stable match counts (stable_n == True) shown
- Groups sorted by failure_rate descending within each panel
- Panel labels in level colors; legend in L2 panel only
```


In [ ]:
def annotation_rates_by_level(data_dir: str = "outputs/") -> None:
    """
    Three-panel horizontal bar chart: correct vs. failure rates per group × level.

    Source files:
    - stage2/s2_annotation_by_level_target.tsv

    Visual encoding:
    - Teal bars = correct_labeling_rate (Case A / (A+B))
    - Vermillion bars = failure_rate (Case B / (A+B))
    - Only groups with stable match counts (stable_n == True) shown
    - Groups sorted by failure_rate descending within each panel
    - Panel labels in level colors; legend in L2 panel only
    """
    apply_style()
    base = Path(data_dir)
    s2_p = base / "stage2/s2_annotation_by_level_target.tsv"
    if not _need([s2_p], "annotation_rates_by_level"):
        return

    s2 = _read(s2_p)
    if s2.empty:
        return

    s2 = s2[s2["stable_n"].astype(bool)].copy()
    s2 = s2[~s2["is_self_referential"].astype(bool)].copy()
    # Explicit exclusion: referential_white_supremacist has is_self_referential=False
    # at L3 in the pipeline data, so the flag alone is insufficient.
    # minority is a pan-group category excluded from per-group reporting.
    _EXCLUDE = {"referential_white_supremacist", "minority"}
    s2 = s2[~s2["target"].isin(_EXCLUDE)].copy()

    LEVELS = ["L2", "L3", "L4"]
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(6.0, 6.0),  # was (columnwidth_in, 6.0)=(3.35,6.0): widened for 1:1 aspect per author request
        sharey=False,
        constrained_layout=True,
    )

    # Max category count across panels: gives every bar the same physical
    # thickness regardless of panel, so a sparse panel (e.g. L4, with far
    # fewer stable groups than L2/L3) shows genuine blank space instead of
    # its few bars stretching to fill the whole axis height.
    max_n = max(1, max((s2["coding_level"] == lvl).sum() for lvl in LEVELS))

    for col_i, (ax, level) in enumerate(zip(axes, LEVELS)):
        ax.set_title("")
        sub = s2[s2["coding_level"] == level].sort_values(
            "failure_rate", ascending=True
        )
        if sub.empty:
            ax.axis("off")
            continue

        groups = sub["target"].tolist()
        n = len(groups)
        ROW_SPACING = 0.6  # tighter row spacing (was 1.0 implicit via np.arange), matches fig3/fig6's row density
        y = np.arange(n) * ROW_SPACING

        # Stacked (was two dodged bars per category, offset +-BAR_H/2) --
        # correct_labeling_rate + failure_rate always sum to 1, so one bar
        # per category with the two rates concatenated communicates the same
        # information more compactly. Matches appA_coverage_presence_vs_type_by_level's
        # stacking convention, per author request.
        correct = sub["correct_labeling_rate"].values
        failure = sub["failure_rate"].values
        ax.barh(y, correct, color=STYLE["colors"]["correct"], height=0.4, label="Correct")  # was default height=0.8: bar height roughly matching text label height per author request
        ax.barh(y, failure, left=correct, color=STYLE["colors"]["failure"], height=0.4, label="Failure")

        ax.set_yticks(y)
        ax.set_yticklabels(groups, fontsize=8)  # bumped from 7: bigger text per author request
        ax.set_ylim(-0.5 * ROW_SPACING, (max_n - 0.5) * ROW_SPACING)  # consistent bar thickness across panels, scaled by ROW_SPACING (was auto-scaled per panel, making L4's few bars look much thicker/wider than L2/L3's)
        ax.set_xlim(0, 1)
        ax.set_xticks([0, 0.5, 1.0])  # restored per author request: panels wide enough now that 3 ticks no longer collide
        ax.tick_params(axis="x", labelsize=7)  # bumped back toward normal: panels wide enough now
        ax.set_xlabel("Rate", fontsize=7)  # was STYLE["axis_label_fontsize"]=10; narrow panel
        ax.axvline(0, color="black", linewidth=0.5)

        _panel_label(ax, level, STYLE["colors"][level])
        ax.spines["left"].set_visible(False)
        ax.tick_params(left=False)

    legend_handles = [
        mpatches.Patch(facecolor=STYLE["colors"]["correct"], label="Correct"),
        mpatches.Patch(facecolor=STYLE["colors"]["failure"], label="Failure"),
    ]
    fig.legend(
        handles=legend_handles,
        loc="outside lower center",
        ncol=2,
        fontsize=STYLE["legend_fontsize"],
        frameon=False,
    )

    _save(fig, base / "figures_final/annotation_rates_by_level.pdf")


In [ ]:
annotation_rates_by_level(data_dir=GFF_DATA_DIR)


### `fig3_case_ab_counts_by_level`

**Output file:** `outputs/figures_final/fig3_case_ab_counts_by_level.pdf`

**Source TSV(s):** `outputs/stage2/s2_annotation_by_level_target.tsv`

**Pipeline stage:** Stage 2 (`stage2/s2_annotation_by_level_target.tsv`)

**Docstring (verbatim from source):**

```
Three-panel horizontal stacked bar: total matches (Case A + Case B) per group × level.

Source files:
- stage2/s2_annotation_by_level_target.tsv

Visual encoding:
- Green segment = Case A (hateful context, correct identification)
- Vermillion segment = Case B (non-hateful context, false positive)
- Top 10 groups per panel by total match count
- If one bar dominates (>3× the 90th percentile of top-10), x-axis is
  truncated and the bar is annotated with its actual count
```


In [ ]:
def fig3_case_ab_counts_by_level(data_dir: str = "outputs/") -> None:
    """
    Three-panel horizontal stacked bar: total matches (Case A + Case B) per group × level.

    Source files:
    - stage2/s2_annotation_by_level_target.tsv

    Visual encoding:
    - Green segment = Case A (hateful context, correct identification)
    - Vermillion segment = Case B (non-hateful context, false positive)
    - Top 10 groups per panel by total match count
    - If one bar dominates (>3× the 90th percentile of top-10), x-axis is
      truncated and the bar is annotated with its actual count
    """
    apply_style()
    base = Path(data_dir)
    s2_p = base / "stage2/s2_annotation_by_level_target.tsv"
    if not _need([s2_p], "fig3"):
        return

    s2 = _read(s2_p)
    if s2.empty:
        return

    s2 = s2[~s2["is_self_referential"].astype(bool)].copy()
    _EXCLUDE = {"referential_white_supremacist", "minority"}
    s2 = s2[~s2["target"].isin(_EXCLUDE)].copy()

    LEVELS = ["L2", "L3", "L4"]
    fig, axes = plt.subplots(
        1, 3, figsize=(8.0, 4.0), constrained_layout=True  # was (columnwidth_in, 4.0)=(3.35,4.0): widened to 2:1 aspect per author request (also fixes rightmost panel's "Match count" label getting cut off at the narrower width)
    )

    for ax, level in zip(axes, LEVELS):
        ax.set_title("")
        sub = (
            s2[s2["coding_level"] == level]
            .dropna(subset=["total_matches"])
            .nlargest(10, "total_matches")
            .sort_values("total_matches", ascending=True)
        )

        if sub.empty:
            ax.axis("off")
            continue

        groups = sub["target"].tolist()
        n = len(groups)
        ROW_SPACING = 0.6  # was 1.0 (np.arange default): tighter row spacing per author request, decreases gap between bars
        y = np.arange(n) * ROW_SPACING
        totals = sub["total_matches"].values
        ca = sub["case_a_present_hateful"].values
        cb = sub["case_b_present_nonhateful"].values

        # Determine x axis limit; truncate if one bar dominates
        pct90 = np.percentile(totals, 90) if len(totals) > 1 else totals[0]
        x_max = totals.max() * 1.05
        truncate = False
        if totals.max() > 3 * pct90 and len(totals) > 2:
            sorted_t = np.sort(totals)
            x_max = sorted_t[-2] * 1.5
            truncate = True

        ax.barh(y, ca, color=STYLE["colors"]["case_a"], height=0.4, label="Case A (hateful)")  # was 0.6 (before that, default 0.8): further thinned per author request
        ax.barh(
            y,
            cb,
            left=ca,
            height=0.4,
            color=STYLE["colors"]["case_b"],
            label="Case B (non-hateful)",
        )

        # Annotate truncated bars: place text just outside the clipped bar end
        if truncate:
            for yi, tot in zip(y, totals):  # was enumerate(totals) giving raw row index -- misaligned once y became ROW_SPACING-scaled
                if tot > x_max:
                    ax.text(
                        x_max * 1.01,
                        yi,
                        f"{tot:,}",
                        ha="left",
                        va="center",
                        fontsize=STYLE["annotation_fontsize"],
                        color="black",
                        clip_on=False,
                    )

        ax.set_yticks(y)
        ax.set_yticklabels(groups, fontsize=8)  # bumped from 6.5: panels much wider now at 2:1 aspect, room for larger text
        ax.set_ylim(-0.5 * ROW_SPACING, (n - 0.5) * ROW_SPACING)
        ax.set_xlim(0, x_max)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=4))
        ax.tick_params(axis="x", labelsize=7)  # bumped from 6, rotation removed: panels wide enough now
        ax.set_xlabel("Match count", fontsize=STYLE["axis_label_fontsize"])  # restored to standard size: panel wide enough now, and no longer at risk of being clipped at the figure edge

        _panel_label(ax, level, STYLE["colors"][level])
        # left spine restored per author request (2026-07-03) -- was hidden to rely on
        # category labels alone; author wants the axis line visible.
        ax.tick_params(left=True, labelleft=True)

    # Legend below all panels
    legend_handles = [
        mpatches.Patch(facecolor=STYLE["colors"]["case_a"], label="Case A (hateful)"),
        mpatches.Patch(
            facecolor=STYLE["colors"]["case_b"], label="Case B (non-hateful)"
        ),
    ]
    fig.legend(
        handles=legend_handles,
        loc="outside lower center",
        ncol=2,
        fontsize=STYLE["legend_fontsize"],
        frameon=False,
    )

    _save(fig, base / "figures_final/fig3_case_ab_counts_by_level.pdf")


In [ ]:
fig3_case_ab_counts_by_level(data_dir=GFF_DATA_DIR)


### `fig4_di_histograms_by_level`

**Output file:** `outputs/figures_final/fig4_di_histograms_by_level.pdf`

**Source TSV(s):** `outputs/stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`

**Pipeline stage:** Stage 4b (`stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`)

**Docstring (verbatim from source):**

```
Three-panel overlapping histograms of pairwise DI ratios (presence and type-coverage).

Source files:
- stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv

Visual encoding:
- Blue histogram = presence_rate_di_ratio (alpha=0.7)
- Sky-blue histogram = type_coverage_di_ratio (alpha=0.7)
- Dashed red vertical line at DI = 0.80 (4/5 rule threshold)
- Only stable pairs (unstable_small_n == False) included
```


In [ ]:
def fig4_di_histograms_by_level(data_dir: str = "outputs/") -> None:
    """
    Three-panel overlapping histograms of pairwise DI ratios (presence and type-coverage).

    Source files:
    - stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv

    Visual encoding:
    - Blue histogram = presence_rate_di_ratio (alpha=0.7)
    - Sky-blue histogram = type_coverage_di_ratio (alpha=0.7)
    - Dashed red vertical line at DI = 0.80 (4/5 rule threshold)
    - Only stable pairs (unstable_small_n == False) included
    """
    apply_style()
    base = Path(data_dir)
    s4b_p = base / "stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv"
    if not _need([s4b_p], "fig4"):
        return

    df = _read(s4b_p)
    if df.empty:
        return

    df = df[~df["unstable_small_n"].astype(bool)].copy()

    LEVELS = ["L2", "L3", "L4"]
    fig, axes = plt.subplots(
        1, 3, figsize=(STYLE["textwidth_in"], 3.2), constrained_layout=True
    )

    # DI ratios are bounded [0, 1]; extend slightly to 1.05 to catch floating-point edge
    # values and avoid a spurious spike at the histogram's right edge.
    BINS = np.linspace(0, 1.05, 22)

    for col_i, (ax, level) in enumerate(zip(axes, LEVELS)):
        ax.set_title("")
        sub = df[df["coding_level"] == level]
        pr_di = sub["presence_rate_di_ratio"].dropna()
        tc_di = sub["type_coverage_di_ratio"].dropna()

        ax.hist(
            pr_di,
            bins=BINS,
            color=STYLE["colors"]["presence"],
            alpha=0.7,
            label="Presence DI",
        )
        ax.hist(
            tc_di,
            bins=BINS,
            color=STYLE["colors"]["type_cov"],
            alpha=0.7,
            label="Type-cov. DI",
        )

        ax.axvline(0.80, color="red", linewidth=1.2, linestyle="--")

        ax.set_xlim(0, 1.05)
        ax.set_xlabel("DI ratio", fontsize=STYLE["axis_label_fontsize"])
        ax.set_ylabel(
            "Count" if col_i == 0 else "", fontsize=STYLE["axis_label_fontsize"]
        )
        # Force integer y-axis ticks: with few stable pairs (L3 has 4, L4 has 1),
        # auto-scaling produces fractional counts like 0.25, 0.50 which are meaningless.
        ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

        _panel_label(ax, level, STYLE["colors"][level])

        if col_i == 0:
            ax.legend(fontsize=STYLE["legend_fontsize"], frameon=False)

    _save(fig, base / "figures_final/fig4_di_histograms_by_level.pdf")


In [ ]:
fig4_di_histograms_by_level(data_dir=GFF_DATA_DIR)


### `fig5_worst_di_by_pair_level`

**Output file:** `outputs/figures_final/fig5_worst_di_by_pair_level.pdf`

**Source TSV(s):** `outputs/stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`

**Pipeline stage:** Stage 4b (`stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`)

**Docstring (verbatim from source):**

```
Three-panel horizontal bar chart: worst (minimum) DI ratio per stable group pair × level.

Source files:
- stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv

Visual encoding:
- Blue bar (pass color) = DI >= 0.80 (passes 4/5 rule)
- Red bar (fail color) = DI < 0.80 (fails 4/5 rule)
- Dashed red vertical line at DI = 0.80
- Only stable pairs (unstable_small_n == False) shown
- Sorted ascending (worst DI at top)
```

**Note:** A **third** distinct source for a "worst pairwise DI" figure (Stage 4b), alongside Stage 3 (`stage5_figures.level_stratified_figures`) and Stage 4a (`stage5_figures.group_collapsed_figures`) above. This is the file `generate_fig2_annotation_di_pairwise.py`'s own docstring calls "Figure 9" while its output filename says `fig2_...` -- another naming inconsistency, surfaced not fixed.


In [ ]:
def fig5_worst_di_by_pair_level(data_dir: str = "outputs/") -> None:
    """
    Three-panel horizontal bar chart: worst (minimum) DI ratio per stable group pair × level.

    Source files:
    - stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv

    Visual encoding:
    - Blue bar (pass color) = DI >= 0.80 (passes 4/5 rule)
    - Red bar (fail color) = DI < 0.80 (fails 4/5 rule)
    - Dashed red vertical line at DI = 0.80
    - Only stable pairs (unstable_small_n == False) shown
    - Sorted ascending (worst DI at top)
    """
    apply_style()
    base = Path(data_dir)
    s4b_p = base / "stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv"
    if not _need([s4b_p], "fig5"):
        return

    df = _read(s4b_p)
    if df.empty:
        return

    df = df[~df["unstable_small_n"].astype(bool)].dropna(subset=["worst_di_ratio"])

    LEVELS = ["L2", "L3", "L4"]
    fig, axes = plt.subplots(
        1, 3, figsize=(6.0, 2.1), constrained_layout=True  # height 2.4->2.1: further vertical tightening per author request, closer to bar+padding == text label height
    )

    # Max bar count across panels: gives every bar the same physical thickness
    # regardless of panel, so a sparse panel (e.g. L4) reads as "mostly empty"
    # rather than a single bar stretched to fill the whole axis. Matches the
    # technique already used in generate_fig2_annotation_di_pairwise.py.
    max_n = max(1, max((df["coding_level"] == lvl).sum() for lvl in LEVELS))

    for ax, level in zip(axes, LEVELS):
        ax.set_title("")
        sub = df[df["coding_level"] == level].sort_values(
            "worst_di_ratio", ascending=True
        )

        if sub.empty:
            ax.axis("off")
            continue

        pairs = [f"{a} vs {b}" for a, b in zip(sub["target_a"], sub["target_b"])]
        di = sub["worst_di_ratio"].values
        n = len(pairs)
        ROW_SPACING = 0.45  # was 0.6: further tightened row spacing per author request
        y = np.arange(n) * ROW_SPACING

        colors = [
            STYLE["colors"]["pass"] if v >= 0.80 else STYLE["colors"]["fail"]
            for v in di
        ]

        ax.barh(y, di, color=colors, height=0.3)  # was 0.4 (before that, 0.6, before that, default 0.8): further thinned per author request
        ax.axvline(0.80, color="red", linewidth=1.2, linestyle="--")

        ax.set_yticks(y)
        ax.set_yticklabels(pairs, fontsize=6)  # bumped from 5: panels wider now at 2:1 aspect
        ax.set_xlim(0, 1.1)
        ax.set_xticks([0, 0.5, 1.0])  # added 0.5 marker per author request; panels wide enough now
        ax.tick_params(axis="x", labelsize=7)  # bumped from 6: panels wider now
        ax.set_ylim(-0.5 * ROW_SPACING, (max_n - 0.5) * ROW_SPACING)  # consistent bar thickness across panels, scaled by ROW_SPACING
        ax.set_xlabel("Worst DI ratio", fontsize=STYLE["axis_label_fontsize"])  # restored to standard size: panel wide enough now, and no longer at risk of being clipped at the figure edge

        _panel_label(ax, level, STYLE["colors"][level])
        # left spine restored per author request (2026-07-03) -- was hidden to rely on
        # category labels alone; author wants the axis line visible.
        ax.tick_params(left=True, labelleft=True)

    _save(fig, base / "figures_final/fig5_worst_di_by_pair_level.pdf")


In [ ]:
fig5_worst_di_by_pair_level(data_dir=GFF_DATA_DIR)


### `fig6_elsherief_annotation_delta`

**Output file:** `outputs/figures_final/fig6_elsherief_annotation_delta.pdf`

**Source TSV(s):** `outputs/stage4/elsherief/s4c_annotation_delta_union_vs_elsherief.tsv`

**Pipeline stage:** Stage 4c (`stage4/elsherief/s4c_annotation_delta_union_vs_elsherief.tsv`)

**Docstring (verbatim from source):**

```
Horizontal bar chart: Δ correct labeling rate (union − ElSherief) per reporting group.

Source files:
- stage4/elsherief/s4c_annotation_delta_union_vs_elsherief.tsv

Visual encoding:
- Green bar = Δ > 0 (union improves on ElSherief alone)
- Red bar = Δ < 0 (ElSherief alone outperforms union)
- Sorted ascending (most negative delta at top)

Data treatment: the source file has one row per (coding_level, report_level, report_target).
We aggregate across coding levels by summing the underlying case counts (A and B) for both
the union and ElSherief corpora, then recompute rates from the pooled totals — rates must
not be averaged directly because denominators differ across levels.  Groups with fewer than
20 pooled union matches or with zero ElSherief matches are excluded.
```


In [ ]:
def fig6_elsherief_annotation_delta(data_dir: str = "outputs/") -> None:
    """
    Horizontal bar chart: Δ correct labeling rate (union − ElSherief) per reporting group.

    Source files:
    - stage4/elsherief/s4c_annotation_delta_union_vs_elsherief.tsv

    Visual encoding:
    - Green bar = Δ > 0 (union improves on ElSherief alone)
    - Red bar = Δ < 0 (ElSherief alone outperforms union)
    - Sorted ascending (most negative delta at top)

    Data treatment: the source file has one row per (coding_level, report_level, report_target).
    We aggregate across coding levels by summing the underlying case counts (A and B) for both
    the union and ElSherief corpora, then recompute rates from the pooled totals — rates must
    not be averaged directly because denominators differ across levels.  Groups with fewer than
    20 pooled union matches or with zero ElSherief matches are excluded.
    """
    apply_style()
    base = Path(data_dir)
    s4c_p = base / "stage4/elsherief/s4c_annotation_delta_union_vs_elsherief.tsv"
    if not _need([s4c_p], "fig6"):
        return

    df = _read(s4c_p)
    if df.empty:
        return
    required = {
        "report_level",
        "report_target",
        "case_a_present_hateful_union",
        "case_b_present_nonhateful_union",
        "case_a_present_hateful_elsherief",
        "case_b_present_nonhateful_elsherief",
    }
    missing_cols = required - set(df.columns)
    if missing_cols:
        print(f"WARNING: fig6: skipping — missing columns {missing_cols}")
        return

    # Pool across coding levels: sum case counts, then recompute rates
    agg = df.groupby(["report_level", "report_target"], as_index=False).agg(
        ca_u=("case_a_present_hateful_union", "sum"),
        cb_u=("case_b_present_nonhateful_union", "sum"),
        ca_e=("case_a_present_hateful_elsherief", "sum"),
        cb_e=("case_b_present_nonhateful_elsherief", "sum"),
    )
    agg["total_u"] = agg["ca_u"] + agg["cb_u"]
    agg["total_e"] = agg["ca_e"] + agg["cb_e"]
    # Require stable pooled count in union (≥ 20) and at least one ElSherief match
    agg = agg[(agg["total_u"] >= 20) & (agg["total_e"] > 0)]
    agg["rate_u"] = agg["ca_u"] / agg["total_u"]
    agg["rate_e"] = agg["ca_e"] / agg["total_e"]
    agg["delta"] = agg["rate_u"] - agg["rate_e"]
    agg = agg.dropna(subset=["delta"])

    agg = agg.sort_values("delta", ascending=True)
    agg["label"] = agg["report_level"].str.title() + ": " + agg["report_target"]

    labels = agg["label"].tolist()
    deltas = agg["delta"].values
    n = len(labels)
    ROW_SPACING = 0.6  # was 1.0 (np.arange default): tighter row spacing per author request, decreases gap between bars
    y = np.arange(n) * ROW_SPACING

    fig, ax = plt.subplots(
        figsize=(STYLE["columnwidth_in"] * 0.6, max(2.0, n * 0.20 + 0.6)),  # .tex: width=0.6\linewidth; height formula shrunk (was n*0.32+0.8) to match tighter row spacing/thinner bars below
        constrained_layout=True,
    )
    ax.set_title("")

    colors = [
        STYLE["colors"]["delta_pos"] if d >= 0 else STYLE["colors"]["delta_neg"]
        for d in deltas
    ]
    ax.barh(y, deltas, color=colors, height=0.5)  # was default height=0.8: thinner bars per author request
    ax.axvline(0, color="black", linewidth=0.8)

    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=7)  # bumped from 6 (was tick_label_fontsize=8 originally): figure narrowed to 0.6x columnwidth (.tex: width=0.6\linewidth)
    ax.set_ylim(-0.5 * ROW_SPACING, (n - 0.5) * ROW_SPACING)  # scaled by ROW_SPACING for consistent bar thickness
    ax.tick_params(axis="x", labelsize=6, labelrotation=90)  # numeric tick labels collided at this panel width
    # Two-line label, further shrunk to fit 0.6x column width (.tex: width=0.6\linewidth)
    ax.set_xlabel(
        "Correct labeling rate Δ\n(union − ElSherief)",
        fontsize=6.5,  # was axis_label_fontsize=10
    )
    ax.spines["left"].set_visible(False)
    ax.tick_params(left=False)

    _save(fig, base / "figures_final/fig6_elsherief_annotation_delta.pdf")


In [ ]:
fig6_elsherief_annotation_delta(data_dir=GFF_DATA_DIR)


### `fig7_elsherief_coverage_delta`

**Output file:** `outputs/figures_final/fig7_elsherief_coverage_delta.pdf`

**Source TSV(s):** `outputs/stage4/elsherief/s4c_coverage_delta_union_vs_elsherief.tsv`

**Pipeline stage:** Stage 4c (`stage4/elsherief/s4c_coverage_delta_union_vs_elsherief.tsv`)

**Docstring (verbatim from source):**

```
Horizontal bar chart: Δ presence rate (union − ElSherief) per reporting group.

Source files:
- stage4/elsherief/s4c_coverage_delta_union_vs_elsherief.tsv

Visual encoding:
- All bars green (union always extends or equals ElSherief coverage)
- Sorted ascending (largest gain at top)
- Only groups with a positive coverage gain are shown

Data treatment: the source file has one row per (coding_level, report_level, report_target).
We aggregate across coding levels by summing distinct_dogwhistles_found and
total_glossary_dogwhistles for both corpora, then recompute presence rates from pooled
counts.  Where ElSherief has no glossary entries for a group, its presence rate is 0.
Groups with no union glossary entries are excluded.
```


In [ ]:
def fig7_elsherief_coverage_delta(data_dir: str = "outputs/") -> None:
    """
    Horizontal bar chart: Δ presence rate (union − ElSherief) per reporting group.

    Source files:
    - stage4/elsherief/s4c_coverage_delta_union_vs_elsherief.tsv

    Visual encoding:
    - All bars green (union always extends or equals ElSherief coverage)
    - Sorted ascending (largest gain at top)
    - Only groups with a positive coverage gain are shown

    Data treatment: the source file has one row per (coding_level, report_level, report_target).
    We aggregate across coding levels by summing distinct_dogwhistles_found and
    total_glossary_dogwhistles for both corpora, then recompute presence rates from pooled
    counts.  Where ElSherief has no glossary entries for a group, its presence rate is 0.
    Groups with no union glossary entries are excluded.
    """
    apply_style()
    base = Path(data_dir)
    s4c_p = base / "stage4/elsherief/s4c_coverage_delta_union_vs_elsherief.tsv"
    if not _need([s4c_p], "fig7"):
        return

    df = _read(s4c_p)
    if df.empty:
        return
    required = {
        "report_level",
        "report_target",
        "distinct_dogwhistles_found_union",
        "total_glossary_dogwhistles_union",
        "distinct_dogwhistles_found_elsherief",
        "total_glossary_dogwhistles_elsherief",
    }
    missing_cols = required - set(df.columns)
    if missing_cols:
        print(f"WARNING: fig7: skipping — missing columns {missing_cols}")
        return

    # Pool across coding levels: sum dogwhistle counts, then recompute rates
    agg = df.groupby(["report_level", "report_target"], as_index=False).agg(
        found_u=("distinct_dogwhistles_found_union", "sum"),
        total_u=("total_glossary_dogwhistles_union", "sum"),
        found_e=("distinct_dogwhistles_found_elsherief", "sum"),
        total_e=("total_glossary_dogwhistles_elsherief", "sum"),
    )
    agg = agg[agg["total_u"] > 0]  # must have union glossary entries
    agg["rate_u"] = agg["found_u"] / agg["total_u"]
    # If ElSherief has no glossary entries, its presence rate is 0 by definition
    agg["rate_e"] = agg.apply(
        lambda r: r["found_e"] / r["total_e"] if r["total_e"] > 0 else 0.0, axis=1
    )
    agg["delta"] = agg["rate_u"] - agg["rate_e"]
    agg = agg[agg["delta"] > 0]  # keep only groups where union extends coverage
    agg = agg.sort_values(
        "delta", ascending=True
    )  # ascending → smallest at y=0 (bottom), largest at y=n-1 (top)
    agg["label"] = agg["report_level"].str.title() + ": " + agg["report_target"]

    labels = agg["label"].tolist()
    deltas = agg["delta"].values
    n = len(labels)
    ROW_SPACING = 0.6  # was 1.0 (np.arange default): tighter row spacing per author request, decreases gap between bars, compresses figure vertically
    y = np.arange(n) * ROW_SPACING

    fig, ax = plt.subplots(
        figsize=(STYLE["columnwidth_in"] * 0.75, max(2.0, n * 0.16 + 0.6)),  # height formula shrunk (was n*0.22+0.7) to match the tighter row spacing/thinner bars below
        constrained_layout=True,
    )
    ax.set_title("")

    ax.barh(y, deltas, color=STYLE["colors"]["delta_pos"], height=0.5)  # was default height=0.8: compressed per author request
    ax.axvline(0, color="black", linewidth=0.8)

    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=7)  # bumped from 6 (was tick_label_fontsize=8 originally): figure narrowed to 0.6x columnwidth (.tex: width=0.6\linewidth)
    ax.set_ylim(-0.5 * ROW_SPACING, (n - 0.5) * ROW_SPACING)  # scaled by ROW_SPACING for consistent bar thickness
    ax.tick_params(axis="x", labelsize=6, labelrotation=90)  # numeric tick labels collided at this panel width
    # Two-line label, further shrunk to fit 0.6x column width (.tex: width=0.6\linewidth)
    ax.set_xlabel(
        "Presence rate Δ\n(union − ElSherief)", fontsize=6.5  # was axis_label_fontsize=10
    )
    ax.spines["left"].set_visible(False)
    ax.tick_params(left=False)

    _save(fig, base / "figures_final/fig7_elsherief_coverage_delta.pdf")


In [ ]:
fig7_elsherief_coverage_delta(data_dir=GFF_DATA_DIR)


### `appA_coverage_presence_vs_type_by_level`

**Output file:** `outputs/figures_final/appA_coverage_presence_vs_type_by_level.pdf`

**Source TSV(s):** `outputs/stage4/by_level_group/s4b_coverage_by_level_group.tsv`

**Pipeline stage:** Stage 4b (`stage4/by_level_group/s4b_coverage_by_level_group.tsv`)

**Docstring (verbatim from source):**

```
Three-panel horizontal stacked bar: presence_rate (dark) and type_coverage gap.

Source files:
- stage4/by_level_group/s4b_coverage_by_level_group.tsv

Visual encoding:
- Dark blue segment = presence_rate (fraction of dogwhistle surface forms found)
- Medium blue gap = type_coverage − presence_rate (found types but not all forms)
- Light gray gap = 1 − type_coverage (entire categories absent from corpus)
- Only non-self-referential rows shown
```


In [ ]:
def appA_coverage_presence_vs_type_by_level(data_dir: str = "outputs/") -> None:
    """
    Three-panel horizontal stacked bar: presence_rate (dark) and type_coverage gap.

    Source files:
    - stage4/by_level_group/s4b_coverage_by_level_group.tsv

    Visual encoding:
    - Dark blue segment = presence_rate (fraction of dogwhistle surface forms found)
    - Medium blue gap = type_coverage − presence_rate (found types but not all forms)
    - Light gray gap = 1 − type_coverage (entire categories absent from corpus)
    - Only non-self-referential rows shown
    """
    apply_style()
    base = Path(data_dir)
    s4b_p = base / "stage4/by_level_group/s4b_coverage_by_level_group.tsv"
    if not _need([s4b_p], "appA"):
        return

    df = _read(s4b_p)
    if df.empty:
        return

    df = df[~df["is_self_referential"].astype(bool)].copy()
    df = df.dropna(subset=["presence_rate", "type_coverage"])

    LEVELS = ["L2", "L3", "L4"]
    fig, axes = plt.subplots(
        1, 3, figsize=(STYLE["textwidth_in"], 3.5), constrained_layout=True  # was height=5.5: compressed vertically per author request
    )

    for ax, level in zip(axes, LEVELS):
        ax.set_title("")
        sub = df[df["coding_level"] == level].sort_values(
            "type_coverage", ascending=True
        )

        if sub.empty:
            ax.axis("off")
            continue

        groups = sub["report_target"].tolist()
        n = len(groups)
        y = np.arange(n)

        pr = sub["presence_rate"].values
        tc = sub["type_coverage"].values
        gap_form = np.clip(tc - pr, 0, None)  # form-level gap; 0 when tc < pr
        # Third segment must start where second ends regardless of float quirks
        seg2_end = pr + gap_form  # == max(pr, tc) element-wise
        gap_type = np.clip(1.0 - seg2_end, 0, None)  # categorical absence

        ax.barh(y, pr, color=STYLE["colors"]["presence"], height=0.6, label="Presence rate")  # was default height=0.8: thinner bars per author request
        ax.barh(y, gap_form, left=pr, height=0.6, color="#7EB9DD", label="Type found, form gap")
        ax.barh(
            y, gap_type, left=seg2_end, height=0.6, color="#cccccc", label="Categorical absence"
        )

        ax.set_yticks(y)
        ax.set_yticklabels(groups, fontsize=STYLE["tick_label_fontsize"])
        ax.set_xlim(0, 1.0)
        ax.set_xlabel("Coverage", fontsize=STYLE["axis_label_fontsize"])

        _panel_label(ax, level, STYLE["colors"][level])
        # left spine restored per author request (2026-07-03) -- was hidden to rely on
        # category labels alone; author wants the axis line visible.
        ax.tick_params(left=True, labelleft=True)

    legend_handles = [
        mpatches.Patch(facecolor=STYLE["colors"]["presence"], label="Presence rate"),
        mpatches.Patch(facecolor="#7EB9DD", label="Type found, form gap"),
        mpatches.Patch(facecolor="#cccccc", label="Categorical absence"),
    ]
    fig.legend(
        handles=legend_handles,
        loc="outside lower center",
        ncol=3,
        fontsize=STYLE["legend_fontsize"],
        frameon=False,
    )

    _save(fig, base / "figures_final/appA_coverage_presence_vs_type_by_level.pdf")


In [ ]:
appA_coverage_presence_vs_type_by_level(data_dir=GFF_DATA_DIR)


### `appB_token_frequency_by_level`

**Output file:** `outputs/figures_final/appB_token_frequency_by_level.pdf`

**Source TSV(s):** `outputs/stage1/s1_coverage_by_level_target.tsv`

**Pipeline stage:** Stage 1 (`stage1/s1_coverage_by_level_target.tsv`)

**Docstring (verbatim from source):**

```
Three-panel horizontal bar: top-10 groups by token frequency per coding level.

Source files:
- stage1/s1_coverage_by_level_target.tsv (token_frequency column)

Visual encoding:
- Bar color matches level color (Okabe-Ito orange/sky-blue/pink)
- Top 10 groups per level by raw token match count
```


In [ ]:
def appB_token_frequency_by_level(data_dir: str = "outputs/") -> None:
    """
    Three-panel horizontal bar: top-10 groups by token frequency per coding level.

    Source files:
    - stage1/s1_coverage_by_level_target.tsv (token_frequency column)

    Visual encoding:
    - Bar color matches level color (Okabe-Ito orange/sky-blue/pink)
    - Top 10 groups per level by raw token match count
    """
    apply_style()
    base = Path(data_dir)
    s1_p = base / "stage1/s1_coverage_by_level_target.tsv"
    if not _need([s1_p], "appB"):
        return

    s1 = _read(s1_p)
    if s1.empty:
        return

    s1 = s1[~s1["is_self_referential"].astype(bool)]
    # Explicit exclusion (2026-07-03, author request): referential_white_supremacist
    # has is_self_referential=False at L3 in the pipeline data, so the flag alone
    # is insufficient to drop it (same issue documented in annotation_rates_by_level).
    s1 = s1[s1["target"] != "referential_white_supremacist"]

    LEVELS = ["L2", "L3", "L4"]
    fig, axes = plt.subplots(
        1, 3, figsize=(5.5, 3.0), constrained_layout=True  # height 5.5->3.0: row spacing was never tightened after the earlier widening, leaving far more vertical room than the text labels need (unlike fig3/fig6, which already got ROW_SPACING)
    )

    for ax, level in zip(axes, LEVELS):
        ax.set_title("")
        sub = (
            s1[s1["coding_level"] == level]
            .nlargest(10, "token_frequency")
            .sort_values("token_frequency", ascending=True)
        )

        if sub.empty:
            ax.axis("off")
            continue

        groups = sub["target"].tolist()
        freqs = sub["token_frequency"].values
        n = len(groups)
        ROW_SPACING = 0.6  # matches fig3/fig6's row density (author confirmed those look right)
        y = np.arange(n) * ROW_SPACING

        ax.barh(y, freqs, color=STYLE["colors"][level], height=0.4)  # was 0.6: thinner still, matches fig3/fig6's bar-to-gap ratio
        ax.set_yticks(y)
        ax.set_yticklabels(groups, fontsize=8)  # bumped from 6.5: panels are wide now (5.5in/3), room for larger text per author request
        ax.set_ylim(-0.5 * ROW_SPACING, (n - 0.5) * ROW_SPACING)
        ax.tick_params(axis="x", labelsize=7)  # bumped from 6, rotation removed: panels wide enough now
        ax.set_xlabel("Token frequency", fontsize=STYLE["axis_label_fontsize"])  # restored to standard size: panel wide enough now

        _panel_label(ax, level, STYLE["colors"][level])
        # left spine restored per author request (2026-07-03) -- was hidden to rely on
        # category labels alone; author wants the axis line visible.
        ax.tick_params(left=True, labelleft=True)

    _save(fig, base / "figures_final/appB_token_frequency_by_level.pdf")


In [ ]:
appB_token_frequency_by_level(data_dir=GFF_DATA_DIR)


### `appC_annotation_rates_pooled`

**Output file:** `outputs/figures_final/appC_annotation_rates_pooled.pdf`

**Source TSV(s):** `outputs/stage4/by_level_group/s4b_annotation_by_level_group.tsv`

**Pipeline stage:** Stage 4b (`stage4/by_level_group/s4b_annotation_by_level_group.tsv`)

**Docstring (verbatim from source):**

```
Pooled correct vs. failure rate bar chart (all groups, all levels combined).

Source files:
- stage4/by_level_group/s4b_annotation_by_level_group.tsv

Visual encoding:
- Same style as Figure 2 but with data pooled across L2/L3/L4
- Aggregate by summing case_a and case_b counts, then recomputing rates
- Only groups with at least 20 total matches after pooling
```


In [ ]:
def appC_annotation_rates_pooled(data_dir: str = "outputs/") -> None:
    """
    Pooled correct vs. failure rate bar chart (all groups, all levels combined).

    Source files:
    - stage4/by_level_group/s4b_annotation_by_level_group.tsv

    Visual encoding:
    - Same style as Figure 2 but with data pooled across L2/L3/L4
    - Aggregate by summing case_a and case_b counts, then recomputing rates
    - Only groups with at least 20 total matches after pooling
    """
    apply_style()
    base = Path(data_dir)
    s4b_p = base / "stage4/by_level_group/s4b_annotation_by_level_group.tsv"
    if not _need([s4b_p], "appC"):
        return

    df = _read(s4b_p)
    if df.empty:
        return

    df = df[~df["is_self_referential"].astype(bool)].copy()

    # Pool across levels
    agg = df.groupby("report_target", as_index=False).agg(
        case_a=("case_a_present_hateful", "sum"),
        case_b=("case_b_present_nonhateful", "sum"),
    )
    agg["total"] = agg["case_a"] + agg["case_b"]
    agg = agg[agg["total"] >= 20]
    agg["correct_rate"] = agg["case_a"] / agg["total"]
    agg["failure_rate"] = agg["case_b"] / agg["total"]
    agg = agg.sort_values("failure_rate", ascending=True)

    if agg.empty:
        print("WARNING: appC has no pooled groups with n >= 20")
        return

    groups = agg["report_target"].tolist()
    n = len(groups)
    ROW_SPACING = 0.6  # tighter row spacing (was 1.0 implicit via np.arange): bar height roughly matching text label height per author request
    y = np.arange(n) * ROW_SPACING

    fig, ax = plt.subplots(
        figsize=(STYLE["columnwidth_in"], max(2.5, n * 0.22 + 0.6)),  # .tex: width=\columnwidth; height formula shrunk (was n*0.38+1.0) to match the tighter row spacing/thinner bars below
        constrained_layout=True,
    )
    ax.set_title("")

    # Stacked (was two dodged bars per category, offset +-BAR_H/2) --
    # correct_rate + failure_rate always sum to 1, so one bar per category
    # with the two rates concatenated communicates the same information more
    # compactly. Matches appA_coverage_presence_vs_type_by_level's stacking
    # convention and annotation_rates_by_level's identical fix, per author request.
    correct = agg["correct_rate"].values
    failure = agg["failure_rate"].values
    ax.barh(y, correct, color=STYLE["colors"]["correct"], height=0.4, label="Correct")  # was default height=0.8: thinner, matches annotation_rates_by_level
    ax.barh(y, failure, left=correct, color=STYLE["colors"]["failure"], height=0.4, label="Failure")

    ax.set_yticks(y)
    ax.set_yticklabels(groups, fontsize=9)  # bumped from tick_label_fontsize=8: bigger text per author request
    ax.set_ylim(-0.5 * ROW_SPACING, (n - 0.5) * ROW_SPACING)
    ax.set_xlim(0, 1)
    ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_xlabel("Rate", fontsize=STYLE["axis_label_fontsize"])
    ax.axvline(0, color="black", linewidth=0.5)
    ax.spines["left"].set_visible(False)
    ax.tick_params(left=False)

    # Legend moved below the plot (was ax-level, lower-right, overlapping the
    # bottom bars once rows were tightened above) -- fig-level legend, outside
    # the axes, matching the convention used elsewhere in this notebook.
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(
        handles, labels,
        loc="outside lower center", ncol=2,
        fontsize=STYLE["legend_fontsize"], frameon=False,
    )

    _save(fig, base / "figures_final/appC_annotation_rates_pooled.pdf")


In [ ]:
appC_annotation_rates_pooled(data_dir=GFF_DATA_DIR)


### `appD_worst_di_and_label_gap_pooled`

**Output file:** `outputs/figures_final/appD_worst_di_and_label_gap_pooled.pdf`

**Source TSV(s):** `outputs/stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`

**Pipeline stage:** Stage 4b (`stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`)

**Docstring (verbatim from source):**

```
Two-panel figure: worst pooled DI ratio (left) and largest label gap (right).

Source files:
- stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv

Visual encoding:
- Left panel: worst (min) DI ratio across levels per pair — colored pass/fail
- Right panel: largest (max) absolute label gap across levels per pair — all red
- Only stable pairs (unstable_small_n == False) considered
```


In [ ]:
def appD_worst_di_and_label_gap_pooled(data_dir: str = "outputs/") -> None:
    """
    Two-panel figure: worst pooled DI ratio (left) and largest label gap (right).

    Source files:
    - stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv

    Visual encoding:
    - Left panel: worst (min) DI ratio across levels per pair — colored pass/fail
    - Right panel: largest (max) absolute label gap across levels per pair — all red
    - Only stable pairs (unstable_small_n == False) considered
    """
    apply_style()
    base = Path(data_dir)
    s4b_p = base / "stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv"
    if not _need([s4b_p], "appD"):
        return

    df = _read(s4b_p)
    if df.empty:
        return

    df = df[~df["unstable_small_n"].astype(bool)].copy()

    # Pool across levels: take min DI and max label gap per pair
    pooled = df.groupby(["target_a", "target_b"], as_index=False).agg(
        worst_di=("worst_di_ratio", "min"), max_gap=("labeling_rate_gap_abs", "max")
    )
    pooled = pooled.dropna(subset=["worst_di"]).sort_values("worst_di", ascending=True)

    pairs = [f"{a} vs {b}" for a, b in zip(pooled["target_a"], pooled["target_b"])]
    n = len(pairs)
    ROW_SPACING = 0.6  # was 1.0 (np.arange default): tighter row spacing per author request, decreases gap between bars
    y = np.arange(n) * ROW_SPACING

    fig, (ax_di, ax_gap) = plt.subplots(
        1,
        2,
        figsize=(6.0, max(1.8, n * 0.13 + 0.6)),  # was max(3.0, n*0.16+1.0): floor/slope were still too generous relative to the (already-thinned) bars -- tightened per author request to match fig2/fig3/fig6's row density
        constrained_layout=True,
    )

    # Left: worst DI
    ax_di.set_title("")
    di_colors = [
        STYLE["colors"]["pass"] if v >= 0.80 else STYLE["colors"]["fail"]
        for v in pooled["worst_di"].values
    ]
    ax_di.barh(y, pooled["worst_di"].values, color=di_colors, height=0.4)  # was 0.6 (before that, default 0.8): thinner bars per author request
    ax_di.axvline(0.80, color="red", linewidth=1.2, linestyle="--")
    ax_di.set_yticks(y)
    ax_di.set_yticklabels(pairs, fontsize=5.5)  # was max(5, tick_label_fontsize-1)=7: narrow panel
    ax_di.set_xlim(0, 1.1)
    ax_di.set_xticks([0, 0.5, 1.0])  # added 0.5 marker per author request
    ax_di.tick_params(axis="x", labelsize=6)
    ax_di.set_xlabel("Worst DI ratio\n(pooled)", fontsize=7)  # was axis_label_fontsize=10, one line: collided with ax_gap's label at this panel width
    # left spine restored per author request (2026-07-03) -- was hidden to rely on
    # category labels alone; author wants the axis line visible.
    ax_di.tick_params(left=True, labelleft=True)

    # Right: largest label gap — NaN means no stable label estimate for that pair
    ax_gap.set_title("")
    gap_vals = pooled["max_gap"].values  # NaN for pairs without stable gap
    gap_plot = np.where(np.isnan(gap_vals), 0, gap_vals)  # 0-length bars for NaN pairs
    ax_gap.barh(y, gap_plot, color=STYLE["colors"]["fail"], height=0.4)  # was 0.6 (before that, default 0.8): thinner bars per author request
    ax_gap.set_yticks(y)
    ax_gap.set_yticklabels(pairs, fontsize=5.5)  # was max(5, tick_label_fontsize-1)=7: narrow panel
    # Dynamic xlim: cap at 1.0, add 10% headroom above observed max
    gap_max = float(np.nanmax(gap_vals)) if np.any(~np.isnan(gap_vals)) else 0.5
    ax_gap.set_xlim(0, min(1.0, gap_max * 1.1))
    ax_gap.xaxis.set_major_locator(mticker.MaxNLocator(nbins=3))
    ax_gap.tick_params(axis="x", labelsize=6)
    ax_gap.set_xlabel(
        "Largest label gap\n(pooled)", fontsize=7  # was axis_label_fontsize=10, one line: collided with ax_di's label at this panel width
    )
    # left spine restored per author request (2026-07-03) -- was hidden to rely on
    # category labels alone; author wants the axis line visible.
    ax_gap.tick_params(left=True, labelleft=True)

    _save(fig, base / "figures_final/appD_worst_di_and_label_gap_pooled.pdf")


In [ ]:
appD_worst_di_and_label_gap_pooled(data_dir=GFF_DATA_DIR)


### `appE_annotation_label_gap_by_level`

**Output file:** `outputs/figures_final/appE_annotation_label_gap_by_level.pdf`

**Source TSV(s):** `outputs/stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`

**Pipeline stage:** Stage 4b (`stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`)

**Docstring (verbatim from source):**

```
Three-panel horizontal bar: absolute correct-labeling-rate gap per pair × level.

Source files:
- stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv

Visual encoding:
- All bars in red (labeling gaps are always negative findings)
- Sorted descending (largest gap at top)
- Only stable pairs (unstable_small_n == False) with non-null gap
```


In [ ]:
def appE_annotation_label_gap_by_level(data_dir: str = "outputs/") -> None:
    """
    Three-panel horizontal bar: absolute correct-labeling-rate gap per pair × level.

    Source files:
    - stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv

    Visual encoding:
    - All bars in red (labeling gaps are always negative findings)
    - Sorted descending (largest gap at top)
    - Only stable pairs (unstable_small_n == False) with non-null gap
    """
    apply_style()
    base = Path(data_dir)
    s4b_p = base / "stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv"
    if not _need([s4b_p], "appE"):
        return

    df = _read(s4b_p)
    if df.empty:
        return

    df = df[~df["unstable_small_n"].astype(bool)].dropna(
        subset=["labeling_rate_gap_abs"]
    )

    LEVELS = ["L2", "L3", "L4"]
    fig, axes = plt.subplots(
        1, 3, figsize=(6.0, 2.4), constrained_layout=True  # height 3.0->2.4: matches fig2's per-row density (author confirmed fig2 looks right)
    )

    # Max bar count across panels: gives every bar the same physical thickness
    # regardless of panel, so a sparse panel reads as "mostly empty" rather
    # than a single bar stretched to fill the whole axis.
    max_n = max(1, max((df["coding_level"] == lvl).sum() for lvl in LEVELS))

    for ax, level in zip(axes, LEVELS):
        ax.set_title("")
        sub = df[df["coding_level"] == level].sort_values(
            "labeling_rate_gap_abs", ascending=False
        )

        if sub.empty:
            ax.axis("off")
            continue

        pairs = [f"{a} vs {b}" for a, b in zip(sub["target_a"], sub["target_b"])]
        gaps = sub["labeling_rate_gap_abs"].values
        n = len(pairs)
        ROW_SPACING = 0.5  # was 0.6: further tightened row spacing per author request
        y = np.arange(n) * ROW_SPACING

        ax.barh(y, gaps, color=STYLE["colors"]["fail"], height=0.35)  # was 0.4: further thinned per author request
        ax.set_yticks(y)
        ax.set_yticklabels(pairs, fontsize=5)  # kept at 5, NOT bumped: full "X vs Y" pair strings are long enough that bumping to 6 made labels bleed across panels even with extra wspace; correctness over the cosmetic bump here
        x_max = float(gaps.max()) * 1.15 if len(gaps) else 1.0  # per-panel dynamic range (was fixed [0,1] across all panels) -- author wants each subplot scaled to its own data
        ax.set_xlim(0, x_max)
        ax.set_xticks([0, x_max / 2, x_max])  # midpoint marker computed per panel, not a hardcoded 0.5
        ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
        ax.tick_params(axis="x", labelsize=7)  # bumped from 6, rotation removed: panel is wide enough now
        ax.set_ylim(-0.5 * ROW_SPACING, (max_n - 0.5) * ROW_SPACING)  # consistent bar thickness across panels, scaled by ROW_SPACING
        ax.set_xlabel(
            "Label gap\n(|Δ correct rate|)", fontsize=7  # was axis_label_fontsize=10
        )

        _panel_label(ax, level, STYLE["colors"][level])
        # left spine restored per author request (2026-07-03) -- was hidden to rely on
        # category labels alone; author wants the axis line visible.
        ax.tick_params(left=True, labelleft=True)

    _save(fig, base / "figures_final/appE_annotation_label_gap_by_level.pdf")


In [ ]:
appE_annotation_label_gap_by_level(data_dir=GFF_DATA_DIR)


### `appF_cross_level_deltas`

**Output file:** `outputs/figures_final/appF_cross_level_deltas.pdf`

**Source TSV(s):** `outputs/stage3/s3_cross_level_consistency.tsv`

**Pipeline stage:** Stage 3 (`stage3/s3_cross_level_consistency.tsv`)

**Docstring (verbatim from source):**

```
Two-panel slope chart (stacked vertically): presence rate and
correct-labeling rate plotted directly at each coding level, one line per
target group connecting its values across L2 -> L3 -> L4.

Source files:
- stage3/s3_cross_level_consistency.tsv

Visual encoding:
- Top panel: presence_rate per level. Bottom panel: correct_labeling_rate
  per level. Same y-domain [0, 1] in both, sharing the x-axis.
- One line per target group. Only groups with a non-null value at all
  three levels (L2, L3, L4) in *both* presence_rate and
  correct_labeling_rate are plotted, so every line spans the full axis
  and the group set is identical in both panels.
- Color encodes the group's reporting dimension (race, religion, lgbtq,
  gender, origin, politics); marker shape distinguishes individual groups
  sharing a dimension. The same (color, marker) pair is used for a given
  group in both panels.
- Each line is labeled directly at its rightmost available point instead
  of via a legend; overlapping labels are greedily separated in pixel
  space (see _declutter_labels).

Design rationale: the previous 39-row diverging bar chart (one row per
group x transition) required a separate delta number per group to convey
a within-group change. Plotting the raw rates as connected lines lets the
slope itself communicate the change, and consolidates each group into a
single visual element spanning both panels.
```


In [ ]:
def appF_cross_level_deltas(data_dir: str = "outputs/") -> None:
    """
    Two-panel slope chart (stacked vertically): presence rate and
    correct-labeling rate plotted directly at each coding level, one line per
    target group connecting its values across L2 -> L3 -> L4.

    Source files:
    - stage3/s3_cross_level_consistency.tsv

    Visual encoding:
    - Top panel: presence_rate per level. Bottom panel: correct_labeling_rate
      per level. Same y-domain [0, 1] in both, sharing the x-axis.
    - One line per target group. Only groups with a non-null value at all
      three levels (L2, L3, L4) in *both* presence_rate and
      correct_labeling_rate are plotted, so every line spans the full axis
      and the group set is identical in both panels.
    - Color encodes the group's reporting dimension (race, religion, lgbtq,
      gender, origin, politics); marker shape distinguishes individual groups
      sharing a dimension. The same (color, marker) pair is used for a given
      group in both panels.
    - Each line is labeled directly at its rightmost available point instead
      of via a legend; overlapping labels are greedily separated in pixel
      space (see _declutter_labels).

    Design rationale: the previous 39-row diverging bar chart (one row per
    group x transition) required a separate delta number per group to convey
    a within-group change. Plotting the raw rates as connected lines lets the
    slope itself communicate the change, and consolidates each group into a
    single visual element spanning both panels.
    """
    apply_style()
    base = Path(data_dir)
    s3_p = base / "stage3/s3_cross_level_consistency.tsv"
    if not _need([s3_p], "appF"):
        return

    df = _read(s3_p)
    if df.empty:
        return

    # Validate required columns
    required = {
        "taxonomy_level",
        "target",
        "coding_level_from",
        "coding_level_to",
        "presence_rate_from",
        "presence_rate_to",
    }
    missing = required - set(df.columns)
    if missing:
        print(f"WARNING: appF: skipping — missing columns {missing}")
        return

    # Resolve labeling rate columns (accept either naming variant)
    if {"correct_labeling_rate_from", "correct_labeling_rate_to"} <= set(df.columns):
        lab_from_col, lab_to_col = (
            "correct_labeling_rate_from",
            "correct_labeling_rate_to",
        )
    elif {"labeling_rate_from", "labeling_rate_to"} <= set(df.columns):
        lab_from_col, lab_to_col = "labeling_rate_from", "labeling_rate_to"
    else:
        lab_from_col = lab_to_col = None
        print(
            "WARNING: appF: no labeling rate from/to columns found — "
            "right panel will be empty."
        )

    # Same group-exclusion logic as the prior delta version: pan-category and
    # self-referential targets excluded from pipeline reporting.
    EXCLUDE_TARGETS = {
        "referential_white_supremacist",
        "minority",
        "unknown_minority",
        "other",
    }
    df = df[~df["target"].isin(EXCLUDE_TARGETS)].copy()

    LEVELS = ["L2", "L3", "L4"]

    # Dimension display mapping, consistent with fig1_coverage_heatmap.
    DIM_MAP = {
        "race": "race",
        "religion": "religion",
        "sexuality": "lgbtq",
        "gender": "gender",
        "origin": "origin",
        "politics": "politics",
        "disability": "disability",
    }
    DIM_ORDER = [
        "race",
        "religion",
        "lgbtq",
        "gender",
        "origin",
        "politics",
        "disability",
    ]
    DIM_COLORS = {
        "race": "#0072B2",
        "religion": "#D55E00",
        "lgbtq": "#009E73",
        "gender": "#CC79A7",
        "origin": "#E69F00",
        "politics": "#56B4E9",
        "disability": "#999999",
    }
    MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]

    groups: list[tuple[str, str]] = sorted(
        df[["taxonomy_level", "target"]]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    )
    if not groups:
        print("WARNING: appF: no groups remain after filtering — skipping.")
        return

    # Build per-group level -> rate dicts from the from/to row pairs.
    pres_series: dict[tuple[str, str], dict[str, float]] = {}
    lab_series: dict[tuple[str, str], dict[str, float]] = {}
    for tl, tg in groups:
        sub = df[(df["taxonomy_level"] == tl) & (df["target"] == tg)]
        pres: dict[str, float] = {}
        lab: dict[str, float] = {}
        for _, row in sub.iterrows():
            pres[row["coding_level_from"]] = row["presence_rate_from"]
            pres[row["coding_level_to"]] = row["presence_rate_to"]
            if lab_from_col is not None:
                lab[row["coding_level_from"]] = row[lab_from_col]
                lab[row["coding_level_to"]] = row[lab_to_col]
        pres_series[(tl, tg)] = pres
        lab_series[(tl, tg)] = lab

    # Keep only groups with a complete L2, L3, L4 trajectory in *both* metrics,
    # so every line spans the full x-axis and the same group set appears in
    # both panels (required for cross-panel color/marker tracking).
    def _is_complete(level_dict: dict[str, float]) -> bool:
        return all(
            lvl in level_dict and not pd.isna(level_dict[lvl]) for lvl in LEVELS
        )

    groups = [
        g
        for g in groups
        if _is_complete(pres_series[g]) and _is_complete(lab_series[g])
    ]
    if not groups:
        print("WARNING: appF: no groups have complete L2/L3/L4 data in both "
              "metrics — skipping.")
        return

    # Assign (color, marker): color by dimension, marker cycles within dimension.
    group_style: dict[tuple[str, str], tuple[str, str]] = {}
    for dim in DIM_ORDER:
        dim_groups = [g for g in groups if DIM_MAP.get(g[0]) == dim]
        for i, g in enumerate(dim_groups):
            group_style[g] = (DIM_COLORS[dim], MARKERS[i % len(MARKERS)])

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(STYLE["textwidth_in"], 6.5),  # height bumped: label-crowding fix (see below)
        constrained_layout=True,
    )

    fig.set_constrained_layout_pads(hspace=0.12, wspace=0.0)

    panel_defs = [
        ("Presence rate", pres_series, axes[0]),
        ("Correct-labeling rate", lab_series, axes[1]),
    ]

    # Stretch the inter-level spacing so the lines use more of the panel's
    # horizontal room instead of being compressed into its left half, while
    # the right-hand pad (reserved for end-of-line labels) stays small
    # enough that the longest label string still fits without overflowing.
    LEVEL_SPACING = 2.2
    RIGHT_PAD = 1.3
    x_idx = [i * LEVEL_SPACING for i in range(len(LEVELS))]
    label_fontsize = STYLE["annotation_fontsize"] + 1.5
    summaries: list[str] = []

    for title, series_map, ax in panel_defs:
        ax.set_title("")

        end_points: list[tuple[str, float, float, str]] = []
        for g in groups:
            series = series_map.get(g, {})
            ys = [series.get(lvl, np.nan) for lvl in LEVELS]
            ys = [float(v) if v is not None and not pd.isna(v) else np.nan for v in ys]
            if all(np.isnan(v) for v in ys):
                continue

            color, marker = group_style[g]
            ax.plot(
                x_idx,
                ys,
                color=color,
                marker=marker,
                markersize=4.5,
                linewidth=1.3,
                alpha=0.85,
                clip_on=False,
                zorder=3,
            )

            valid_idx = [i for i, v in enumerate(ys) if not np.isnan(v)]
            x_final = x_idx[valid_idx[-1]]
            y_final = ys[valid_idx[-1]]
            tl, tg = g
            end_points.append((f"{DIM_MAP.get(tl, tl)}:{tg}", x_final, y_final, color))

        ax.set_xlim(-0.15 * LEVEL_SPACING, x_idx[-1] + RIGHT_PAD)
        ax.set_xticks(x_idx)
        ax.set_xticklabels(LEVELS)
        ax.set_ylim(0, 1.0)
        ax.set_ylabel(title, fontsize=STYLE["axis_label_fontsize"])
        ax.tick_params(axis="x", labelsize=STYLE["tick_label_fontsize"])
        ax.tick_params(axis="y", labelsize=STYLE["tick_label_fontsize"])

        _declutter_labels(ax, fig, end_points, label_fontsize)

        summaries.append(f"  {title}: {len(end_points)} groups plotted")

    _save(fig, base / "figures_final/appF_cross_level_deltas.pdf")

    print("appF summary:")
    for s in summaries:
        print(s)


In [ ]:
appF_cross_level_deltas(data_dir=GFF_DATA_DIR)


### `appG_elsherief_pairwise_di_delta`

**Output file:** `outputs/figures_final/appG_elsherief_pairwise_di_delta.pdf`

**Source TSV(s):** `outputs/stage4/elsherief/s4c_pairwise_delta_union_vs_elsherief.tsv`

**Pipeline stage:** Stage 4c (`stage4/elsherief/s4c_pairwise_delta_union_vs_elsherief.tsv`)

**Docstring (verbatim from source):**

```
Scatter plot: Δ worst DI (x) vs Δ label gap (y) for group pairs comparing union to ElSherief.

Source files:
- stage4/elsherief/s4c_pairwise_delta_union_vs_elsherief.tsv

Visual encoding:
- Each point = one group pair with stable estimates in both conditions
- Dashed gray lines at x=0 and y=0 divide the four quadrants
- Points labeled with pair names in 7pt font
- Points above y=0 line = worse label gap in union; below = better label gap
```


In [ ]:
def appG_elsherief_pairwise_di_delta(data_dir: str = "outputs/") -> None:
    """
    Scatter plot: Δ worst DI (x) vs Δ label gap (y) for group pairs comparing union to ElSherief.

    Source files:
    - stage4/elsherief/s4c_pairwise_delta_union_vs_elsherief.tsv

    Visual encoding:
    - Each point = one group pair with stable estimates in both conditions
    - Dashed gray lines at x=0 and y=0 divide the four quadrants
    - Points labeled with pair names in 7pt font
    - Points above y=0 line = worse label gap in union; below = better label gap
    """
    apply_style()
    base = Path(data_dir)
    s4c_p = base / "stage4/elsherief/s4c_pairwise_delta_union_vs_elsherief.tsv"
    if not _need([s4c_p], "appG"):
        return

    df = _read(s4c_p)
    if df.empty:
        return

    df = (
        df[
            ~df["unstable_small_n_union"].astype(bool)
            & ~df["unstable_small_n_elsherief"].astype(bool)
        ]
        .dropna(
            subset=[
                "worst_di_ratio_delta_union_minus_elsherief",
                "label_gap_abs_delta_union_minus_elsherief",
            ]
        )
        .copy()
    )

    if df.empty:
        print("WARNING: appG: no stable pairs in both conditions")
        return

    x = df["worst_di_ratio_delta_union_minus_elsherief"].values
    y_vals = df["label_gap_abs_delta_union_minus_elsherief"].values
    pair_labels = [f"{a}/{b}" for a, b in zip(df["target_a"], df["target_b"])]

    fig, ax = plt.subplots(
        figsize=(STYLE["columnwidth_in"], STYLE["columnwidth_in"]),
        constrained_layout=True,
    )
    ax.set_title("")

    ax.scatter(x, y_vals, s=30, color=STYLE["colors"]["presence"], alpha=0.8, zorder=3)

    for xi, yi, lbl in zip(x, y_vals, pair_labels):
        ax.text(xi, yi, f" {lbl}", fontsize=7, va="center", ha="left")

    ax.axvline(0, color="gray", linewidth=0.8, linestyle="--", zorder=1)
    ax.axhline(0, color="gray", linewidth=0.8, linestyle="--", zorder=1)

    # Two-line labels to fit within column width (3.35 in)
    ax.set_xlabel(
        "Δ worst DI\n(union − ElSherief)", fontsize=STYLE["axis_label_fontsize"]
    )
    ax.set_ylabel(
        "Δ label gap abs\n(union − ElSherief)", fontsize=STYLE["axis_label_fontsize"]
    )

    _save(fig, base / "figures_final/appG_elsherief_pairwise_di_delta.pdf")


In [ ]:
appG_elsherief_pairwise_di_delta(data_dir=GFF_DATA_DIR)


## Pairwise annotation DI figure (`generate_fig2_annotation_di_pairwise.py`)

**Output file:** `outputs/figures_final/fig2_annotation_di_by_pair_level.pdf`
(or `..._v2.pdf` if the primary name already exists -- see `make_figure`
below, which deliberately refuses to overwrite an existing file. That
built-in guard is why running this cell will not clobber any existing
production artifact.)

**Source TSVs read (two, explicitly merged by this script itself):**
- `stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv` -- **Stage 4b**
- `stage4/by_group/s4a_pairwise_disparity_by_group.tsv` -- **Stage 4a**

**Pipeline stage:** Stage 4 (both 4a and 4b, merged with 4b preferred on key
overlap -- see `load_and_merge` below).

This script's own docstring says it matches "Figure 9
(fig5_worst_di_by_pair_level.pdf)" in style, but its output filename uses a
`fig2_` prefix -- a naming inconsistency between the code comment and the
actual artifact name, left as found.

Reuses `STYLE`, `apply_style`, `_read`, `_save`, `_panel_label` from the
`generate_figures_final.py` utilities section above (imported once, not
duplicated), matching the source file's own
`from generate_figures_final import STYLE, _panel_label, _read, _save, apply_style`.

Hardcodes `DI_THRESHOLD = 0.80` as a local module constant rather than
importing `audit_pipeline.config.DI_THRESHOLD` (numerically identical, 0.8 ==
0.80, but a second, independent definition of the same constant in a third
place -- flagged, not consolidated, since consolidating it would be a logic
change beyond copy-paste).


In [ ]:
DATA_DIR = OUTPUTS_DIR  # absolute; original source hardcoded relative Path("outputs/")
S4B_PATH = DATA_DIR / "stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv"
S4A_PATH = DATA_DIR / "stage4/by_group/s4a_pairwise_disparity_by_group.tsv"
OUT_PATH = DATA_DIR / "figures_final/fig2_annotation_di_by_pair_level.pdf"

DI_THRESHOLD = 0.80
KEY_COLS = ["target_a", "target_b", "coding_level"]
LEVELS = ["L2", "L3", "L4"]


In [ ]:
def _filter_stable(df: pd.DataFrame) -> pd.DataFrame:
    return df[~df["unstable_small_n"].astype(bool)].dropna(subset=["annotation_di_ratio"])

def load_and_merge() -> pd.DataFrame:
    s4b_stable = _filter_stable(_read(S4B_PATH)).copy()
    s4a_stable = _filter_stable(_read(S4A_PATH)).copy()
    s4b_stable["source_file"] = "s4b"
    s4a_stable["source_file"] = "s4a"

    s4b_keys = set(map(tuple, s4b_stable[KEY_COLS].values))
    s4a_keys = set(map(tuple, s4a_stable[KEY_COLS].values))
    overlap_keys = s4b_keys & s4a_keys

    print("=" * 80)
    print("OVERLAP CHECK (s4a vs s4b, stable rows only)")
    print("=" * 80)
    if overlap_keys:
        print(f"{len(overlap_keys)} pair x coding_level combination(s) appear in BOTH files. "
              f"Using s4b value (finer granularity) for these; dropping the s4a duplicate.")
        overlap_mask_a = s4a_stable[KEY_COLS].apply(tuple, axis=1).isin(overlap_keys)
        overlap_mask_b = s4b_stable[KEY_COLS].apply(tuple, axis=1).isin(overlap_keys)
        cols = KEY_COLS + ["annotation_di_ratio"]
        print("-- overlapping s4a rows (dropped) --")
        print(s4a_stable[overlap_mask_a][cols].to_string(index=False))
        print("-- overlapping s4b rows (kept) --")
        print(s4b_stable[overlap_mask_b][cols].to_string(index=False))
        s4a_stable = s4a_stable[~overlap_mask_a]
    else:
        print("No overlapping pair x coding_level combinations found.")
    print("=" * 80 + "\n")

    return pd.concat([s4b_stable, s4a_stable], ignore_index=True)

def build_pair_table(merged: pd.DataFrame) -> pd.DataFrame:
    # target_a/target_b already use the paper's canonical casing (e.g. "LGB",
    # "Trans/NB"; lowercase for everything else) — do not force-lowercase here,
    # or acronyms/proper nouns lose their casing.
    out = pd.DataFrame(
        {
            "coding_level": merged["coding_level"],
            "pair_label": [
                f"{a} vs {b}" for a, b in zip(merged["target_a"], merged["target_b"])
            ],
            "annotation_di_ratio": merged["annotation_di_ratio"],
            "passes_4_5_rule": merged["annotation_di_ratio"] >= DI_THRESHOLD,
            "source_file": merged["source_file"],
        }
    )
    level_order = {lvl: i for i, lvl in enumerate(LEVELS)}
    out = out.assign(_lvl=out["coding_level"].map(level_order))
    out = out.sort_values(["_lvl", "annotation_di_ratio"], ascending=[True, True])
    return out.drop(columns="_lvl").reset_index(drop=True)

def compute_level_candidate_counts(raw_s4b: pd.DataFrame, raw_s4a: pd.DataFrame) -> dict[str, dict[str, int]]:
    """Per-level count of all candidate pairs (pre stability-filter) vs. how many
    were excluded as unstable_small_n. Used to explain sparse panels honestly."""
    s4b = raw_s4b.copy()
    s4a = raw_s4a.copy()
    s4b_keys = set(map(tuple, s4b[KEY_COLS].values))
    s4a_unique = s4a[~s4a[KEY_COLS].apply(tuple, axis=1).isin(s4b_keys)]
    raw_merged = pd.concat([s4b, s4a_unique], ignore_index=True)

    counts = {}
    for level in LEVELS:
        sub = raw_merged[raw_merged["coding_level"] == level]
        total = len(sub)
        unstable = int(sub["unstable_small_n"].astype(bool).sum())
        counts[level] = {"total": total, "unstable": unstable, "stable": total - unstable}
    return counts

def make_figure(table: pd.DataFrame, level_counts: dict[str, dict[str, int]]) -> Path:
    apply_style()
    fig, axes = plt.subplots(
        1, 3, figsize=(STYLE["textwidth_in"], STYLE["textwidth_in"] * 0.26), constrained_layout=True  # was *0.34 (before that, *0.45, *2/3): further vertical compression per author request
    )

    panel_counts: dict[str, int] = {}
    panel_fails: dict[str, list[str]] = {}

    # Max bar count across panels: used to give every bar the same physical
    # thickness regardless of panel, so a sparse panel reads as "mostly empty"
    # rather than "one giant bar" — the resulting empty rows are then labeled.
    max_n = max(
        1, max(len(table[table["coding_level"] == lvl]) for lvl in LEVELS)
    )

    for ax, level in zip(axes, LEVELS):
        ax.set_title("")
        sub = table[table["coding_level"] == level].sort_values(
            "annotation_di_ratio", ascending=True
        )
        n = len(sub)

        if sub.empty:
            ax.axis("off")
            ax.text(
                0.5,
                0.5,
                "No stable pairs\nat this level",
                ha="center",
                va="center",
                transform=ax.transAxes,
                fontsize=STYLE["annotation_fontsize"],
                color=STYLE["colors"]["neutral"],
            )
            _panel_label(ax, level, STYLE["colors"][level])
            panel_counts[level] = 0
            panel_fails[level] = []
            continue

        pairs = sub["pair_label"].tolist()
        di = sub["annotation_di_ratio"].values
        ROW_SPACING = 0.35  # was 0.45: further tightened row spacing per author request
        y = np.arange(n) * ROW_SPACING

        colors = [
            STYLE["colors"]["pass"] if v >= DI_THRESHOLD else STYLE["colors"]["fail"]
            for v in di
        ]

        ax.barh(y, di, color=colors, height=0.22)  # was 0.3 (before that, 0.4, 0.6, 0.8): further thinned per author request
        ax.axvline(DI_THRESHOLD, color="red", linewidth=1.2, linestyle="--")

        ax.set_yticks(y)
        ax.set_yticklabels(pairs, fontsize=8)  # panels are now full textwidth/3 each (was 0.6x, narrower): room for larger labels again
        ax.set_xlim(0.0, 1.0)
        ax.set_xticks([0.0, 0.5, 1.0])  # restored middle tick: panels wide enough now that 3 ticks no longer collide
        ax.tick_params(axis="x", labelsize=7)
        # Fix the y-range to the busiest panel's bar count so bar thickness is
        # consistent across panels; sparse panels get genuine blank space
        # instead of a single bar stretched to fill the whole axis.
        ax.set_ylim(-0.5 * ROW_SPACING, (max_n - 0.5) * ROW_SPACING)  # scaled by ROW_SPACING for consistent bar thickness across panels
        ax.set_xlabel("Annotation DI ratio", fontsize=STYLE["axis_label_fontsize"])  # restored to standard size: panels wide enough now

        _panel_label(ax, level, STYLE["colors"][level])
        # left spine restored per author request (2026-07-03) -- was hidden to rely on
        # category labels alone; author wants the axis line visible.
        ax.tick_params(left=True, labelleft=True)

        # 2026-07-03: the sparse-panel explanatory note (gray text) that used to
        # render here was removed at the author's request -- it overlaid the L4
        # panel's visible bar/label. panel_counts/panel_fails below still feed the
        # printed candidate-count summary at the end of this cell.
        panel_counts[level] = n
        panel_fails[level] = [p for p, v in zip(pairs, di) if v < DI_THRESHOLD]

    out_path = OUT_PATH
    if out_path.exists():
        out_path = out_path.with_name(out_path.stem + "_v2" + out_path.suffix)
        print(f"FLAG: {OUT_PATH.name} already exists — saving as {out_path.name} instead.\n")

    _save(fig, out_path)

    print(f"\nOutput file: {out_path}")
    for level in LEVELS:
        print(f"  {level}: {panel_counts[level]} bars")
    for level in LEVELS:
        fails = panel_fails[level]
        print(f"  {level} red (fail) pairs: {fails if fails else 'none'}")

    print("\nStyle parameters matched to Figure 9 (fig5_worst_di_by_pair_level.pdf):")
    print(f"  figsize = ({STYLE['textwidth_in']}, {STYLE['textwidth_in'] * 0.26}) in, dpi = {STYLE['dpi']}")
    print(
        f"  font.size (tick label) = {STYLE['tick_label_fontsize']}, "
        f"axis label fontsize = {STYLE['axis_label_fontsize']}, "
        f"panel label fontsize = {STYLE['panel_label_fontsize']}"
    )
    print(
        f"  pass (blue) color = {STYLE['colors']['pass']}, "
        f"fail (red) color = {STYLE['colors']['fail']}"
    )
    print("  threshold line: color=red, linewidth=1.2, linestyle='--'")
    print(
        f"  panel label colors: L2={STYLE['colors']['L2']}, "
        f"L3={STYLE['colors']['L3']}, L4={STYLE['colors']['L4']}"
    )
    return out_path


In [ ]:
def main() -> None:
    merged = load_and_merge()
    table = build_pair_table(merged)
    level_counts = compute_level_candidate_counts(_read(S4B_PATH), _read(S4A_PATH))

    print("=" * 80)
    print("VERIFICATION TABLE — stable pairwise annotation DI ratios by coding level")
    print("=" * 80)
    print(table.to_string(index=False))
    print("=" * 80 + "\n")

    print("Candidate pair counts by level (context for sparse panels):")
    for level in LEVELS:
        c = level_counts[level]
        print(
            f"  {level}: {c['stable']} stable / {c['total']} candidate "
            f"({c['unstable']} excluded as unstable_small_n)"
        )
    print()

    make_figure(table, level_counts)


In [ ]:
main()


## DEPRECATED exploratory pipeline -- `auditing/03_audit_visualizations.ipynb`

**This is a separate pipeline from `audit_pipeline/`**, with its own
Stage 00-03 notebooks (`auditing/00_coverage_audit.ipynb` through
`auditing/03_audit_visualizations.ipynb`, see `auditing/README.md`). Its
outputs currently live under `outputs/deprecated/{coverage,annotation,disparity}_audits/`
(originally `outputs/{coverage,annotation,disparity}_audits/` per the
notebook's own source -- **that top-level path no longer exists**, the data
was relocated under `deprecated/` at some point after this notebook was
last run). The cells below are otherwise byte-for-byte copies of the
original notebook's cells; only the three input directory assignments were
updated to point at the relocated data so this section can execute at all.
The output directory (`viz_output_dir`) is left exactly as the original
source wrote it (`outputs/audit_visualizations/`, not moved under
`deprecated/`) -- the source code never said to move it, so this notebook
doesn't move it either; running this section for the first time in a while
will (re)create that directory.

**Structural finding surfaced by cross-checking this pipeline against
`audit_pipeline` (see Step 3 below):** `outputs/deprecated/coverage_audits/audit_metrics.tsv`,
`outputs/deprecated/annotation_audits/annotation_quality.tsv`, and
`outputs/deprecated/disparity_audits/coverage_disparity.tsv` have **no
`coding_level` column at all** -- every metric here is pooled across all
Mendelsohn dogwhistle types (L1-L4) for a given `(taxonomy_level, target)`,
whereas every `audit_pipeline` stage stratifies by `coding_level`. A
same-named metric (`presence_rate`, `correct_labeling_rate`,
`presence_rate_di_ratio`, ...) computed by this deprecated pipeline is
therefore **not a like-for-like numeric re-derivation** of the corresponding
`audit_pipeline` metric -- it's a coarser aggregate. Step 3 logs this as
"not directly comparable -- different aggregation granularity" rather than
joining them and reporting a spurious mismatch.

**Figures produced** (all under `outputs/audit_visualizations/`):
`coverage_top20_token_frequency.png`, `coverage_presence_vs_type.png`,
`annotation_quality_top20_rates.png`, `annotation_case_ab_top15.png`,
`disparity_di_ratio_histograms.png`, `disparity_worst_di_top20.png`,
`annotation_disparity_label_gap_top20.png`, `cross_level_deltas.png`. A final
diagnostic (non-figure) cell prints high-confidence misclassified dogwhistle
forms; reproduced too since it's part of this file's real content.


### Setup + self-referential filtering (adapted paths, otherwise verbatim)


In [ ]:
DEPRECATED_DIR = OUTPUTS_DIR / "deprecated"
coverage_dir = DEPRECATED_DIR / "coverage_audits"       # was: OUTPUTS_DIR / 'coverage_audits'
annotation_dir = DEPRECATED_DIR / "annotation_audits"    # was: OUTPUTS_DIR / 'annotation_audits'
disparity_dir = DEPRECATED_DIR / "disparity_audits"      # was: OUTPUTS_DIR / 'disparity_audits'
viz_output_dir = OUTPUTS_DIR / 'audit_visualizations'    # unchanged from source
viz_output_dir.mkdir(parents=True, exist_ok=True)

def read_tsv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path}')
    return pd.read_csv(path, sep='\t')

audit_metrics = read_tsv(coverage_dir / 'audit_metrics.tsv')
audit_detailed = read_tsv(coverage_dir / 'audit_detailed.tsv')
audit_matches = read_tsv(coverage_dir / 'audit_matches.tsv')
annotation_quality = read_tsv(annotation_dir / 'annotation_quality.tsv')
case_breakdown = read_tsv(annotation_dir / 'case_breakdown.tsv')
form_labeling = read_tsv(annotation_dir / 'form_labeling_detail.tsv')
coverage_disparity = read_tsv(disparity_dir / 'coverage_disparity.tsv')
annotation_disparity = read_tsv(disparity_dir / 'annotation_disparity.tsv')
cross_level_consistency = read_tsv(disparity_dir / 'cross_level_consistency.tsv')

# Define self-referential categories from form-level type annotations.
self_ref_mask = form_labeling['type'].fillna('').astype(str).str.contains('self-referential', case=False)
self_ref_rows = form_labeling[self_ref_mask].copy()
self_ref_pairs = set(
    zip(
        self_ref_rows['taxonomy_level'].astype(str),
        self_ref_rows['target'].astype(str),
    )
)
self_ref_triples = set(
    zip(
        self_ref_rows['taxonomy_level'].astype(str),
        self_ref_rows['target'].astype(str),
        self_ref_rows['dogwhistle'].astype(str),
    )
)

def exclude_self_ref_pairs(df: pd.DataFrame, level_col: str, target_col: str) -> pd.DataFrame:
    key = list(zip(df[level_col].astype(str), df[target_col].astype(str)))
    mask = [k not in self_ref_pairs for k in key]
    return df.loc[mask].copy()

def exclude_self_ref_triples(df: pd.DataFrame, level_col: str, target_col: str, dogwhistle_col: str) -> pd.DataFrame:
    key = list(zip(df[level_col].astype(str), df[target_col].astype(str), df[dogwhistle_col].astype(str)))
    mask = [k not in self_ref_triples for k in key]
    return df.loc[mask].copy()

# Build filtered views used by all visualizations.
audit_metrics_viz = exclude_self_ref_pairs(audit_metrics, 'taxonomy_level', 'target')
audit_detailed_viz = exclude_self_ref_triples(audit_detailed, 'taxonomy_level', 'target', 'dogwhistle')
audit_matches_viz = exclude_self_ref_triples(audit_matches, 'taxonomy_level', 'target', 'dogwhistle')
annotation_quality_viz = exclude_self_ref_pairs(annotation_quality, 'taxonomy_level', 'target')
case_breakdown_viz = exclude_self_ref_pairs(case_breakdown, 'taxonomy_level', 'target')
form_labeling_viz = exclude_self_ref_triples(form_labeling, 'taxonomy_level', 'target', 'dogwhistle')

coverage_disparity_viz = coverage_disparity[
    ~coverage_disparity.apply(
        lambda row: (
            (str(row['taxonomy_level']), str(row['target_a'])) in self_ref_pairs
            or (str(row['taxonomy_level']), str(row['target_b'])) in self_ref_pairs
        ),
        axis=1,
    )
].copy()

annotation_disparity_viz = annotation_disparity[
    ~annotation_disparity.apply(
        lambda row: (
            (str(row['taxonomy_level']), str(row['target_a'])) in self_ref_pairs
            or (str(row['taxonomy_level']), str(row['target_b'])) in self_ref_pairs
        ),
        axis=1,
    )
].copy()

cross_level_consistency_viz = cross_level_consistency[
    ~cross_level_consistency.apply(
        lambda row: (
            (str(row['level_from']), str(row['target'])) in self_ref_pairs
            or (str(row['level_to']), str(row['target'])) in self_ref_pairs
        ),
        axis=1,
    )
].copy()

coarse_levels = sorted(audit_metrics_viz['taxonomy_level'].dropna().astype(str).unique().tolist())

def make_facet_axes(levels: list[str], ncols: int = 3, panel_w: float = 5.0, panel_h: float = 4.0):
    n = len(levels)
    ncols = max(1, min(ncols, n))
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel_w * ncols, panel_h * nrows))
    axes = np.atleast_1d(axes).reshape(nrows, ncols).flatten()
    for idx in range(len(axes)):
        if idx >= n:
            axes[idx].axis('off')
    return fig, axes

print('Loaded datasets:')
for name, df in [
    ('audit_metrics', audit_metrics),
    ('audit_detailed', audit_detailed),
    ('audit_matches', audit_matches),
    ('annotation_quality', annotation_quality),
    ('case_breakdown', case_breakdown),
    ('form_labeling', form_labeling),
    ('coverage_disparity', coverage_disparity),
    ('annotation_disparity', annotation_disparity),
    ('cross_level_consistency', cross_level_consistency),
]:
    print(f'  - {name}: {len(df)} rows')

print('\nRemoved self-referential rows for visualizations:')
for name, full_df, filtered_df in [
    ('audit_metrics', audit_metrics, audit_metrics_viz),
    ('audit_detailed', audit_detailed, audit_detailed_viz),
    ('audit_matches', audit_matches, audit_matches_viz),
    ('annotation_quality', annotation_quality, annotation_quality_viz),
    ('case_breakdown', case_breakdown, case_breakdown_viz),
    ('form_labeling', form_labeling, form_labeling_viz),
    ('coverage_disparity', coverage_disparity, coverage_disparity_viz),
    ('annotation_disparity', annotation_disparity, annotation_disparity_viz),
    ('cross_level_consistency', cross_level_consistency, cross_level_consistency_viz),
]:
    print(f'  - {name}: removed {len(full_df) - len(filtered_df)} rows')

print('Facet levels:', coarse_levels)


### `coverage_top20_token_frequency.png`

**Output file:** `outputs/audit_visualizations/coverage_top20_token_frequency.png`

**Source TSV:** `audit_metrics.tsv` (in `outputs/deprecated/{coverage,annotation,disparity}_audits/`)

**Pipeline stage:** Stage 00 (deprecated `auditing/` numbering) of the deprecated `auditing/` pipeline (not
`audit_pipeline`'s Stage numbering).


In [ ]:
fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=5.6, panel_h=4.6)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    plot_df = (
        audit_metrics_viz[audit_metrics_viz['taxonomy_level'] == level]
        .sort_values('token_frequency', ascending=False)
        .head(10)
        .copy()
    )

    if plot_df.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{level.title()}: token frequency')
        ax.set_axis_off()
        continue

    plot_df = plot_df.sort_values('token_frequency', ascending=True)
    ax.barh(plot_df['target'], plot_df['token_frequency'], color='#2f6db5')
    ax.set_title(f'{level.title()}: token frequency')
    ax.set_xlabel('Token frequency')
    ax.set_ylabel('Target')

fig.suptitle(
    'Top token-frequency targets per coarse label (self-referential excluded)',
    y=0.99,
    fontsize=16,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(viz_output_dir / 'coverage_top20_token_frequency.png', dpi=180)
plt.show()


### `coverage_presence_vs_type.png`

**Output file:** `outputs/audit_visualizations/coverage_presence_vs_type.png`

**Source TSV:** `audit_metrics.tsv` (in `outputs/deprecated/{coverage,annotation,disparity}_audits/`)

**Pipeline stage:** Stage 00 of the deprecated `auditing/` pipeline (not
`audit_pipeline`'s Stage numbering).


In [ ]:
norm = mpl.colors.Normalize(
    vmin=max(float(audit_metrics_viz['token_frequency'].min()), 1.0),
    vmax=max(float(audit_metrics_viz['token_frequency'].max()), 1.0),
)

fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=5.6, panel_h=4.6)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    level_df = audit_metrics_viz[audit_metrics_viz['taxonomy_level'] == level].copy()

    if level_df.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{level.title()}: presence vs type coverage')
        ax.set_axis_off()
        continue

    sizes = np.clip(level_df['token_frequency'].to_numpy(), 1, None)
    sizes = 30 + 150 * (sizes / sizes.max())

    ax.scatter(
        level_df['presence_rate'],
        level_df['type_coverage'],
        s=sizes,
        c=level_df['token_frequency'],
        cmap='viridis',
        norm=norm,
        alpha=0.8,
        edgecolor='black',
        linewidth=0.3,
    )
    ax.set_title(f'{level.title()}: presence vs type coverage')
    ax.set_xlabel('Presence rate')
    ax.set_ylabel('Type coverage')
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)

fig.suptitle(
    'Coverage quality by coarse label (self-referential excluded)',
    y=0.99,
    fontsize=16,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0.10, 1, 0.95])

sm = mpl.cm.ScalarMappable(norm=norm, cmap='viridis')
sm.set_array([])
left = axes[0].get_position().x0
right = axes[1].get_position().x1
cax = fig.add_axes([left, 0.055, right - left, 0.028])
cbar = fig.colorbar(sm, cax=cax, orientation='horizontal')
cbar.set_label('Token frequency')

fig.savefig(viz_output_dir / 'coverage_presence_vs_type.png', dpi=180)
plt.show()


### `annotation_quality_top20_rates.png`

**Output file:** `outputs/audit_visualizations/annotation_quality_top20_rates.png`

**Source TSV:** `annotation_quality.tsv` (in `outputs/deprecated/{coverage,annotation,disparity}_audits/`)

**Pipeline stage:** Stage 01 of the deprecated `auditing/` pipeline (not
`audit_pipeline`'s Stage numbering).


In [ ]:
fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=6.2, panel_h=4.8)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    annot_plot = (
        annotation_quality_viz[annotation_quality_viz['taxonomy_level'] == level]
        .sort_values('total_matches', ascending=False)
        .head(8)
        .copy()
    )

    if annot_plot.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{level.title()}: annotation quality')
        ax.set_axis_off()
        continue

    x = np.arange(len(annot_plot))
    width = 0.4
    ax.bar(
        x - width / 2,
        annot_plot['correct_labeling_rate'],
        width=width,
        label='correct',
        color='#009E73',
    )
    ax.bar(
        x + width / 2,
        annot_plot['annotator_failure_ratio'],
        width=width,
        label='failure',
        color='#D55E00',
    )
    ax.set_xticks(x)
    ax.set_xticklabels(annot_plot['target'], rotation=50, ha='right')
    ax.set_ylim(0, 1)
    ax.set_title(f'{level.title()}: annotation quality')
    ax.set_ylabel('Rate')
    ax.legend(frameon=False, fontsize=8)

fig.suptitle(
    'Annotation quality rates by coarse label (self-referential excluded)',
    y=0.99,
    fontsize=16,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(viz_output_dir / 'annotation_quality_top20_rates.png', dpi=180)
plt.show()


### `annotation_case_ab_top15.png`

**Output file:** `outputs/audit_visualizations/annotation_case_ab_top15.png`

**Source TSV:** `annotation_quality.tsv` (in `outputs/deprecated/{coverage,annotation,disparity}_audits/`)

**Pipeline stage:** Stage 01 of the deprecated `auditing/` pipeline (not
`audit_pipeline`'s Stage numbering).


In [ ]:
fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=5.8, panel_h=4.8)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    case_plot = (
        annotation_quality_viz[annotation_quality_viz['taxonomy_level'] == level]
        .sort_values('total_matches', ascending=False)
        .head(8)
        .sort_values('total_matches', ascending=True)
        .copy()
    )

    if case_plot.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{level.title()}: Case A/B composition')
        ax.set_axis_off()
        continue

    ax.barh(case_plot['target'], case_plot['case_a_present_hateful'], color='#009E73', label='Case A')
    ax.barh(
        case_plot['target'],
        case_plot['case_b_present_nonhateful'],
        left=case_plot['case_a_present_hateful'],
        color='#D55E00',
        label='Case B',
    )
    ax.set_title(f'{level.title()}: Case A/B composition')
    ax.set_xlabel('Matched count')
    ax.set_ylabel('Target')
    ax.legend(frameon=False, fontsize=8)

fig.suptitle(
    'Case A/B composition by coarse label (self-referential excluded)',
    y=0.99,
    fontsize=16,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(viz_output_dir / 'annotation_case_ab_top15.png', dpi=180)
plt.show()


### `disparity_di_ratio_histograms.png`

**Output file:** `outputs/audit_visualizations/disparity_di_ratio_histograms.png`

**Source TSV:** `coverage_disparity.tsv` (in `outputs/deprecated/{coverage,annotation,disparity}_audits/`)

**Pipeline stage:** Stage 02 of the deprecated `auditing/` pipeline (not
`audit_pipeline`'s Stage numbering).


In [ ]:
fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=5.8, panel_h=4.4)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    level_df = coverage_disparity_viz[coverage_disparity_viz['taxonomy_level'] == level]

    if level_df.empty:
        ax.text(0.5, 0.5, 'No pairwise rows', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{level.title()}: DI distribution')
        ax.set_axis_off()
        continue

    ax.hist(level_df['presence_rate_di_ratio'], bins=12, color='#4c78a8', alpha=0.65, label='presence DI')
    ax.hist(level_df['type_coverage_di_ratio'], bins=12, color='#72b7b2', alpha=0.65, label='type DI')
    ax.axvline(0.8, color='red', linestyle='--', linewidth=1.3)
    ax.set_title(f'{level.title()}: DI distribution')
    ax.set_xlabel('DI ratio')
    ax.set_ylabel('Pair count')
    ax.legend(frameon=False, fontsize=8)

fig.suptitle(
    'DI-ratio distributions by coarse label (self-referential excluded)',
    y=0.99,
    fontsize=16,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(viz_output_dir / 'disparity_di_ratio_histograms.png', dpi=180)
plt.show()


### `disparity_worst_di_top20.png`

**Output file:** `outputs/audit_visualizations/disparity_worst_di_top20.png`

**Source TSV:** `coverage_disparity.tsv` (in `outputs/deprecated/{coverage,annotation,disparity}_audits/`)

**Pipeline stage:** Stage 02 of the deprecated `auditing/` pipeline (not
`audit_pipeline`'s Stage numbering).


In [ ]:
fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=6.4, panel_h=5.0)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    worst_di = coverage_disparity_viz[coverage_disparity_viz['taxonomy_level'] == level].copy()

    if worst_di.empty:
        ax.text(0.5, 0.5, 'No pairwise rows', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{level.title()}: lowest DI comparisons')
        ax.set_axis_off()
        continue

    worst_di['pair'] = worst_di['target_a'] + ' vs ' + worst_di['target_b']
    worst_di['worst_di_ratio'] = worst_di[['presence_rate_di_ratio', 'type_coverage_di_ratio']].min(axis=1)
    worst_di = worst_di.sort_values('worst_di_ratio', ascending=True).head(8)

    ax.barh(worst_di['pair'], worst_di['worst_di_ratio'], color='#b279a2')
    ax.axvline(0.8, color='red', linestyle='--', linewidth=1.3)
    ax.set_title(f'{level.title()}: lowest DI comparisons')
    ax.set_xlabel('Worst DI ratio')
    ax.set_xlim(0, 1.05)

fig.suptitle(
    'Lowest DI-ratio comparisons by coarse label (self-referential excluded)',
    y=0.99,
    fontsize=16,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(viz_output_dir / 'disparity_worst_di_top20.png', dpi=180)
plt.show()


### `annotation_disparity_label_gap_top20.png`

**Output file:** `outputs/audit_visualizations/annotation_disparity_label_gap_top20.png`

**Source TSV:** `annotation_disparity.tsv` (in `outputs/deprecated/{coverage,annotation,disparity}_audits/`)

**Pipeline stage:** Stage 02 of the deprecated `auditing/` pipeline (not
`audit_pipeline`'s Stage numbering).


In [ ]:
fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=6.4, panel_h=5.0)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    ann_gap = annotation_disparity_viz[annotation_disparity_viz['taxonomy_level'] == level].copy()

    if ann_gap.empty:
        ax.text(0.5, 0.5, 'No pairwise rows', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{level.title()}: labeling-rate gaps')
        ax.set_axis_off()
        continue

    ann_gap['pair'] = ann_gap['target_a'] + ' vs ' + ann_gap['target_b']
    ann_gap = ann_gap.sort_values('labeling_rate_gap_abs', ascending=False).head(8).sort_values('labeling_rate_gap_abs')

    ax.barh(ann_gap['pair'], ann_gap['labeling_rate_gap_abs'], color='#e45756')
    ax.set_title(f'{level.title()}: labeling-rate gaps')
    ax.set_xlabel('Absolute correct-labeling-rate gap')

fig.suptitle(
    'Annotation quality gaps by coarse label (self-referential excluded)',
    y=0.99,
    fontsize=16,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(viz_output_dir / 'annotation_disparity_label_gap_top20.png', dpi=180)
plt.show()


### `cross_level_deltas.png`

**Output file:** `outputs/audit_visualizations/cross_level_deltas.png`

**Source TSV:** `cross_level_consistency.tsv` (in `outputs/deprecated/{coverage,annotation,disparity}_audits/`)

**Pipeline stage:** Stage 02 of the deprecated `auditing/` pipeline (not
`audit_pipeline`'s Stage numbering).


In [ ]:
if cross_level_consistency_viz.empty:
    print('cross_level_consistency is empty for this run after self-referential exclusion.')
else:
    fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=6.8, panel_h=4.8)

    for idx, level in enumerate(coarse_levels):
        ax = axes[idx]
        level_df = cross_level_consistency_viz[cross_level_consistency_viz['level_from'] == level].copy()

        if level_df.empty:
            ax.text(0.5, 0.5, 'No transitions from this level', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{level.title()}: cross-level deltas')
            ax.set_axis_off()
            continue

        labels = level_df['target'] + ' -> ' + level_df['level_to']
        y = np.arange(len(level_df))

        ax.barh(y - 0.2, level_df['presence_rate_delta'], height=0.38, color='#4c78a8', label='presence delta')
        ax.barh(y + 0.2, level_df['correct_labeling_rate_delta'], height=0.38, color='#54a24b', label='labeling delta')
        ax.set_yticks(y)
        ax.set_yticklabels(labels)
        ax.axvline(0, color='black', linewidth=1)
        ax.set_title(f'{level.title()}: cross-level deltas')
        ax.set_xlabel('Delta (to - from)')
        ax.legend(frameon=False, fontsize=8)

    fig.suptitle(
        'Cross-level deltas faceted by source coarse label (self-referential excluded)',
        y=0.99,
        fontsize=16,
        fontweight='semibold',
    )
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(viz_output_dir / 'cross_level_deltas.png', dpi=180)
    plt.show()


### Diagnostic printout (not a figure) -- high-confidence misclassifications

**Reads:** `form_labeling_detail.tsv` (Stage 01) and `audit_matches.tsv`
(Stage 00) of the deprecated `auditing/` pipeline. Produces no image file;
reproduced here because it is part of the original notebook's real content
(guardrail: reproduce everything discovered in Step 1, not just the parts
that happen to be plots).


In [ ]:
print('=== High-confidence misclassifications (self-referential excluded) ===')

# Define high-confidence misclassification at form level.
# - Misclassification-prone: Case B exceeds Case A.
# - High-confidence: enough volume and low accuracy.
misclassified_forms = form_labeling_viz.copy()
misclassified_forms['total_matches'] = misclassified_forms['case_a_count'] + misclassified_forms['case_b_count']
misclassified_forms = misclassified_forms[
    (misclassified_forms['case_b_count'] > misclassified_forms['case_a_count'])
    & (misclassified_forms['total_matches'] >= 5)
    & (misclassified_forms['labeling_accuracy'] <= 0.20)
].copy()

misclassified_forms = misclassified_forms.sort_values(
    ['taxonomy_level', 'target', 'dogwhistle', 'labeling_accuracy', 'case_b_count'],
    ascending=[True, True, True, True, False],
)

if misclassified_forms.empty:
    print('No high-confidence misclassifications found under current filters.')
else:
    print(f"High-confidence misclassified forms: {len(misclassified_forms)}")
    print(
        misclassified_forms[
            [
                'taxonomy_level',
                'target',
                'dogwhistle',
                'type',
                'case_a_count',
                'case_b_count',
                'total_matches',
                'labeling_accuracy',
            ]
        ].to_string(index=False)
    )

    print('\n=== All matched non-hateful examples for these forms ===')
    all_examples = audit_matches_viz.merge(
        misclassified_forms[['taxonomy_level', 'target', 'dogwhistle']],
        on=['taxonomy_level', 'target', 'dogwhistle'],
        how='inner',
    )

    all_examples = all_examples[all_examples['binary_hate'] == 0].copy()
    all_examples = all_examples.drop_duplicates(
        ['taxonomy_level', 'target', 'dogwhistle', 'text_dedup_key']
    )
    all_examples = all_examples.sort_values(
        ['taxonomy_level', 'target', 'dogwhistle', 'text_dedup_key']
    )

    print(f"Total non-hateful matched examples: {len(all_examples)}")
    for _, row in all_examples.iterrows():
        print(f"- [{row['taxonomy_level']} | {row['target']} | {row['dogwhistle']}] {str(row['text'])}")


In [ ]:
print(f'Visualization images written to: {viz_output_dir}')


# Step 3 -- Cross-stage numerical consistency checks

Every quantity below is re-derived independently at **two or more**
pipeline stages/files. Each check loads the raw TSVs from every stage where
the quantity appears, joins on the shared keys, and asserts agreement within
`atol=1e-6` (`np.isclose(..., equal_nan=True)`, so two NaNs -- e.g. both
sides correctly suppressing an unstable small-n comparison -- count as
agreement, not a mismatch). Every mismatch is collected into one combined
table at the end rather than raising, per the task spec. The only numeric
literal used anywhere below is the `1e-6` comparison tolerance; every
"expected" value is read from a TSV or derived via `audit_pipeline.helpers`
functions imported from the real pipeline code (`map_reporting_group`,
`norm_target`) -- nothing is retyped from memory.

**What's being compared and why:**
- Stage 1 (`s1_coverage_by_level_target.tsv`, per raw target) vs Stage 4b
  (`s4b_coverage_by_level_group.tsv`, per reporting group) -- same
  `presence_rate` / `type_coverage`, computed by two independent code paths
  (Stage 1 aggregates over `s1_matches.tsv` directly in `stage1_coverage.py`;
  Stage 4 re-aggregates from the same match file via `compute_rollup()` in
  `stage4_rollup.py`). Restricted to reporting groups that are a 1:1 relabel
  of a raw target (`report_target == norm_target(target)`) -- collapsed
  groups like `LGB` / `Trans/NB` are legitimately many-to-one and are logged
  separately, not compared row-for-row.
- Stage 2 vs Stage 4b: same logic for `correct_labeling_rate` /
  `failure_rate` / `total_matches`.
- Stage 4a (`by_group`) vs Stage 4b (`by_level_group`): these two files
  differ only in whether `taxonomy_level` is kept as a group-by key. Since
  `map_reporting_group()` makes `report_level` a deterministic function of
  `taxonomy_level`, `taxonomy_level` is redundant once `report_level` /
  `report_target` are fixed -- so Stage 4a and Stage 4b should be exactly
  the same numbers, just with/without an extra (redundant) column. This is
  the concrete version of the "which by_* file is authoritative" ambiguity
  flagged throughout Step 2.
- Stage 3 pairwise DI (`target_a`/`target_b` = raw targets) vs Stage 4b and
  Stage 4a pairwise DI (`target_a`/`target_b` = reporting groups): three
  independently-built pairwise tables that different figures elsewhere in
  this notebook each treat as *the* source for "worst pairwise DI." Pairs
  are canonicalized (`tuple(sorted([a, b]))`) before joining since
  `itertools.combinations` order is not guaranteed to agree between the two
  independent pairwise-builders; only symmetric metrics (DI ratios and
  `*_gap_abs`, not signed `*_gap`) are compared for this reason.
- `generate_fig2_annotation_di_pairwise.py`'s own Stage 4a/Stage 4b merge:
  that script already detects overlapping `(target_a, target_b, coding_level)`
  keys between the two files and silently prefers Stage 4b -- but it never
  checks whether the two actually *agree*. This notebook adds that check.
- The deprecated `auditing/` pipeline vs `audit_pipeline`: logged as
  **not directly comparable** (see structural finding above) rather than
  joined, since the deprecated pipeline pools across coding levels.


In [ ]:
mismatches = []      # list[dict] -- rows for the final combined mismatch table
not_comparable = []  # list[str]  -- quantities with only one source, or structurally incompatible sources
checked = {}         # quantity label -> number of (group/pair, coding_level) rows compared

def _record_mismatches(quantity, df, key_cols, a_col, b_col, stage_a, stage_b):
    """Compare two already-joined columns; append any disagreement to `mismatches`."""
    n = len(df)
    checked[quantity] = checked.get(quantity, 0) + n
    if n == 0:
        return 0
    a = pd.to_numeric(df[a_col], errors="coerce")
    b = pd.to_numeric(df[b_col], errors="coerce")
    close = np.isclose(a.astype(float), b.astype(float), atol=1e-6, equal_nan=True)
    bad = df.loc[~close]
    a_bad = a.loc[~close]
    b_bad = b.loc[~close]
    for idx in bad.index:
        key = " / ".join(str(bad.loc[idx, c]) for c in key_cols)
        va, vb = a_bad.loc[idx], b_bad.loc[idx]
        delta = abs(va - vb) if pd.notna(va) and pd.notna(vb) else np.nan
        mismatches.append({
            "quantity": quantity,
            "group_or_pair": key,
            "coding_level": bad.loc[idx, "coding_level"] if "coding_level" in bad.columns else "",
            "stage_a": stage_a, "value_a": va,
            "stage_b": stage_b, "value_b": vb,
            "delta": delta,
        })
    return int((~close).sum())


## 3a -- Stage 1 vs Stage 4b (coverage: `presence_rate`, `type_coverage`)


In [ ]:
s1_cov = pd.read_csv(variant.out_s1 / "s1_coverage_by_level_target.tsv", sep="\t")
s4b_cov = pd.read_csv(variant.out_s4 / "by_level_group" / "s4b_coverage_by_level_group.tsv", sep="\t")

_mapped = s1_cov.apply(lambda r: map_reporting_group(r["taxonomy_level"], r["target"]), axis=1)
s1_cov = s1_cov.assign(
    report_level=[m.report_level for m in _mapped],
    report_target=[m.report_target for m in _mapped],
    report_include=[m.include for m in _mapped],
)
s1_cov_incl = s1_cov[s1_cov["report_include"]].copy()
identity_mask = (
    (s1_cov_incl["report_target"] == s1_cov_incl["target"].map(norm_target))
    & (s1_cov_incl["report_level"] == s1_cov_incl["taxonomy_level"].map(norm_target))
)
s1_identity = s1_cov_incl[identity_mask]
n_collapsed_s1 = int((~identity_mask).sum())

joined_a = s1_identity.merge(
    s4b_cov,
    on=["taxonomy_level", "coding_level", "report_level", "report_target"],
    suffixes=("_s1", "_s4b"),
    how="inner",
)
print(f"Stage1 vs Stage4b coverage: {len(joined_a)} matched rows joined "
      f"({n_collapsed_s1} Stage-1 rows excluded: many-to-one reporting-group "
      f"collapse, e.g. LGB/Trans-NB -- legitimate aggregation difference, not a discrepancy).")

for metric in ["presence_rate", "type_coverage"]:
    n_bad = _record_mismatches(
        f"{metric} (Stage1 vs Stage4b)", joined_a,
        ["report_level", "report_target"],
        f"{metric}_s1", f"{metric}_s4b", "Stage 1", "Stage 4b",
    )
    print(f"  {metric}: {n_bad} mismatch(es)")


## 3b -- Stage 2 vs Stage 4b (annotation: `correct_labeling_rate`, `failure_rate`, `total_matches`)


In [ ]:
s2_ann = pd.read_csv(variant.out_s2 / "s2_annotation_by_level_target.tsv", sep="\t")
s4b_ann = pd.read_csv(variant.out_s4 / "by_level_group" / "s4b_annotation_by_level_group.tsv", sep="\t")

_mapped2 = s2_ann.apply(lambda r: map_reporting_group(r["taxonomy_level"], r["target"]), axis=1)
s2_ann = s2_ann.assign(
    report_level=[m.report_level for m in _mapped2],
    report_target=[m.report_target for m in _mapped2],
    report_include=[m.include for m in _mapped2],
)
s2_incl = s2_ann[s2_ann["report_include"]].copy()
identity_mask2 = (
    (s2_incl["report_target"] == s2_incl["target"].map(norm_target))
    & (s2_incl["report_level"] == s2_incl["taxonomy_level"].map(norm_target))
)
s2_identity = s2_incl[identity_mask2]
n_collapsed_s2 = int((~identity_mask2).sum())

joined_b = s2_identity.merge(
    s4b_ann,
    on=["taxonomy_level", "coding_level", "report_level", "report_target"],
    suffixes=("_s2", "_s4b"),
    how="inner",
)
print(f"Stage2 vs Stage4b annotation: {len(joined_b)} matched rows joined "
      f"({n_collapsed_s2} Stage-2 rows excluded: many-to-one collapse).")

for metric in ["correct_labeling_rate", "failure_rate", "total_matches"]:
    n_bad = _record_mismatches(
        f"{metric} (Stage2 vs Stage4b)", joined_b,
        ["report_level", "report_target"],
        f"{metric}_s2", f"{metric}_s4b", "Stage 2", "Stage 4b",
    )
    print(f"  {metric}: {n_bad} mismatch(es)")


## 3c -- Stage 4a (`by_group`) vs Stage 4b (`by_level_group`)

Tests the hypothesis stated above: since `taxonomy_level` is a deterministic
function of `report_level` under `map_reporting_group()`, dropping it from
the group-by keys (as `by_group`/4a does) should produce numerically
identical rows to keeping it (as `by_level_group`/4b does).


In [ ]:
s4a_cov = pd.read_csv(variant.out_s4 / "by_group" / "s4a_coverage_by_group.tsv", sep="\t")
joined_c1 = s4a_cov.merge(
    s4b_cov, on=["coding_level", "report_level", "report_target"],
    suffixes=("_s4a", "_s4b"), how="outer", indicator=True,
)
n_onesided = int((joined_c1["_merge"] != "both").sum())
print(f"Stage4a vs Stage4b coverage: {len(joined_c1)} rows after outer join "
      f"({n_onesided} present on only one side -- 0 expected if the redundant-key hypothesis holds).")
joined_c1_both = joined_c1[joined_c1["_merge"] == "both"]
for metric in ["presence_rate", "type_coverage"]:
    n_bad = _record_mismatches(
        f"{metric} (Stage4a vs Stage4b)", joined_c1_both,
        ["report_level", "report_target"],
        f"{metric}_s4a", f"{metric}_s4b", "Stage 4a", "Stage 4b",
    )
    print(f"  {metric}: {n_bad} mismatch(es)")

s4a_ann = pd.read_csv(variant.out_s4 / "by_group" / "s4a_annotation_by_group.tsv", sep="\t")
joined_c2 = s4a_ann.merge(
    s4b_ann, on=["coding_level", "report_level", "report_target"],
    suffixes=("_s4a", "_s4b"), how="outer", indicator=True,
)
n_onesided2 = int((joined_c2["_merge"] != "both").sum())
print(f"Stage4a vs Stage4b annotation: {len(joined_c2)} rows after outer join "
      f"({n_onesided2} present on only one side).")
joined_c2_both = joined_c2[joined_c2["_merge"] == "both"]
for metric in ["correct_labeling_rate", "failure_rate", "total_matches"]:
    n_bad = _record_mismatches(
        f"{metric} (Stage4a vs Stage4b)", joined_c2_both,
        ["report_level", "report_target"],
        f"{metric}_s4a", f"{metric}_s4b", "Stage 4a", "Stage 4b",
    )
    print(f"  {metric}: {n_bad} mismatch(es)")


## 3d/3e -- Stage 3 pairwise DI vs Stage 4b and Stage 4a pairwise DI

Three independently-built pairwise tables (`s3_coverage_disparity.tsv` /
`s3_annotation_disparity.tsv` at raw-target granularity; the two Stage-4
`*_pairwise_disparity_*` files at reporting-group granularity). Only
orientation-symmetric metrics are compared (DI ratios and `*_gap_abs`
columns) since `target_a`/`target_b` order is not guaranteed to match across
independently-run `itertools.combinations` builds.


In [ ]:
s3_cov_disp = pd.read_csv(variant.out_s3 / "s3_coverage_disparity.tsv", sep="\t")
s3_ann_disp = pd.read_csv(variant.out_s3 / "s3_annotation_disparity.tsv", sep="\t")
s4b_pair = pd.read_csv(variant.out_s4 / "by_level_group" / "s4b_pairwise_disparity_by_level_group.tsv", sep="\t")
s4a_pair = pd.read_csv(variant.out_s4 / "by_group" / "s4a_pairwise_disparity_by_group.tsv", sep="\t")

def _map_pair_to_report(df, a_col="target_a", b_col="target_b", level_col="taxonomy_level"):
    df = df.copy()
    ma = df.apply(lambda r: map_reporting_group(r[level_col], r[a_col]), axis=1)
    mb = df.apply(lambda r: map_reporting_group(r[level_col], r[b_col]), axis=1)
    df["report_level"] = [m.report_level for m in ma]
    df["report_target_a"] = [m.report_target for m in ma]
    df["report_target_b"] = [m.report_target for m in mb]
    df["report_include"] = [m.include and n.include for m, n in zip(ma, mb)]
    df["identity_a"] = df["report_target_a"] == df[a_col].map(norm_target)
    df["identity_b"] = df["report_target_b"] == df[b_col].map(norm_target)
    df["pair_key"] = [tuple(sorted([a, b])) for a, b in zip(df["report_target_a"], df["report_target_b"])]
    return df

s3_cov_mapped = _map_pair_to_report(s3_cov_disp)
s3_ann_mapped = _map_pair_to_report(s3_ann_disp)

s3_cov_id = s3_cov_mapped[s3_cov_mapped["report_include"] & s3_cov_mapped["identity_a"] & s3_cov_mapped["identity_b"]]
s3_ann_id = s3_ann_mapped[s3_ann_mapped["report_include"] & s3_ann_mapped["identity_a"] & s3_ann_mapped["identity_b"]]
print(f"Stage3 coverage-disparity: {len(s3_cov_id)}/{len(s3_cov_mapped)} pairs kept for exact-match "
      f"checks (rest touch a collapsed reporting group on at least one side).")
print(f"Stage3 annotation-disparity: {len(s3_ann_id)}/{len(s3_ann_mapped)} pairs kept.")

s4b_pair2 = s4b_pair.copy()
s4b_pair2["pair_key"] = [tuple(sorted([a, b])) for a, b in zip(s4b_pair2["target_a"], s4b_pair2["target_b"])]
s4a_pair2 = s4a_pair.copy()
s4a_pair2["pair_key"] = [tuple(sorted([a, b])) for a, b in zip(s4a_pair2["target_a"], s4a_pair2["target_b"])]

# --- Stage 3 vs Stage 4b ---
joined_d_cov = s3_cov_id.merge(
    s4b_pair2, on=["taxonomy_level", "coding_level", "pair_key"],
    suffixes=("_s3", "_s4b"), how="inner",
)
print(f"\nStage3 coverage-disparity vs Stage4b pairwise: {len(joined_d_cov)} matched pairs.")
for metric in ["presence_rate_di_ratio", "type_coverage_di_ratio", "worst_di_ratio"]:
    n_bad = _record_mismatches(
        f"{metric} (Stage3 vs Stage4b)", joined_d_cov,
        ["taxonomy_level", "coding_level", "pair_key"],
        f"{metric}_s3", f"{metric}_s4b", "Stage 3", "Stage 4b",
    )
    print(f"  {metric}: {n_bad} mismatch(es)")

joined_d_ann = s3_ann_id.merge(
    s4b_pair2, on=["taxonomy_level", "coding_level", "pair_key"],
    suffixes=("_s3", "_s4b"), how="inner",
)
print(f"Stage3 annotation-disparity vs Stage4b pairwise: {len(joined_d_ann)} matched pairs.")
for metric in ["annotation_di_ratio", "labeling_rate_gap_abs", "failure_rate_gap_abs"]:
    n_bad = _record_mismatches(
        f"{metric} (Stage3 vs Stage4b)", joined_d_ann,
        ["taxonomy_level", "coding_level", "pair_key"],
        f"{metric}_s3", f"{metric}_s4b", "Stage 3", "Stage 4b",
    )
    print(f"  {metric}: {n_bad} mismatch(es)")

# --- Stage 3 vs Stage 4a (no taxonomy_level column on the 4a side; join on report_level instead) ---
joined_e_cov = s3_cov_id.merge(
    s4a_pair2, on=["coding_level", "report_level", "pair_key"],
    suffixes=("_s3", "_s4a"), how="inner",
)
print(f"\nStage3 coverage-disparity vs Stage4a pairwise: {len(joined_e_cov)} matched pairs.")
for metric in ["presence_rate_di_ratio", "type_coverage_di_ratio", "worst_di_ratio"]:
    n_bad = _record_mismatches(
        f"{metric} (Stage3 vs Stage4a)", joined_e_cov,
        ["coding_level", "report_level", "pair_key"],
        f"{metric}_s3", f"{metric}_s4a", "Stage 3", "Stage 4a",
    )
    print(f"  {metric}: {n_bad} mismatch(es)")

joined_e_ann = s3_ann_id.merge(
    s4a_pair2, on=["coding_level", "report_level", "pair_key"],
    suffixes=("_s3", "_s4a"), how="inner",
)
print(f"Stage3 annotation-disparity vs Stage4a pairwise: {len(joined_e_ann)} matched pairs.")
for metric in ["annotation_di_ratio", "labeling_rate_gap_abs", "failure_rate_gap_abs"]:
    n_bad = _record_mismatches(
        f"{metric} (Stage3 vs Stage4a)", joined_e_ann,
        ["coding_level", "report_level", "pair_key"],
        f"{metric}_s3", f"{metric}_s4a", "Stage 3", "Stage 4a",
    )
    print(f"  {metric}: {n_bad} mismatch(es)")


## 3f -- `generate_fig2_annotation_di_pairwise.py`'s own Stage 4a/4b overlap

That script's `load_and_merge()` already finds `(target_a, target_b,
coding_level)` keys present as stable rows in *both* `s4b` and `s4a`, and
silently keeps the `s4b` copy while dropping `s4a`'s -- without ever
checking whether the two actually agree. Reusing its own `KEY_COLS` and
`_filter_stable` to check that now.


In [ ]:
from generate_fig2_annotation_di_pairwise import KEY_COLS as FIG2_KEY_COLS, _filter_stable as fig2_filter_stable

s4b_stable = fig2_filter_stable(s4b_pair.copy())
s4a_stable = fig2_filter_stable(s4a_pair.copy())
overlap_keys = set(map(tuple, s4b_stable[FIG2_KEY_COLS].values)) & set(map(tuple, s4a_stable[FIG2_KEY_COLS].values))
print(f"{len(overlap_keys)} (target_a, target_b, coding_level) key(s) are stable in both Stage 4a and Stage 4b.")

if overlap_keys:
    s4b_ov = s4b_stable[s4b_stable[FIG2_KEY_COLS].apply(tuple, axis=1).isin(overlap_keys)]
    s4a_ov = s4a_stable[s4a_stable[FIG2_KEY_COLS].apply(tuple, axis=1).isin(overlap_keys)]
    joined_f = s4a_ov.merge(s4b_ov, on=FIG2_KEY_COLS, suffixes=("_s4a", "_s4b"))
    n_bad = _record_mismatches(
        "annotation_di_ratio (Stage4a/Stage4b overlap keys used by generate_fig2_annotation_di_pairwise.py)",
        joined_f, list(FIG2_KEY_COLS),
        "annotation_di_ratio_s4a", "annotation_di_ratio_s4b", "Stage 4a", "Stage 4b",
    )
    print(f"  annotation_di_ratio: {n_bad} mismatch(es) among overlap keys")
else:
    print("  no overlapping stable keys found for this pipeline run.")


## 3g -- Deprecated `auditing/` pipeline vs `audit_pipeline`

No join is attempted here (see structural finding in the `auditing/`
section above): the deprecated pipeline's coverage/annotation/disparity
files have no `coding_level` column, so every metric is pooled across L1-L4
for a `(taxonomy_level, target)` pair -- not a like-for-like re-derivation
of any single `audit_pipeline` stage's per-coding-level metric. Logged as
not-comparable rather than fabricating a join that would either fan out or
silently misrepresent a pooled number as a per-level one.


In [ ]:
not_comparable.append(
    "presence_rate / type_coverage / correct_labeling_rate / failure_rate / "
    "presence_rate_di_ratio / annotation_di_ratio (deprecated auditing/ Stage 00-02 "
    "vs audit_pipeline Stage 1-4): the deprecated pipeline's coverage_disparity.tsv, "
    "annotation_quality.tsv, and audit_metrics.tsv have NO coding_level column -- "
    "every metric is pooled across all Mendelsohn dogwhistle types (L1-L4) for a "
    "(taxonomy_level, target) pair, while every audit_pipeline stage stratifies by "
    "coding_level. A row-level join would fan out (1 deprecated row vs up to 4 "
    "audit_pipeline rows) or require independently reproducing the deprecated "
    "pipeline's pooling logic, which is out of scope for this pass. Logged as "
    "single-source / not directly comparable rather than fabricating a join."
)
for note in not_comparable:
    print("- " + note)


## Step 3 result -- combined cross-stage mismatch table

Empty is the success condition.


In [ ]:
mismatch_df = pd.DataFrame(
    mismatches,
    columns=["quantity", "group_or_pair", "coding_level", "stage_a", "value_a", "stage_b", "value_b", "delta"],
)
if not mismatch_df.empty:
    mismatch_df = mismatch_df.sort_values("delta", ascending=False, na_position="last").reset_index(drop=True)

total_checked = sum(checked.values())
print("=" * 88)
print("CROSS-STAGE CONSISTENCY SUMMARY")
print("=" * 88)
print(f"Quantity x group/pair x coding_level combinations checked: {total_checked} "
      f"across {len(checked)} quantity comparisons.")
for q, n in checked.items():
    print(f"  - {q}: {n} rows compared")
print(f"\nQuantities logged as single-source / not directly comparable: {len(not_comparable)}")
print(f"\nMismatches found (|delta| > 1e-6): {len(mismatch_df)}")
if mismatch_df.empty:
    print("EMPTY -- every cross-stage-computed quantity checked above agrees within 1e-6.")
else:
    print("See table below, sorted by delta descending.")
    with pd.option_context("display.max_rows", None, "display.width", 160):
        print(mismatch_df.to_string(index=False))

mismatch_df


# Step 4 -- Final summary

## Step 1 inventory -- every figure-generation file found in the repo

| File | Figures produced | Input files read (exact paths) |
|---|---|---|
| `audit_pipeline/stage5_figures.py` | 16 PNGs across `level_stratified_figures` (8), `group_collapsed_figures` (5), `elsherief_figures` (3) | `stage1/s1_coverage_by_level_target.tsv`, `stage2/s2_annotation_by_level_target.tsv`, `stage3/s3_coverage_disparity.tsv`, `stage3/s3_annotation_disparity.tsv`, `stage3/s3_cross_level_consistency.tsv`, `stage4/by_level_group/s4b_coverage_by_level_group.tsv`, `stage4/by_group/s4a_annotation_by_group.tsv`, `stage4/by_group/s4a_pairwise_disparity_by_group.tsv`, `stage4/elsherief/s4c_coverage_delta_union_vs_elsherief.tsv`, `stage4/elsherief/s4c_annotation_delta_union_vs_elsherief.tsv`, `stage4/elsherief/s4c_pairwise_delta_union_vs_elsherief.tsv` |
| `generate_figures_final.py` (repo root) | 13 PDFs: `fig1`-`fig7`, `appA`-`appG` | `stage1/s1_coverage_by_level_target.tsv`, `unioned_data/06_glossary_label_reference.tsv`, `stage2/s2_annotation_by_level_target.tsv`, `stage3/s3_cross_level_consistency.tsv`, `stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`, `stage4/by_level_group/s4b_coverage_by_level_group.tsv`, `stage4/by_level_group/s4b_annotation_by_level_group.tsv`, `stage4/elsherief/s4c_annotation_delta_union_vs_elsherief.tsv`, `stage4/elsherief/s4c_coverage_delta_union_vs_elsherief.tsv`, `stage4/elsherief/s4c_pairwise_delta_union_vs_elsherief.tsv` |
| `generate_fig2_annotation_di_pairwise.py` (repo root) | 1 PDF: `fig2_annotation_di_by_pair_level.pdf` (or `_v2` if it already exists) | `stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`, `stage4/by_group/s4a_pairwise_disparity_by_group.tsv` |
| `auditing/03_audit_visualizations.ipynb` (deprecated pipeline) | 8 PNGs + 1 diagnostic printout | `outputs/deprecated/coverage_audits/{audit_metrics,audit_detailed,audit_matches}.tsv`, `outputs/deprecated/annotation_audits/{annotation_quality,form_labeling_detail}.tsv`, `outputs/deprecated/disparity_audits/{coverage_disparity,annotation_disparity,cross_level_consistency}.tsv` (originally pointed at `outputs/{coverage,annotation,disparity}_audits/`, which no longer exist at that path) |

**Total: 4 files, 38 figures + 1 diagnostic printout, all reproduced in this notebook.**

## Step 2 provenance -- figure to source file to pipeline stage

See the markdown cell immediately preceding each figure's code cell above
for the full per-figure provenance block (output filename, exact source
TSV path(s), pipeline stage). Section-level summary:

| Section | Pipeline stage(s) used |
|---|---|
| Level-stratified figures (`stage5_figures.py`) | Stage 1, Stage 2, Stage 3 |
| Group-collapsed figures (`stage5_figures.py`) | Stage 4b (coverage only), Stage 4a (annotation + pairwise) |
| ElSherief delta figures (`stage5_figures.py`) | Stage 4c |
| `generate_figures_final.py` (`fig1`-`fig7`, `appA`-`appG`) | Stage 1, Stage 2, Stage 3, Stage 4b, Stage 4c, and one upstream preprocessing file |
| `generate_fig2_annotation_di_pairwise.py` | Stage 4a + Stage 4b, merged |
| `auditing/03_audit_visualizations.ipynb` | Deprecated pipeline's own Stage 00-02 (not `audit_pipeline` numbering) |

**Key ambiguity surfaced (the reason this task exists):** a "worst pairwise
DI" figure exists reading **Stage 3** (`stage5_figures.level_stratified_figures`
-> `s5_lev_disparity_worst_di.png`), a second reading **Stage 4a**
(`stage5_figures.group_collapsed_figures` -> `s5_grp_worst_di_and_label_gap.png`),
and a third reading **Stage 4b** (`generate_figures_final.fig5_worst_di_by_pair_level`
/ `appD_worst_di_and_label_gap_pooled`, and `generate_fig2_annotation_di_pairwise.py`,
which merges 4a+4b itself). None of the four figure-generation files agree
on a single canonical pairwise-DI source, and the Stage 4b file
(`by_level_group`) -- the one a human manually spot-checking numbers would
most likely reach for first, since it's the only one whose name mentions
both "level" and "group" -- is used by exactly one of the four call sites
that produce a "worst DI" figure. Step 3 above numerically confirms whether
these different sources actually agree.

## Step 3 result -- cross-stage consistency

See the "Step 3 result" cell above for the live, executed numbers (row
counts checked, mismatch count, full mismatch table if non-empty). That
cell is authoritative; this paragraph is a static description of what was
checked: Stage 1 vs Stage 4b coverage, Stage 2 vs Stage 4b annotation,
Stage 4a vs Stage 4b (coverage and annotation), Stage 3 vs Stage 4b pairwise
DI, Stage 3 vs Stage 4a pairwise DI, and `generate_fig2_annotation_di_pairwise.py`'s
own Stage 4a/4b overlap keys -- seven independent cross-checks in total. The
deprecated `auditing/` pipeline was found to be structurally incompatible
with `audit_pipeline` (no `coding_level` stratification) and is logged as
not-comparable rather than joined.

## Known bugs fixed

Both occurrences of the `_top_n_s5` sort-direction bug described in the task
brief have been fixed. `_top_n_s5` now takes an explicit `ascending=`
argument (default `False`, unchanged for every other call site -- token
frequency, label gaps, etc., where higher = worse); both
`"worst_di_ratio"` call sites now pass `ascending=True` so the n MOST
disparate pairs are selected, not the least:

1. **Cell 6** (Level-stratified figures / `level_stratified_figures`),
   `d = _top_n_s5(sub, "worst_di_ratio", 8, ascending=True).sort_values(`
   -- affects `s5_lev_disparity_worst_di.png`.
2. **Cell 9** (Group-collapsed figures / `group_collapsed_figures`),
   `left = _top_n_s5(pair_plot, "worst_di_ratio", 20, ascending=True).sort_values(`
   -- affects `s5_grp_worst_di_and_label_gap.png`.

(Cell indices are 0-based over the full notebook's cell list, counting both
markdown and code cells, as saved in this build.)

## Other surfaced (not fixed) discrepancies

- `generate_fig2_annotation_di_pairwise.py`'s docstring says it matches
  "Figure 9 (fig5_worst_di_by_pair_level.pdf)" but its own output file is
  named `fig2_annotation_di_by_pair_level.pdf` -- a `fig2` name that also
  collides (in name only, not content) with `generate_figures_final.py`'s
  unrelated `annotation_rates_by_level.pdf`.
- `generate_figures_final.py` and `generate_fig2_annotation_di_pairwise.py`
  are not `PipelineVariant`-aware (no tier1+2 robustness-check support),
  unlike `stage5_figures.py`.
- `generate_fig2_annotation_di_pairwise.py` hardcodes a local
  `DI_THRESHOLD = 0.80` instead of importing `audit_pipeline.config.DI_THRESHOLD`
  (numerically identical, but a third independent definition of the same
  constant).
- The deprecated `auditing/` pipeline pools all metrics across coding levels
  (no `coding_level` column anywhere in its output), unlike every
  `audit_pipeline` stage.

## Success checklist

- [x] Notebook runs top-to-bottom with a fresh kernel, no errors (verified
      via a redirected-output copy -- see build notes; this delivered copy
      targets the real production paths and was not itself executed, so
      that no existing figure artifact is modified as a side effect of
      building this notebook).
- [x] Every figure produced by any discovered script is reproduced here
      under the same output filename.
- [x] Every figure cell is preceded by a provenance markdown block.
- [x] The cross-stage consistency section runs and produces a summary table
      for every quantity with a second independent source (7 checks; see
      Step 3 above for the live mismatch count).
- [x] Zero hardcoded "expected" values -- all Step 3 comparisons read from
      TSVs or `audit_pipeline` code; the only numeric literal introduced is
      the `1e-6` comparison tolerance.
- [x] Both `_top_n_s5` "worst DI" bug sites fixed above (see "Known bugs
      fixed" section) with cell/line pointers.
- [x] No existing file modified -- this notebook is the only new file this
      task produced.


In [ ]:
from IPython.display import Markdown, display

summary_text = '# Step 4 -- Final summary\n\n## Step 1 inventory -- every figure-generation file found in the repo\n\n| File | Figures produced | Input files read (exact paths) |\n|---|---|---|\n| `audit_pipeline/stage5_figures.py` | 16 PNGs across `level_stratified_figures` (8), `group_collapsed_figures` (5), `elsherief_figures` (3) | `stage1/s1_coverage_by_level_target.tsv`, `stage2/s2_annotation_by_level_target.tsv`, `stage3/s3_coverage_disparity.tsv`, `stage3/s3_annotation_disparity.tsv`, `stage3/s3_cross_level_consistency.tsv`, `stage4/by_level_group/s4b_coverage_by_level_group.tsv`, `stage4/by_group/s4a_annotation_by_group.tsv`, `stage4/by_group/s4a_pairwise_disparity_by_group.tsv`, `stage4/elsherief/s4c_coverage_delta_union_vs_elsherief.tsv`, `stage4/elsherief/s4c_annotation_delta_union_vs_elsherief.tsv`, `stage4/elsherief/s4c_pairwise_delta_union_vs_elsherief.tsv` |\n| `generate_figures_final.py` (repo root) | 13 PDFs: `fig1`-`fig7`, `appA`-`appG` | `stage1/s1_coverage_by_level_target.tsv`, `unioned_data/06_glossary_label_reference.tsv`, `stage2/s2_annotation_by_level_target.tsv`, `stage3/s3_cross_level_consistency.tsv`, `stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`, `stage4/by_level_group/s4b_coverage_by_level_group.tsv`, `stage4/by_level_group/s4b_annotation_by_level_group.tsv`, `stage4/elsherief/s4c_annotation_delta_union_vs_elsherief.tsv`, `stage4/elsherief/s4c_coverage_delta_union_vs_elsherief.tsv`, `stage4/elsherief/s4c_pairwise_delta_union_vs_elsherief.tsv` |\n| `generate_fig2_annotation_di_pairwise.py` (repo root) | 1 PDF: `fig2_annotation_di_by_pair_level.pdf` (or `_v2` if it already exists) | `stage4/by_level_group/s4b_pairwise_disparity_by_level_group.tsv`, `stage4/by_group/s4a_pairwise_disparity_by_group.tsv` |\n| `auditing/03_audit_visualizations.ipynb` (deprecated pipeline) | 8 PNGs + 1 diagnostic printout | `outputs/deprecated/coverage_audits/{audit_metrics,audit_detailed,audit_matches}.tsv`, `outputs/deprecated/annotation_audits/{annotation_quality,form_labeling_detail}.tsv`, `outputs/deprecated/disparity_audits/{coverage_disparity,annotation_disparity,cross_level_consistency}.tsv` (originally pointed at `outputs/{coverage,annotation,disparity}_audits/`, which no longer exist at that path) |\n\n**Total: 4 files, 38 figures + 1 diagnostic printout, all reproduced in this notebook.**\n\n## Step 2 provenance -- figure to source file to pipeline stage\n\nSee the markdown cell immediately preceding each figure\'s code cell above\nfor the full per-figure provenance block (output filename, exact source\nTSV path(s), pipeline stage). Section-level summary:\n\n| Section | Pipeline stage(s) used |\n|---|---|\n| Level-stratified figures (`stage5_figures.py`) | Stage 1, Stage 2, Stage 3 |\n| Group-collapsed figures (`stage5_figures.py`) | Stage 4b (coverage only), Stage 4a (annotation + pairwise) |\n| ElSherief delta figures (`stage5_figures.py`) | Stage 4c |\n| `generate_figures_final.py` (`fig1`-`fig7`, `appA`-`appG`) | Stage 1, Stage 2, Stage 3, Stage 4b, Stage 4c, and one upstream preprocessing file |\n| `generate_fig2_annotation_di_pairwise.py` | Stage 4a + Stage 4b, merged |\n| `auditing/03_audit_visualizations.ipynb` | Deprecated pipeline\'s own Stage 00-02 (not `audit_pipeline` numbering) |\n\n**Key ambiguity surfaced (the reason this task exists):** a "worst pairwise\nDI" figure exists reading **Stage 3** (`stage5_figures.level_stratified_figures`\n-> `s5_lev_disparity_worst_di.png`), a second reading **Stage 4a**\n(`stage5_figures.group_collapsed_figures` -> `s5_grp_worst_di_and_label_gap.png`),\nand a third reading **Stage 4b** (`generate_figures_final.fig5_worst_di_by_pair_level`\n/ `appD_worst_di_and_label_gap_pooled`, and `generate_fig2_annotation_di_pairwise.py`,\nwhich merges 4a+4b itself). None of the four figure-generation files agree\non a single canonical pairwise-DI source, and the Stage 4b file\n(`by_level_group`) -- the one a human manually spot-checking numbers would\nmost likely reach for first, since it\'s the only one whose name mentions\nboth "level" and "group" -- is used by exactly one of the four call sites\nthat produce a "worst DI" figure. Step 3 above numerically confirms whether\nthese different sources actually agree.\n\n## Step 3 result -- cross-stage consistency\n\nSee the "Step 3 result" cell above for the live, executed numbers (row\ncounts checked, mismatch count, full mismatch table if non-empty). That\ncell is authoritative; this paragraph is a static description of what was\nchecked: Stage 1 vs Stage 4b coverage, Stage 2 vs Stage 4b annotation,\nStage 4a vs Stage 4b (coverage and annotation), Stage 3 vs Stage 4b pairwise\nDI, Stage 3 vs Stage 4a pairwise DI, and `generate_fig2_annotation_di_pairwise.py`\'s\nown Stage 4a/4b overlap keys -- seven independent cross-checks in total. The\ndeprecated `auditing/` pipeline was found to be structurally incompatible\nwith `audit_pipeline` (no `coding_level` stratification) and is logged as\nnot-comparable rather than joined.\n\n## Known bugs fixed\n\nBoth occurrences of the `_top_n_s5` sort-direction bug described in the task\nbrief have been fixed. `_top_n_s5` now takes an explicit `ascending=`\nargument (default `False`, unchanged for every other call site -- token\nfrequency, label gaps, etc., where higher = worse); both\n`"worst_di_ratio"` call sites now pass `ascending=True` so the n MOST\ndisparate pairs are selected, not the least:\n\n1. **Cell 6** (Level-stratified figures / `level_stratified_figures`),\n   `d = _top_n_s5(sub, "worst_di_ratio", 8, ascending=True).sort_values(`\n   -- affects `s5_lev_disparity_worst_di.png`.\n2. **Cell 9** (Group-collapsed figures / `group_collapsed_figures`),\n   `left = _top_n_s5(pair_plot, "worst_di_ratio", 20, ascending=True).sort_values(`\n   -- affects `s5_grp_worst_di_and_label_gap.png`.\n\n(Cell indices are 0-based over the full notebook\'s cell list, counting both\nmarkdown and code cells, as saved in this build.)\n\n## Other surfaced (not fixed) discrepancies\n\n- `generate_fig2_annotation_di_pairwise.py`\'s docstring says it matches\n  "Figure 9 (fig5_worst_di_by_pair_level.pdf)" but its own output file is\n  named `fig2_annotation_di_by_pair_level.pdf` -- a `fig2` name that also\n  collides (in name only, not content) with `generate_figures_final.py`\'s\n  unrelated `annotation_rates_by_level.pdf`.\n- `generate_figures_final.py` and `generate_fig2_annotation_di_pairwise.py`\n  are not `PipelineVariant`-aware (no tier1+2 robustness-check support),\n  unlike `stage5_figures.py`.\n- `generate_fig2_annotation_di_pairwise.py` hardcodes a local\n  `DI_THRESHOLD = 0.80` instead of importing `audit_pipeline.config.DI_THRESHOLD`\n  (numerically identical, but a third independent definition of the same\n  constant).\n- The deprecated `auditing/` pipeline pools all metrics across coding levels\n  (no `coding_level` column anywhere in its output), unlike every\n  `audit_pipeline` stage.\n\n## Success checklist\n\n- [x] Notebook runs top-to-bottom with a fresh kernel, no errors (verified\n      via a redirected-output copy -- see build notes; this delivered copy\n      targets the real production paths and was not itself executed, so\n      that no existing figure artifact is modified as a side effect of\n      building this notebook).\n- [x] Every figure produced by any discovered script is reproduced here\n      under the same output filename.\n- [x] Every figure cell is preceded by a provenance markdown block.\n- [x] The cross-stage consistency section runs and produces a summary table\n      for every quantity with a second independent source (7 checks; see\n      Step 3 above for the live mismatch count).\n- [x] Zero hardcoded "expected" values -- all Step 3 comparisons read from\n      TSVs or `audit_pipeline` code; the only numeric literal introduced is\n      the `1e-6` comparison tolerance.\n- [x] Both `_top_n_s5` "worst DI" bug sites fixed above (see "Known bugs\n      fixed" section) with cell/line pointers.\n- [x] No existing file modified -- this notebook is the only new file this\n      task produced.\n'

print(summary_text)  # echoed to stdout per the task spec
display(Markdown(summary_text))


# Style Pass Summary (2026-07-03)

**Scope:** per author direction, careful render-and-verify styling was applied
to the 14 figures actually embedded in the paper's `.tex`
(`generate_figures_final.py`'s 13 + `generate_fig2_annotation_di_pairwise.py`'s
1). The 16 `stage5_figures.py` PNGs and 8 deprecated `auditing/` PNGs got the
same mechanical palette-harmonization and proportional resize, but were not
individually render-verified, since none of them appear in the paper.

**fig2/fig8 collision (Step 1):** `fig2_annotation_rates_by_level.pdf` renamed
to `annotation_rates_by_level.pdf` (position-independent, per author choice)
across the notebook, `generate_figures_final.py`, and `acl_latex (5).tex`.
Confirmed position: **Figure 1** in the main body (first `\begin{figure}`
before `\appendix` in `acl_latex (5).tex`) -- not "Figure 8" as originally
believed; that discrepancy was reported, not silently resolved.
`fig2_annotation_di_by_pair_level.pdf` (from `generate_fig2_annotation_di_pairwise.py`)
is confirmed as the real Figure 2, unchanged.

**Deprecation (Step 2):** `stage5_figures.py`, `generate_figures_final.py`,
and `generate_fig2_annotation_di_pairwise.py` each got a module-level
deprecation banner and an abort-by-default `__main__` guard
(`--i-know-this-is-deprecated` flag or `I_KNOW_THIS_IS_DEPRECATED=1` env var
to opt in). All three tested: direct execution aborts with no writes and
exit code 1 without the opt-in; both opt-in mechanisms verified to bypass
correctly. Imports are unaffected.

**Sandbox (Step 3):** the prior pass's monkeypatch-after-definition ordering
hazard is fixed structurally -- `OUTPUTS_DIR` itself is redirected to
`outputs/_style_pass_sandbox/` (git-ignored) in the first executable cell,
before any figure code is defined. A canary cell proves the redirect is live
before any real plotting runs. `SANDBOX_MODE = True` for this run; flipping
it to `False` is the one-line change described in the promotion note above.

**Style changes applied (Step 4):** all changes are figsize, color hex,
fontsize, tick locator/rotation, or padding-constant edits -- see the diff
walkthrough in the chat for the complete list. Highlights:
- Corrected figsize width to match confirmed `.tex` `\includegraphics` widths
  for every embedded figure that was mismatched (most were sized for full
  double-column width but embedded at single-column or partial width).
- Fixed a real regression the width correction exposed: 3-panel figures at
  `\columnwidth` had y-axis category labels bleeding across panels and
  x-axis tick labels colliding. Fixed via smaller tick fonts, fewer/rotated
  ticks -- verified by rendering at final size, not by eyeballing the
  larger original.
- Harmonized the `stage5_figures.py` palette (`_BLUE`/`_GREEN`/`_ORANGE`/
  `_RED`/`_PURPLE`/`_TEAL`, `_CL_COLOR` L2-L4) to exactly match
  `generate_figures_final.py`'s `STYLE["colors"]` hex values, at their single
  definition point.
- Bumped `appF_cross_level_deltas`'s label-decluttering gap and figure
  height for the known line-end-label crowding issue.

**Content-equivalence (Step 5):** zero changed lines across all 19 touched
cells reference any data/threshold/filter/sort keyword (full diff reviewed
line-by-line). Runtime data-extraction spot-check on the two highest-stakes
figures (`annotation_rates_by_level` = Figure 1, `fig2_annotation_di_by_pair_level`
= Figure 2) confirms identical bar widths/heights and text content before vs.
after.

**Known bugs / open questions:** the `_top_n` sort-direction bug noted
in the prior pass has since been fixed (see "Known bugs fixed" in the
Step 4 summary above); the Stage 3 vs. 4a vs. 4b canonical-pairwise-source
question from the prior pass remains open and is unchanged by this pass.

**Production promotion:** deferred. `SANDBOX_MODE` is still `True` in this
saved notebook; no real `outputs/stage5/**`, `outputs/figures_final/**`, or
`outputs/audit_visualizations/**` path was written to during this pass.


## Addendum: second round of fixes from author visual review (2026-07-03)

After the first sandbox render, the author reviewed the actual PDFs and found
four more issues that only showed up at final rendered size:

1. **`fig5_worst_di_by_pair_level` / `appE_annotation_label_gap_by_level`:**
   sparse panels (e.g. L4, often 1 pair) showed a single bar stretched to
   fill the whole panel height instead of matching the other panels' bar
   thickness. Fixed by porting the `max_n`-based fixed-`ylim` technique
   already used in `generate_fig2_annotation_di_pairwise.py`'s `make_figure`
   to both.
2. **`fig2_annotation_di_by_pair_level`:** the gray sparse-panel explanatory
   note (`STYLE["colors"]["neutral"]`) overlaid the L4 panel's visible bar
   and label. Removed entirely per author request.
3. **`fig1_coverage_heatmap`:** three vertical white/light gridlines were
   cutting through the L2/L3/L4 columns. Root cause: `stage5_figures.py`'s
   utilities cell sets `axes.grid`/`axes.grid.axis`/`axes.facecolor` globally
   via `mpl.rcParams.update()`, and `generate_figures_final.py`'s
   `apply_style()` never reset them back -- every gff figure was silently
   inheriting stage5's grid lines and off-white background from shared
   kernel-global rcParams state. Fixed by having `apply_style()` explicitly
   reset `axes.grid: False` / `axes.facecolor: white`, so it's insulated
   regardless of what ran before it in the same kernel.
4. **Label text size bumped** across the bar-plot figures that had been
   shrunk to fix panel-width crowding (`annotation_rates_by_level`, `fig3`,
   `appB`, `fig6`, `fig7`, `fig2_annotation_di_by_pair_level`). Attempted the
   same bump on `fig5`/`appE`, but their labels are full "X vs Y" pair
   strings (longer than the single-word category labels elsewhere) and the
   bump reintroduced cross-panel text bleed even with extra `wspace` --
   reverted those two back to their original (smaller) size rather than ship
   a regression for a cosmetic bump.

Re-executed top-to-bottom after these fixes: 0 errors, canary passed, real
production paths still untouched, outputs kept in this saved file.


## Addendum: third round of fixes from author visual review (2026-07-03, cont.)

1. **`fig2_annotation_di_by_pair_level`**: resized to a 3:2 aspect ratio at
   full `\textwidth` (was 0.6x width, 4.18in), so it spans the top of the
   page in its existing `figure*` environment. `acl_latex (5).tex` updated
   to `\includegraphics[width=\textwidth]`. Bars thinned (`height=0.6`),
   x-ticks restored to `[0, 0.5, 1.0]` and labels back to standard size now
   that panels have more room; left spine restored.
2. **Left y-axis spine restored** on `fig2_annotation_di_by_pair_level`,
   `fig3_case_ab_counts_by_level`, `appB_token_frequency_by_level`,
   `appD_worst_di_and_label_gap_pooled`, `appE_annotation_label_gap_by_level`
   -- these had `ax.spines["left"].set_visible(False)` /
   `tick_params(left=False)` from the original source (pre-dating this style
   pass); author wants the axis line visible. `fig5_worst_di_by_pair_level`,
   `fig6`/`fig7` (ElSherief), and `appC` were not in the author's list and
   were left with the spine hidden, matching original behavior -- flagged in
   chat in case the author wants those changed too for consistency.
3. **Aspect ratio brought closer to 1:1** for `fig5_worst_di_by_pair_level`
   (was 3.35x6.5in, now 3.35x4.0in, aspect 0.84), `fig3_case_ab_counts_by_level`
   (3.35x4.0in, 0.84), and `appD_worst_di_and_label_gap_pooled` (3.35x~3.0in,
   1.11) -- all were unnecessarily tall, making differently-sized bars look
   similar. Fixed via thinner bars (`height=0.6`, was default 0.8) rather
   than just shrinking the figure and letting bars visually merge.
4. **Text-cutoff check**: `_save`'s `bbox_inches="tight"` structurally
   prevents label clipping (the saved page always expands to include all
   rendered text) -- confirmed via direct PDF page-bbox inspection
   (`fitz`/PyMuPDF) rather than assuming. `appD`'s two-panel x-axis labels
   ("Worst DI ratio (pooled)" / "Largest label gap (pooled)") were
   overlapping each other at the narrow panel width (not clipped, but
   colliding) -- fixed by wrapping each onto two lines and shrinking to
   match its sibling figures.

Re-executed top-to-bottom after these fixes: 0 errors, real production paths
still untouched, outputs kept in this saved file.


## Addendum: fourth round of fixes from author visual review (2026-07-03, cont.)

1. **`appB_token_frequency_by_level`, `appD_worst_di_and_label_gap_pooled`**:
   author reported these still much too vertically long. Fixed by
   *widening* rather than shortening -- since `.tex` scales embedded width
   to a fixed target (`\columnwidth` for `appB`; `appD` is not currently
   embedded at all, commented out), a wider source canvas at the same data
   range makes the eventual on-page aspect ratio squarer without touching
   any font/bar-thickness settings. `appB`: (3.35, 5.5) -> (5.5, 5.5),
   aspect now exactly 1.00. `appD`: (3.35, ~3.0) -> (6.0, ~3.0), aspect 1.98
   (wide, per author's explicit "increasing horizontal scale" request).
   Bonus: `appB`'s previously near-invisible low-count bars are now clearly
   visible bars at this wider scale, and its x-tick labels no longer need
   90-degree rotation.
2. **`fig5_worst_di_by_pair_level`**: left spine was still hidden (this
   figure wasn't in the original 5-figure spine-restoration list from the
   previous round) -- restored per this round's explicit request.
3. **Further vertical compression** on `fig5_worst_di_by_pair_level`,
   `appE_annotation_label_gap_by_level`, `fig2_annotation_di_by_pair_level`:
   added an explicit `ROW_SPACING = 0.6` multiplier on the y-positions (was
   1.0 implicit via `np.arange`), decreasing the gap between bars, and
   thinned bars further (`height=0.4`, was `0.6`). Figure heights reduced
   accordingly. `fig2_annotation_di_by_pair_level`'s aspect ratio drifted
   from the previous round's 3:2 target to ~2.2:1 as a direct result --
   explicitly accepted by the author ("aspect ratio is not as important as
   the space being used effectively").

Re-executed top-to-bottom after these fixes: 0 errors, real production
paths still untouched, outputs kept in this saved file. Final aspect ratios
(w:h, measured from the actual saved PDF page box via PyMuPDF): `appB`=1.00,
`appD`=1.98, `fig5`=1.11, `appE`=0.96, `fig2_annotation_di_by_pair_level`=2.19.


## Addendum: fifth round -- batch feedback on generate_figures_final.py outputs (2026-07-03, cont.)

Two naming corrections made before applying this round's feedback (both
confirmed from actual code/renders, not guessed):
- "appB (appB_annotation_case_ab_counts)" does not exist; the described
  content (an unfiltered `referential_white_supremacist` bar) matches the
  real `appB_token_frequency_by_level` -- applied there.
- "fig3 (fig3_cross_level_deltas)" does not exist either; the described
  content ("Match count" label, rightmost-panel cutoff) matches the real
  `fig3_case_ab_counts_by_level` -- applied there. (The actual cross-level
  figure is `appF_cross_level_deltas`, untouched by this round.)

Changes applied, all verified by rendering + PDF page-bbox measurement:

1. `annotation_rates_by_level`: widened to (6.0, 6.0), aspect 1.00. Restored
   `[0, 0.5, 1.0]` x-ticks.
2. `appA_coverage_presence_vs_type_by_level`: left spine restored, height
   5.5->3.5, bars `height=0.6`.
3. `appB_token_frequency_by_level`: bars `height=0.6`, y-tick fontsize
   6.5->8, x-tick fontsize 6->7 (rotation removed, panels wide enough),
   `referential_white_supremacist` dropped (same fix already applied
   elsewhere in this notebook; `is_self_referential` alone doesn't catch it).
4. `appD_worst_di_and_label_gap_pooled`: `ROW_SPACING=0.6`, bars
   `height=0.4` (was 0.6), `[0, 0.5, 1.0]` x-ticks added to the left (DI)
   panel.
5. `appE_annotation_label_gap_by_level`: widened to (6.0, 3.0), fixed
   `set_xlim(0, 1.0)` + `[0, 0.5, 1.0]` ticks (was auto-scaled per panel),
   `ROW_SPACING` 0.6->0.5, bars `height` 0.4->0.35.
6. `fig2_annotation_di_by_pair_level`: `ROW_SPACING` 0.6->0.45, bars
   `height` 0.4->0.3, figure height further reduced (aspect now ~2.9:1,
   drifted further from the 3:2 target per author's explicit go-ahead).
7. `fig3_case_ab_counts_by_level`: widened to (8.0, 4.0), aspect ~2:1 (fixes
   the rightmost panel's "Match count" label getting clipped). `ROW_SPACING
   =0.6`, bars `height` 0.6->0.4. **Fixed a bug introduced by this change**:
   the truncated-bar count annotation (`ax.text` showing the true count next
   to bars clipped by the x-axis) was using the raw enumerate index as its
   y-position, which desynced from the bars once `y` became
   `ROW_SPACING`-scaled -- switched to `zip(y, totals)`.
8. `fig5_worst_di_by_pair_level`: widened to (6.0, 3.0), aspect ~2:1 (fixes
   the rightmost panel's "Worst DI ratio" label getting clipped).
   `[0, 0.5, 1.0]` x-ticks restored. `ROW_SPACING` 0.6->0.45, bars `height`
   0.4->0.3.
9. `fig6_elsherief_annotation_delta`: `ROW_SPACING=0.6` (new), bars
   `height=0.5` (was default 0.8), height formula shrunk to match.
10. `fig7_elsherief_coverage_delta`: bars `height=0.5` (was default 0.8),
    width `0.6x -> 0.75x` columnwidth per author request so bar-length
    differences read more clearly (e.g. Race:white vs Race:latinx).

Re-executed top-to-bottom: 0 errors, real production paths still untouched,
outputs kept in this saved file. Final aspect ratios (w:h, measured from
actual PDF page box via PyMuPDF): `annotation_rates_by_level`=1.00,
`appA`=1.98, `appB`=1.00, `appD`=1.98, `appE`=1.98,
`fig2_annotation_di_by_pair_level`=2.88, `fig3`=1.99, `fig5`=1.98.


## Addendum: sixth round -- match row density to fig2/fig3/fig6 (2026-07-03, cont.)

Author confirmed `fig2_annotation_di_by_pair_level`, `fig3_case_ab_counts_by_level`,
and `fig6_elsherief_annotation_delta` have the right row density (bar +
padding roughly matching the text label's own height) and asked for
`appB_token_frequency_by_level`, `fig5_worst_di_by_pair_level`, and
`appE_annotation_label_gap_by_level` to match. Root cause: those three had
their bar `height`/`ROW_SPACING` tightened in earlier rounds but their
overall figure *height* was never reduced to match -- leaving unused
vertical room the earlier bar-thinning alone couldn't close.

- `appB_token_frequency_by_level`: had no `ROW_SPACING` at all (still plain
  `np.arange`) despite `fig3`/`fig6` already using one -- added
  `ROW_SPACING=0.6`, bars `height` 0.6->0.4 (matches `fig3`'s ratio), figure
  height 5.5->3.0.
- `fig5_worst_di_by_pair_level`: `ROW_SPACING`/bar-height ratio already
  matched `fig2`'s; figure height alone was too generous -- 3.0->2.4.
- `appE_annotation_label_gap_by_level`: same issue, figure height 3.0->2.4.

Re-executed top-to-bottom: 0 errors, real production paths still untouched,
outputs kept in this saved file.


## Addendum: seventh round -- fig5/appD tightening, appE dynamic axes, fig1 aspect (2026-07-03, cont.)

1. `fig5_worst_di_by_pair_level`: figure height 2.4->2.1in, further toward
   fig2/fig3/fig6's row density.
2. `appD_worst_di_and_label_gap_pooled`: height formula floor/slope
   `max(3.0, n*0.16+1.0)` -> `max(1.8, n*0.13+0.6)` -- this had not yet had
   its figure-height examined the way appB/fig5/appE were in the previous
   round, and the floor was binding well above what its ~6 rows need.
3. `appE_annotation_label_gap_by_level`: reverted the fixed `[0,1]` x-axis
   (added last round for a hardcoded 0.5 marker) back to a per-panel
   dynamic range -- each subplot now scales to `gaps.max() * 1.15` with a
   midpoint tick computed from that panel's own range, not a literal 0.5.
   Known minor limitation: the L4 panel here has only one very small-valued
   bar, so its computed midpoint rounds to the same 2-decimal display as
   zero at default formatting -- flagged, not fixed unless it matters to
   the author.
4. `fig1_coverage_heatmap`: width was a fixed `STYLE["textwidth_in"] * 0.67`
   independent of the content-driven height, giving an extreme ~0.35:1
   final aspect for a tall single-page appendix figure. Changed to
   `fig_h * 0.85` (empirically calibrated so the final *saved* PDF -- after
   `bbox_inches="tight"` trims unused margin at the new width -- lands at
   2:3; a naive `fig_h * 2/3` nominal undershoots to ~0.53:1 once trimmed).
   Measured final aspect: 0.669 (target 0.667). `.tex` embed width
   (`0.67\textwidth`) unchanged -- the wider source makes the final printed
   height shrink from ~13in to a reasonable single-page ~7in instead.

Re-executed top-to-bottom: 0 errors, real production paths still untouched,
outputs kept in this saved file.


## Addendum: eighth round -- annotation_rates_by_level stacked bars (2026-07-04)

`annotation_rates_by_level` (the paper's actual Figure 1) changed from two
dodged bars per category (`Correct` and `Failure`, offset +-`BAR_H/2`) to a
single stacked bar per category (`Correct` from 0, `Failure` concatenated
via `left=correct`), matching `appA_coverage_presence_vs_type_by_level`'s
stacking convention per author request. `correct_labeling_rate` and
`failure_rate` always sum to 1, so this communicates the same information
in half the vertical space per category. `BAR_H` (only used by the removed
dodged-bar calls) removed as dead code.

Re-executed top-to-bottom: 0 errors, real production paths still untouched,
outputs kept in this saved file.


## Addendum: ninth round -- consistent bar thickness in annotation_rates_by_level (2026-07-04)

`annotation_rates_by_level` never had the `max_n`-based fixed-`ylim`
technique (already used in `fig5`/`appE`/`fig2_annotation_di_by_pair_level`)
applied to it, so its sparse L4 panel (4 stable groups vs. L2/L3's 12-14)
auto-scaled its axis to fit just those 4 bars, making them visually much
thicker than L2/L3's. Added `max_n = max(1, max((s2["coding_level"] == lvl).sum()
for lvl in LEVELS))` and `ax.set_ylim(-0.5, max_n - 0.5)` per panel, matching
the convention already used elsewhere. Sparse panels now show genuine blank
space rather than stretched bars.

Re-executed top-to-bottom: 0 errors, real production paths still untouched,
outputs kept in this saved file.


## Addendum: tenth round -- bar height/text size in annotation_rates_by_level (2026-07-04, cont.)

Added `ROW_SPACING=0.6` and explicit bar `height=0.4` (was default 0.8),
matching `fig3`/`fig6`'s established row density. Y-tick label fontsize
bumped 7->8. Final aspect stayed close to 1:1 (6.08 x 6.06in).

Re-executed top-to-bottom: 0 errors, real production paths still untouched,
outputs kept in this saved file.


## Addendum: eleventh round -- appC_annotation_rates_pooled matches annotation_rates_by_level (2026-07-04, cont.)

Applied the same set of changes just made to `annotation_rates_by_level`
(its unpooled sibling, same underlying Correct/Failure rate semantics):
- Stacked bars (was two dodged bars offset +-`BAR_H/2`) -- `correct_rate`
  and `failure_rate` always sum to 1, matches `appA`'s stacking convention.
- `ROW_SPACING=0.6`, bar `height=0.4` (was default 0.8), figure height
  formula shrunk to match (bar height roughly matching text label height).
- Y-tick label fontsize 8->9 (bigger text).
- **Found and fixed along the way**: the axes-level legend
  (`loc="lower right"`, inside the plot) started overlapping the bottom two
  bars once rows were tightened -- moved to a fig-level legend below the
  plot (`loc="outside lower center"`), matching the convention used
  throughout this notebook (and the same class of fix just applied to
  `fpr_iaa_analysis.ipynb`'s Figure 4).

Re-executed top-to-bottom: 0 errors, real production paths still untouched,
outputs kept in this saved file.


## Addendum: twelfth round -- fig7_elsherief_coverage_delta row spacing (2026-07-04, cont.)

`fig7` never had `ROW_SPACING` applied (unlike its sibling `fig6`, which
already used `ROW_SPACING=0.6`) despite both being ElSherief delta bar
charts with the same structure -- added `ROW_SPACING=0.6`, mirroring `fig6`,
and shrunk the figure-height formula to match (was `n*0.22+0.7`, now
`n*0.16+0.6`).

Re-executed top-to-bottom: 0 errors, real production paths still untouched,
outputs kept in this saved file.


## Addendum: thirteenth round -- fig2_annotation_di_by_pair_level further compression (2026-07-04, cont.)

Further tightened: `ROW_SPACING` 0.45->0.35, bar `height` 0.3->0.22, figure
height `STYLE["textwidth_in"] * 0.34 -> * 0.26`. Verified via render: bars
remain distinguishable with visible gaps, no text cutoff.

Re-executed top-to-bottom: 0 errors, real production paths still untouched,
outputs kept in this saved file.
